In [ ]:
# ============================================================
# Notebook 5 - Cell 1 (REDO): eICU-CRD v2.0 external validation - setup
#  (a) paths + output folders   (b) SHA-256 integrity (skipped if already all OK)
#  (c) software versions vs Notebook 4   (d) frozen MIMIC models + locked rules
# Reads NO patient outcomes. Prints no patient-level data.
# ============================================================
import sys, json, re, hashlib, platform, time
from pathlib import Path
import numpy as np, pandas as pd, joblib
import sklearn, xgboost, catboost, lightgbm, scipy

# ---------- (a) paths ----------
ROOT   = Path.home() / "Documents" / "DBMS_resarch"
EICU   = ROOT / "data" / "eicu.nosync" / "eicu-collaborative-research-database-2.0"
MODELS = ROOT / "models_v2"
OUT    = ROOT / "data" / "processed_v2" / "nb5";  OUT.mkdir(parents=True, exist_ok=True)
FIGS   = ROOT / "figures_v2" / "nb5";            FIGS.mkdir(parents=True, exist_ok=True)

def read_retry(path, mode="text", tries=6, wait=10):
    """Read a file; if iCloud has offloaded it, wait and retry."""
    for i in range(tries):
        try:
            return path.read_text() if mode == "text" else joblib.load(path)
        except (TimeoutError, OSError) as e:
            print(f"  waiting for iCloud to restore {path.name} ({i+1}/{tries}) ...")
            time.sleep(wait)
    raise RuntimeError(f"Could not read {path} - run: brctl download {path.parent}")

helper_dirs = sorted({p.parent for p in ROOT.rglob("v2_*.py") if "github_release" not in str(p)})
for d in helper_dirs:
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))
print("eICU folder exists:", EICU.exists(), "|", EICU)
print("Helper module folders:", [str(d) for d in helper_dirs])

# ---------- (b) integrity check (reuse earlier result if all OK) ----------
NEEDED = ["patient", "hospital", "diagnosis", "pastHistory", "admissionDx",
          "apacheApsVar", "apachePredVar", "apachePatientResult",
          "vitalPeriodic", "vitalAperiodic", "nurseCharting", "nurseAssessment",
          "lab", "infusionDrug", "medication", "treatment",
          "respiratoryCare", "respiratoryCharting", "intakeOutput", "physicalExam"]
ck = OUT / "nb5_file_checksums.csv"
if ck.exists() and (pd.read_csv(ck).status == "OK").all():
    c = pd.read_csv(ck)
    print(f"Checksums: already verified - {len(c)} OK (from {ck.name})")
else:
    sums_txt = read_retry(EICU / "SHA256SUMS.txt")
    expected = {Path(m.group(2)).name: m.group(1).lower()
                for m in (re.search(r"\b([0-9a-fA-F]{64})\b\s*\*?\s*(\S+)", l) for l in sums_txt.splitlines()) if m}
    def sha256(p, chunk=8 * 1024 * 1024):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(chunk), b""): h.update(b)
        return h.hexdigest()
    rows = [(n, "MISSING" if not (EICU / n).exists() else ("OK" if sha256(EICU / n) == e else "FAILED"))
            for n, e in sorted(expected.items())]
    c = pd.DataFrame(rows, columns=["file", "status"]); c.to_csv(ck, index=False)
    print(f"Checksums: {(c.status=='OK').sum()} OK, {(c.status!='OK').sum()} not OK")
miss = [t for t in NEEDED if not (EICU / f"{t}.csv.gz").exists()]
print("Tables needed by Notebook 5 present:", "ALL" if not miss else f"MISSING {miss}")

# ---------- (c) software versions vs Notebook 4 ----------
now = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
       "catboost": catboost.__version__, "lightgbm": lightgbm.__version__}
sv = ROOT / "data" / "processed_v2" / "nb4" / "software_versions.json"
nb4 = json.loads(read_retry(sv)) if sv.exists() else {}
flat = {}
for k, v in nb4.items():
    flat.update(v) if isinstance(v, dict) else flat.__setitem__(k, v)
print("\nSoftware (now vs Notebook 4):")
for k, v in now.items():
    old = str(flat.get(k, flat.get(k.replace("-", "_"), "?")))
    print(f"  {k:<13} {v:<10} NB4: {old:<10} {'' if old in ('?', v) else '<-- DIFFERENT'}")

# ---------- (d) frozen models + locked rules ----------
print("\nFrozen model bundles:")
bundles = {}
for f in sorted(MODELS.glob("final_*.joblib")):
    b = read_retry(f, mode="joblib"); bundles[f.stem] = b
    keys = list(b.keys()) if isinstance(b, dict) else [type(b).__name__]
    print(f"  {f.name}: keys = {keys}")
    if isinstance(b, dict):
        for k in b:
            if "feat" in k.lower() and isinstance(b[k], (list, tuple, pd.Index, np.ndarray)):
                print(f"    {k} ({len(b[k])}): {list(b[k])}")
for f in sorted(MODELS.glob("locked_rules_*.json")):
    print(f"  {f.name}: {json.loads(read_retry(f))}")

print("\nCell 1 done - no outcome data read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 2 (NEW): inspect frozen preparation + locate eICU sources
#  (a) structure of the frozen preparation object (prep_random_split.joblib)
#  (b) what the frozen final model expects as input
#  (c) eICU: where cancer, delirium/CAM-ICU and GCS are recorded (label counts only)
#  (d) eICU patient table: stays, first unit visit, >=24 h stays (NO outcome columns read)
# Prints only labels and counts - no patient-level rows, no outcomes.
# ============================================================
import collections

def describe(obj, name, depth=0, max_depth=2):
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict with {len(obj)} keys")
        if depth < max_depth:
            for k, v in obj.items():
                describe(v, str(k), depth + 1, max_depth)
    elif isinstance(obj, (list, tuple, pd.Index, np.ndarray)):
        print(f"{pad}{name}: {type(obj).__name__} len {len(obj)}  first: {list(obj)[:5]}")
    elif isinstance(obj, pd.DataFrame):
        print(f"{pad}{name}: DataFrame {obj.shape}  cols: {list(obj.columns)[:6]} ...")
    else:
        extra = ""
        for attr in ("n_features_in_", "feature_names_in_"):
            if hasattr(obj, attr):
                v = getattr(obj, attr)
                extra += f" {attr}={len(v) if hasattr(v, '__len__') else v}"
        print(f"{pad}{name}: {type(obj).__name__}{extra}")

# ---------- (a) frozen preparation ----------
print("=" * 70, "\n(a) prep_random_split.joblib")
prep = read_retry(MODELS / "prep_random_split.joblib", mode="joblib")
describe(prep, "prep")

# ---------- (b) frozen final model input ----------
print("=" * 70, "\n(b) final model (random split)")
B = bundles["final_random_split"]
for k in ["final_calibrated", "final_uncalibrated", "cvap", "SOFA", "APACHE-II"]:
    describe(B[k], k)
fm = B["final_calibrated"]
names = getattr(fm, "feature_names_in_", None)
print("final_calibrated input columns:", list(names) if names is not None else "not stored (uses 'features' order)")
print("final_model_type:", B["final_model_type"], "| threshold:", B["threshold"])

# ---------- (c) eICU sources ----------
def scan(table, cols, pattern, top=40, chunksize=2_000_000):
    """Count label combinations whose text matches pattern (case-insensitive)."""
    cnt = collections.Counter()
    for ch in pd.read_csv(EICU / f"{table}.csv.gz", usecols=cols, chunksize=chunksize, dtype=str):
        m = np.zeros(len(ch), dtype=bool)
        for c in cols:
            m |= ch[c].str.contains(pattern, case=False, na=False, regex=True).values
        if m.any():
            cnt.update(map(tuple, ch.loc[m, cols].fillna("").values))
    out = pd.DataFrame([(*k, v) for k, v in cnt.most_common(top)], columns=cols + ["rows"])
    print(f"\n--- {table}: rows matching /{pattern}/  (top {top} of {len(cnt)} combinations)")
    print(out.to_string(index=False, max_colwidth=70))
    return out

print("=" * 70, "\n(c) eICU: delirium / CAM-ICU")
d1 = scan("diagnosis", ["diagnosisstring"], r"delirium|encephalopathy|altered mental")
d2 = scan("nurseAssessment", ["cellattribute", "cellattributepath", "cellattributevalue"], r"delir|CAM-ICU|\bCAM\b")
d3 = scan("nurseCharting", ["nursingchartcelltypecat", "nursingchartcelltypevallabel",
                            "nursingchartcelltypevalname", "nursingchartvalue"], r"delir|CAM-ICU|\bCAM\b")

print("=" * 70, "\n(c) eICU: cancer")
c1 = scan("diagnosis", ["diagnosisstring"], r"oncology|cancer|malignan|lymphoma|leukemia|myeloma|metasta", top=50)
c2 = scan("pastHistory", ["pasthistorypath"], r"cancer|malignan|lymphoma|leukemia|myeloma|metasta|chemo|radiation", top=50)
icd = pd.read_csv(EICU / "diagnosis.csv.gz", usecols=["icd9code"], dtype=str)["icd9code"].dropna()
print("\nICD code field: non-empty rows", f"{len(icd):,}", "| example formats:", icd.drop_duplicates().head(8).tolist())

print("=" * 70, "\n(c) eICU: GCS sources")
g1 = scan("nurseCharting", ["nursingchartcelltypevallabel", "nursingchartcelltypevalname"], r"glasgow|GCS", top=20)
g2 = scan("physicalExam", ["physicalexampath"], r"GCS", top=20)

# ---------- (d) patient table, no outcomes ----------
print("=" * 70, "\n(d) eICU patient table (no outcome columns)")
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str,
                 usecols=["patientunitstayid", "uniquepid", "patienthealthsystemstayid", "unitvisitnumber",
                          "age", "hospitalid", "unitdischargeoffset", "hospitaladmityear"])
pt["unitdischargeoffset"] = pd.to_numeric(pt["unitdischargeoffset"], errors="coerce")
print(f"ICU stays {len(pt):,} | patients {pt.uniquepid.nunique():,} | hospitals {pt.hospitalid.nunique()}")
print("unitvisitnumber==1:", f"{(pt.unitvisitnumber=='1').sum():,}",
      "| unit stay >= 24 h:", f"{(pt.unitdischargeoffset >= 1440).sum():,}")
print("age values that are not plain numbers:", pt.age[~pt.age.fillna('').str.fullmatch(r'\d+')].value_counts().to_dict())
print("hospital admit years:", pt.hospitaladmityear.value_counts().sort_index().to_dict())

for name, df in {"eicu_scan_delirium_dx": d1, "eicu_scan_delirium_nurseassess": d2, "eicu_scan_delirium_nursechart": d3,
                 "eicu_scan_cancer_dx": c1, "eicu_scan_cancer_pasthx": c2, "eicu_scan_gcs_nursechart": g1,
                 "eicu_scan_gcs_physexam": g2}.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
print("\nCell 2 done - labels and counts only, no outcomes read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 2b (NEW): finish Cell 2 + save the CAM-ICU extract
#  (d) patient table (correct column names; NO outcome columns)
#  (e) full past-history cancer labels (to decide inclusion / exclusion)
#  (f) delirium ICD codes in eICU diagnosis (F05 / 293.0 / 293.1)
#  (g) extract Delirium Scale/Score rows -> local file; count CAM-ICU positive stays
# Prints only counts and labels. The extract stays on this Mac (patient-level).
# ============================================================
# ---------- (d) patient table ----------
print("=" * 70, "\n(d) eICU patient table (no outcome columns)")
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str,
                 usecols=["patientunitstayid", "uniquepid", "patienthealthsystemstayid", "unitvisitnumber",
                          "age", "hospitalid", "unitdischargeoffset", "hospitaldischargeyear"])
pt["unitdischargeoffset"] = pd.to_numeric(pt["unitdischargeoffset"], errors="coerce")
print(f"ICU stays {len(pt):,} | patients {pt.uniquepid.nunique():,} | hospitals {pt.hospitalid.nunique()}")
print("unitvisitnumber==1:", f"{(pt.unitvisitnumber=='1').sum():,}",
      "| unit stay >= 24 h:", f"{(pt.unitdischargeoffset >= 1440).sum():,}")
print("age values that are not plain numbers:", pt.age[~pt.age.fillna('').str.fullmatch(r'\d+')].value_counts().to_dict())
print("hospital discharge years:", pt.hospitaldischargeyear.value_counts().sort_index().to_dict())

# ---------- (e) full past-history cancer labels ----------
print("=" * 70, "\n(e) pastHistory oncology labels (full text)")
ph = pd.read_csv(EICU / "pastHistory.csv.gz", dtype=str, usecols=["pasthistorypath"])
onc = ph[ph.pasthistorypath.str.contains("Hematology/Oncology", na=False)]
lab = (onc.pasthistorypath.str.replace("notes/Progress Notes/Past History/Organ Systems/Hematology/Oncology", "", regex=False)
          .value_counts())
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 200):
    print(lab.head(80).to_string())
lab.to_csv(OUT / "eicu_pasthistory_oncology_labels.csv")

# ---------- (f) delirium ICD codes ----------
print("=" * 70, "\n(f) delirium ICD codes in diagnosis.icd9code")
dx = pd.read_csv(EICU / "diagnosis.csv.gz", dtype=str, usecols=["patientunitstayid", "icd9code", "diagnosisstring"])
codes = dx.icd9code.fillna("").str.replace(" ", "").str.split(",")
dx["has_delir_icd"] = codes.apply(lambda c: any(x.upper().startswith("F05") or x in ("293.0", "293.1", "2930", "2931") for x in c))
dx["has_delir_txt"] = dx.diagnosisstring.str.contains(r"\|delirium", case=False, na=False)
print("rows with delirium ICD:", f"{dx.has_delir_icd.sum():,}", "| stays:", dx.loc[dx.has_delir_icd, "patientunitstayid"].nunique())
print("rows with 'delirium' label:", f"{dx.has_delir_txt.sum():,}", "| stays:", dx.loc[dx.has_delir_txt, "patientunitstayid"].nunique())
print("label rows that also carry a delirium ICD:", f"{(dx.has_delir_txt & dx.has_delir_icd).sum():,}")

# ---------- (g) CAM-ICU extract ----------
print("=" * 70, "\n(g) Delirium Scale/Score extract from nurseCharting")
cols = ["patientunitstayid", "nursingchartoffset", "nursingchartcelltypevallabel",
        "nursingchartcelltypevalname", "nursingchartvalue"]
parts = []
for ch in pd.read_csv(EICU / "nurseCharting.csv.gz", usecols=cols, dtype=str, chunksize=3_000_000):
    m = ch.nursingchartcelltypevallabel.isin(["Delirium Scale/Score", "Symptoms of Delirium Present"])
    if m.any():
        parts.append(ch.loc[m])
dl = pd.concat(parts, ignore_index=True)
dl["nursingchartoffset"] = pd.to_numeric(dl.nursingchartoffset, errors="coerce")
dl.to_csv(OUT / "eicu_delirium_charting.csv.gz", index=False)   # patient-level: stays on this Mac
print("delirium rows saved:", f"{len(dl):,}", "->", "eicu_delirium_charting.csv.gz (do not share)")

sc = dl[dl.nursingchartcelltypevallabel == "Delirium Scale/Score"]
scale = sc[sc.nursingchartcelltypevalname == "Delirium Scale"][["patientunitstayid", "nursingchartoffset", "nursingchartvalue"]] \
          .rename(columns={"nursingchartvalue": "scale"})
score = sc[sc.nursingchartcelltypevalname == "Delirium Score"][["patientunitstayid", "nursingchartoffset", "nursingchartvalue"]] \
          .rename(columns={"nursingchartvalue": "score"})
pair = scale.merge(score, on=["patientunitstayid", "nursingchartoffset"], how="inner")
pair["scale_u"] = pair.scale.str.strip().str.upper()
pair["score_u"] = pair.score.str.strip().str.upper()
print("\nscale x score combinations at the same time (top 15):")
print(pair.groupby(["scale_u", "score_u"]).size().sort_values(ascending=False).head(15).to_string())
cam = pair[pair.scale_u == "CAM-ICU"]
cam_pos = cam[cam.score_u == "YES"]
print(f"\nstays with any CAM-ICU assessment: {cam.patientunitstayid.nunique():,}")
print(f"stays with >=1 positive CAM-ICU:    {cam_pos.patientunitstayid.nunique():,}")
unpaired = len(score) - len(pair)
print(f"score rows without a scale row at the same time: {unpaired:,} (handled in the cohort cell)")
print("\nCell 2b done - counts only, no outcomes read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 3 (REDO): eICU cohort with the SAME rules as MIMIC - NO outcomes read
#  cancer   : ICD code in our MIMIC list (2,052 codes)  OR  active oncology problem in the ICU diagnosis list
#  delirium : ICD F05 / 293.0 / 293.1  OR  CAM-ICU positive (scale = CAM-ICU and score = Yes, same time)
#  adults; FIRST QUALIFYING ICU stay per patient (as in MIMIC); ICU stay >= 24 h
# ============================================================
import re, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

# ---------- MIMIC cancer code list (cancer rows only) ----------
ap = pd.read_csv(ROOT / "data" / "processed_v2" / "icd_code_appendix.csv", dtype=str)
ap = ap[ap.category.str.strip().str.lower() == "cancer"].copy()
ap["code"] = ap.icd_code.str.replace(".", "", regex=False).str.strip().str.upper()
CODES = {9: set(ap.loc[ap.icd_version.str.strip() == "9", "code"]),
         10: set(ap.loc[ap.icd_version.str.strip() == "10", "code"])}
assert len(ap) == 2052
print(f"MIMIC cancer codes: ICD-9 {len(CODES[9]):,} | ICD-10 {len(CODES[10]):,}")

def cancer_icd(field):
    if not isinstance(field, str) or not field.strip():
        return False
    parts = [p.strip().replace(".", "").upper() for p in field.split(",") if p.strip()]
    for i, c in enumerate(parts):
        ver = (9 if i == 0 else 10) if len(parts) == 2 else (10 if c[:1].isalpha() and c[:1] not in "EV" else 9)
        pool = CODES[ver]
        if c in pool or (len(c) >= 3 and any(x.startswith(c) for x in pool)):
            return True
    return False

ONC_DX = re.compile(r"^oncology\||hematology\|oncology and leukemia\|(?!myelodysplastic)|s/p surgery for cancer|"
                    r"s/p head and neck cancer surgery|lung parenchyma\|malignancy|metastatic brain tumor|"
                    r"pleural effusion\|malignant", re.I)

# ---------- diagnosis table ----------
dx = pd.read_csv(EICU / "diagnosis.csv.gz", dtype=str,
                 usecols=["patientunitstayid", "diagnosisoffset", "diagnosisstring", "icd9code"])
dx["onc_text"] = dx.diagnosisstring.fillna("").str.contains(ONC_DX) & \
                 ~dx.diagnosisstring.fillna("").str.contains("benign", case=False)
icd_hit = {f: cancer_icd(f) for f in dx.icd9code.dropna().unique()}
dx["onc_icd"] = dx.icd9code.map(icd_hit).astype("boolean").fillna(False).astype(bool)
codes = dx.icd9code.fillna("").str.replace(" ", "").str.upper().str.split(",")
dx["delir_icd"] = codes.apply(lambda c: any(x.startswith("F05") or x in ("293.0", "293.1", "2930", "2931") for x in c))
stay_onc_icd = set(dx.loc[dx.onc_icd, "patientunitstayid"])
stay_onc_txt = set(dx.loc[dx.onc_text, "patientunitstayid"])
stay_del_icd = set(dx.loc[dx.delir_icd, "patientunitstayid"])

# ---------- past-history cancer (sensitivity only) ----------
ph = pd.read_csv(EICU / "pastHistory.csv.gz", dtype=str, usecols=["patientunitstayid", "pasthistorypath"])
ph_c = ph.pasthistorypath.fillna("").str.contains(
    r"Hematology/Oncology \(R\)/Cancer/(?:Cancer-Primary Site/(?!none)|Hematologic Malignancy|Metastases)", regex=True)
stay_onc_ph = set(ph.loc[ph_c, "patientunitstayid"])

# ---------- CAM-ICU from the saved extract ----------
dl = pd.read_csv(OUT / "eicu_delirium_charting.csv.gz", dtype={"patientunitstayid": str})
sc = dl[dl.nursingchartcelltypevallabel == "Delirium Scale/Score"]
scale = sc[sc.nursingchartcelltypevalname == "Delirium Scale"][["patientunitstayid", "nursingchartoffset", "nursingchartvalue"]]
score = sc[sc.nursingchartcelltypevalname == "Delirium Score"][["patientunitstayid", "nursingchartoffset", "nursingchartvalue"]]
pair = scale.merge(score, on=["patientunitstayid", "nursingchartoffset"], suffixes=("_scale", "_score"))
cam_pos = pair[(pair.nursingchartvalue_scale.str.strip().str.upper() == "CAM-ICU") &
               (pair.nursingchartvalue_score.str.strip().str.upper() == "YES")]
first_cam = cam_pos.groupby("patientunitstayid").nursingchartoffset.min()
stay_cam = set(first_cam.index)

# ---------- patient table (no outcome columns) ----------
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str,
                 usecols=["patientunitstayid", "uniquepid", "patienthealthsystemstayid", "unitvisitnumber",
                          "age", "gender", "hospitalid", "unitdischargeoffset", "hospitaldischargeyear",
                          "hospitaladmitoffset", "admissionweight"])
pt["age_num"] = pd.to_numeric(pt.age.replace("> 89", "90"), errors="coerce")
pt["unitdischargeoffset"] = pd.to_numeric(pt.unitdischargeoffset, errors="coerce")
pt["visit_num"] = pd.to_numeric(pt.unitvisitnumber, errors="coerce")
pt["hosp_year"] = pd.to_numeric(pt.hospitaldischargeyear, errors="coerce")
pt["hsid"] = pd.to_numeric(pt.patienthealthsystemstayid, errors="coerce")

att = []
def step(label, df):
    att.append((label, len(df), df.uniquepid.nunique()))
    return df

c = step("All eICU ICU stays", pt)
c = step("Cancer (ICD list or active oncology diagnosis)", c[c.patientunitstayid.isin(stay_onc_icd | stay_onc_txt)])
c = step("... and delirium (ICD or CAM-ICU positive)", c[c.patientunitstayid.isin(stay_del_icd | stay_cam)])
c = step("... age >= 18", c[c.age_num >= 18])
# first QUALIFYING stay per patient: earliest year, then earliest hospital stay, then earliest ICU visit
c = c.sort_values(["uniquepid", "hosp_year", "hsid", "visit_num"])
c = step("... first qualifying ICU stay per patient", c.drop_duplicates("uniquepid", keep="first"))
c = step("... valid ICU discharge time", c[c.unitdischargeoffset.notna()])
c = step("... still in ICU at 24 h (ICU stay >= 24 h)", c[c.unitdischargeoffset >= 1440])

coh = c.drop(columns=["hosp_year", "hsid"]).copy()
coh["cancer_icd"] = coh.patientunitstayid.isin(stay_onc_icd).astype(int)
coh["cancer_dx_text"] = coh.patientunitstayid.isin(stay_onc_txt).astype(int)
coh["cancer_pasthx"] = coh.patientunitstayid.isin(stay_onc_ph).astype(int)
coh["delirium_icd"] = coh.patientunitstayid.isin(stay_del_icd).astype(int)
coh["delirium_cam"] = coh.patientunitstayid.isin(stay_cam).astype(int)
coh["first_cam_pos_min"] = coh.patientunitstayid.map(first_cam)
coh["cam_pos_by_24h"] = (coh.first_cam_pos_min <= 1440).astype(int)
coh.to_csv(OUT / "eicu_cohort.csv", index=False)                       # patient-level: stays on this Mac

A = pd.DataFrame(att, columns=["step", "icu_stays", "patients"])
A.to_csv(OUT / "eicu_attrition.csv", index=False)
print("\n=== eICU ATTRITION (no outcomes read) ===")
print(A.to_string(index=False))
n = len(coh)
print(f"\nFinal eICU cohort: {n:,} patients from {coh.hospitalid.nunique()} hospitals")
print("Not the first ICU visit of that hospital stay:", int((coh.visit_num > 1).sum()))
print("Cancer source  : ICD only", int(((coh.cancer_icd == 1) & (coh.cancer_dx_text == 0)).sum()),
      "| text only", int(((coh.cancer_icd == 0) & (coh.cancer_dx_text == 1)).sum()),
      "| both", int(((coh.cancer_icd == 1) & (coh.cancer_dx_text == 1)).sum()),
      "| also in past history", int(coh.cancer_pasthx.sum()))
print("Delirium source: CAM-ICU only", int(((coh.delirium_cam == 1) & (coh.delirium_icd == 0)).sum()),
      "| ICD only", int(((coh.delirium_cam == 0) & (coh.delirium_icd == 1)).sum()),
      "| both", int(((coh.delirium_cam == 1) & (coh.delirium_icd == 1)).sum()))
print(f"CAM-ICU positive by 24 h (intended-use group): {int(coh.cam_pos_by_24h.sum()):,} ({100*coh.cam_pos_by_24h.mean():.1f}%)")
print(f"Age median {coh.age_num.median():.0f} | male {100*(coh.gender=='Male').mean():.1f}%")
print(f"\nPre-set go/no-go rule (>= 300 patients): {'GO' if n >= 300 else 'NO-GO'}")
print("Cell 3 done - no outcome column was read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 4 (NEW): LOCK the external-validation analysis plan BEFORE any outcome is read
# Writes nb5/analysis_plan_locked.json with a timestamp + fingerprint of the cohort file.
# Every later cell checks this file and refuses to run if the cohort changed.
# ============================================================
import hashlib, datetime

coh_file = OUT / "eicu_cohort.csv"
fp = hashlib.md5(coh_file.read_bytes()).hexdigest()
coh = pd.read_csv(coh_file, dtype={"patientunitstayid": str})

PLAN = {
    "locked_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "cohort_file_md5": fp,
    "cohort_n": int(len(coh)),
    "hospitals": int(coh.hospitalid.nunique()),
    "intended_use_n": int(coh.cam_pos_by_24h.sum()),
    "outcome": ("Death in hospital within 28 days of ICU admission: patient.hospitaldischargestatus == 'Expired' "
                "AND hospitaldischargeoffset <= 40320 min (offsets are from ICU admission). eICU has no "
                "post-discharge death date, so deaths after hospital discharge are not captured (limitation)."),
    "primary_model": "final_random_split.joblib -> final_calibrated (Soft-Voting XGB+RF+CatBoost, isotonic), 25 predictors",
    "secondary_model": "final_temporal_split.joblib -> final_calibrated, 32 predictors",
    "predictors": "built from eICU first 24 h exactly as Notebook 2 (vitals 0-24 h, labs -6 to +24 h, same cleaning ranges, "
                  "same TPIFE thresholds/carry-forward caps); predictors not used by the model are not built",
    "missing_data": "frozen MIMIC preparation object (prep_<arm>.joblib): clip to MIMIC training limits, frozen MICE imputation; "
                    "NOTHING is refitted on eICU",
    "populations": {"primary": "all eICU cohort patients",
                    "secondary": "CAM-ICU positive within 24 h (intended-use group)"},
    "metrics": ["AUROC with DeLong 95% CI", "AUPRC", "Brier score",
                "calibration intercept and slope (logistic), E/O ratio, ICI, LOESS calibration plot",
                "locked threshold (0.421 random / 0.368 temporal): sensitivity, specificity, PPV, NPV (bootstrap CI)",
                "locked risk tiers (22.9% / 49.1%): observed mortality per tier with Wilson CI",
                "Venn-Abers uncertainty: share of patients whose interval crosses a tier cut-off",
                "decision curve: net benefit at 30% vs treat-all"],
    "comparators": ["SOFA and APACHE II (rebuilt from eICU first-24-h data with the frozen MIMIC logistic models), DeLong",
                    "eICU's own APACHE IVa predicted hospital mortality (apachePatientResult), discrimination only, as a reference"],
    "updating": ("Secondary only: logistic recalibration (intercept and slope) reported as an UPDATED model; "
                 "the primary result is always the frozen, unchanged model"),
    "sensitivity": ["ICD-confirmed cancer only (drop text-only cancer)",
                    "complete-case: patients with all 25 predictors measured",
                    "hospital-cluster bootstrap CI for AUROC (64 hospitals)"],
    "rules": ["The outcome is read once, in the evaluation cell, after predictors and predictions are saved",
              "No model, threshold, tier or preprocessing change after the outcome is read",
              "All results are reported whatever they show"],
}
plan_file = OUT / "analysis_plan_locked.json"
if plan_file.exists():
    old = json.loads(plan_file.read_text())
    assert old["cohort_file_md5"] == fp, "Cohort changed after the plan was locked - STOP and tell me"
    print("Plan already locked at", old["locked_at"], "- unchanged, not overwritten.")
else:
    plan_file.write_text(json.dumps(PLAN, indent=2))
    print("Analysis plan LOCKED at", PLAN["locked_at"])
print(f"Cohort fingerprint: {fp} | n = {PLAN['cohort_n']} | hospitals = {PLAN['hospitals']} | "
      f"intended-use n = {PLAN['intended_use_n']}")
print("Primary model:", PLAN["primary_model"])
print("Cell 4 done - no outcome read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 5 (NEW): extract first-day raw data for the eICU cohort - NO outcomes read
# Keeps cohort stays only, window -6 h to +24 h from ICU admission (offsets in minutes).
# Saves patient-level extracts in nb5/ (stay on this Mac) and prints label inventories (counts only).
# ============================================================
import time

plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
assert hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"], "cohort changed - STOP"
coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str})
STAYS = set(coh.patientunitstayid)
print(f"Cohort check OK: {len(STAYS)} stays (plan locked {plan['locked_at']})")

def extract(table, cols, offset_col, lo=-360, hi=1440, chunksize=3_000_000):
    t0, parts = time.time(), []
    for ch in pd.read_csv(EICU / f"{table}.csv.gz", usecols=cols, dtype=str, chunksize=chunksize):
        ch = ch[ch.patientunitstayid.isin(STAYS)]
        if len(ch) == 0:
            continue
        if offset_col:
            off = pd.to_numeric(ch[offset_col], errors="coerce")
            ch = ch[(off >= lo) & (off <= hi)]
        parts.append(ch)
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=cols)
    df.to_csv(OUT / f"extract_{table}.csv.gz", index=False)
    print(f"{table:20s} {len(df):>10,} rows | {df.patientunitstayid.nunique():>4} stays | {(time.time()-t0)/60:4.1f} min")
    return df

print("\n--- extracting (window -6 h to +24 h) ---")
vp = extract("vitalPeriodic", ["patientunitstayid", "observationoffset", "temperature", "sao2", "heartrate",
                               "respiration", "systemicsystolic", "systemicdiastolic", "systemicmean"],
             "observationoffset", lo=0)
va = extract("vitalAperiodic", ["patientunitstayid", "observationoffset", "noninvasivesystolic",
                                "noninvasivediastolic", "noninvasivemean"], "observationoffset", lo=0)
nc = extract("nurseCharting", ["patientunitstayid", "nursingchartoffset", "nursingchartcelltypecat",
                               "nursingchartcelltypevallabel", "nursingchartcelltypevalname", "nursingchartvalue"],
             "nursingchartoffset", lo=0)
lb = extract("lab", ["patientunitstayid", "labresultoffset", "labname", "labresult", "labmeasurenamesystem"],
             "labresultoffset", lo=-360)
inf = extract("infusionDrug", ["patientunitstayid", "infusionoffset", "drugname", "drugrate", "patientweight"],
              "infusionoffset", lo=-1440)                      # earlier rows kept to know the rate running at 0 h
io = extract("intakeOutput", ["patientunitstayid", "intakeoutputoffset", "cellpath", "celllabel", "cellvaluenumeric"],
             "intakeoutputoffset", lo=0)
rc = extract("respiratoryCharting", ["patientunitstayid", "respchartoffset", "respcharttypecat",
                                     "respchartvaluelabel", "respchartvalue"], "respchartoffset", lo=0)
rcare = extract("respiratoryCare", ["patientunitstayid", "respcarestatusoffset", "airwaytype", "ventstartoffset",
                                    "ventendoffset"], None)
trt = extract("treatment", ["patientunitstayid", "treatmentoffset", "treatmentstring"], "treatmentoffset", lo=-360)

# ---------- label inventories (counts only) ----------
def inv(df, cols, title, top=60, pattern=None):
    d = df
    if pattern:
        m = np.zeros(len(d), bool)
        for c in cols:
            m |= d[c].fillna("").str.contains(pattern, case=False, regex=True).values
        d = d[m]
    out = d.groupby(cols, dropna=False).size().sort_values(ascending=False).head(top)
    print(f"\n=== {title} ===")
    with pd.option_context("display.max_colwidth", 70, "display.width", 200):
        print(out.to_string())
    return out

inv(nc, ["nursingchartcelltypecat", "nursingchartcelltypevallabel", "nursingchartcelltypevalname"],
    "nurseCharting labels (top 60)")
inv(nc[nc.nursingchartcelltypevallabel.fillna("").str.contains("O2 Admin Device|O2 L/%|FiO2", case=False)],
    ["nursingchartcelltypevallabel", "nursingchartvalue"], "O2 device / FiO2 values in nurseCharting", top=40)
inv(lb, ["labname"], "lab names", top=80,
    pattern=r"WBC|lactate|creatinine|platelet|bilirubin|sodium|potassium|Hct|hematocrit|pH|paO2|paCO2|FiO2")
inv(inf, ["drugname"], "vasoactive infusions", top=40,
    pattern=r"norepi|levophed|epineph|dopamine|dobut|phenyleph|neo-syn|vasopressin")
inv(io, ["cellpath"], "urine output cells", top=30, pattern=r"urine|foley|void")
inv(rc, ["respchartvaluelabel"], "respiratoryCharting labels", top=30, pattern=r"FiO2|PEEP|mode|vent")
inv(rcare, ["airwaytype"], "airway types", top=20)
inv(trt, ["treatmentstring"], "ventilation / vasopressor treatments", top=30,
    pattern=r"mechanical ventilation|non-invasive|CPAP|BiPAP|vasopressor|norepinephrine|intubat")
w = pd.to_numeric(coh.admissionweight, errors="coerce")
print(f"\nAdmission weight available: {w.between(25, 300).sum()}/{len(coh)} | median {w[w.between(25,300)].median():.1f} kg")
print("\nCell 5 done - extracts saved in nb5/ (patient-level, do not share); no outcome read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 6 (NEW): build ALL Notebook-2 predictors for the eICU cohort - NO outcome read
# Same windows, cleaning ranges, TPIFE thresholds, SOFA / APACHE II / OSBI / interaction rules as Notebook 2.
# eICU-specific mapping (fixed before outcomes are seen):
#   vitals  : nurse-validated nurseCharting (~hourly, like MIMIC chartevents); if a stay has NO nurse value
#             for a variable, hourly medians of the automated vitalPeriodic / vitalAperiodic are used instead
#   labs    : lab table (blood gases assumed arterial - eICU has no specimen field)
#   FiO2    : respiratoryCharting FiO2 + lab FiO2 ; ventilation: O2 device, airway, ventilator charting, treatment
#   drugs   : infusionDrug charted rates (each rate valid until the next charted rate, max 24 h)
#   Elixhauser: comorbidipy (Quan, van Walraven) on the eICU diagnosis ICD-9 / ICD-10 codes
# Output: nb5/eicu_model_inputs.csv (patient-level, stays on this Mac) + comparison with MIMIC training data
# ============================================================
import warnings, re
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import polars as pl
from comorbidipy import comorbidity
from comorbidipy.calculators.comorbidity import colnames

plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
assert hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"], "cohort changed - STOP"
coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
ids = pd.Index(coh.index, name="stay_id")
N = len(ids)
num = lambda s: pd.to_numeric(s, errors="coerce")
def E(t):
    return pd.read_csv(OUT / f"extract_{t}.csv.gz", dtype=str)
nc, vp, va, lb = E("nurseCharting"), E("vitalPeriodic"), E("vitalAperiodic"), E("lab")
inf, io, rc, rcare, trt = E("infusionDrug"), E("intakeOutput"), E("respiratoryCharting"), E("respiratoryCare"), E("treatment")
log = []
def note(var, step, before, after):
    log.append({"variable": var, "step": step, "rows_before": before, "rows_after": after})

# ============================================================
# A. VITALS (0-24 h)
# ============================================================
NC_MAP = {"Heart Rate|Heart Rate": "HR", "Respiratory Rate|Respiratory Rate": "RR", "O2 Saturation|O2 Saturation": "SpO2",
          "Non-Invasive BP|Non-Invasive BP Systolic": "SBP", "Invasive BP|Invasive BP Systolic": "SBP",
          "Non-Invasive BP|Non-Invasive BP Diastolic": "DBP", "Invasive BP|Invasive BP Diastolic": "DBP",
          "Non-Invasive BP|Non-Invasive BP Mean": "MAP", "Invasive BP|Invasive BP Mean": "MAP",
          "Arterial Line MAP (mmHg)|Value": "MAP", "MAP (mmHg)|Value": "MAP",
          "Temperature|Temperature (C)": "Temp",
          "Glasgow coma score|GCS Total": "GCS_Total", "Score (Glasgow Coma Scale)|Value": "GCS_Total",
          "Glasgow coma score|Eyes": "GCS_Eye", "Glasgow coma score|Motor": "GCS_Motor", "Glasgow coma score|Verbal": "GCS_Verbal"}
n = nc.copy()
n["variable"] = (n.nursingchartcelltypevallabel.fillna("") + "|" + n.nursingchartcelltypevalname.fillna("")).map(NC_MAP)
n = n[n.variable.notna()]
n = pd.DataFrame({"stay_id": n.patientunitstayid, "hours": num(n.nursingchartoffset) / 60,
                  "variable": n.variable, "value": num(n.nursingchartvalue)}).dropna()

# automated monitor values -> hourly medians (used only where the nurse charting has no value for that variable)
VP = {"heartrate": "HR", "respiration": "RR", "sao2": "SpO2", "systemicsystolic": "SBP",
      "systemicdiastolic": "DBP", "systemicmean": "MAP", "temperature": "Temp"}
VA = {"noninvasivesystolic": "SBP", "noninvasivediastolic": "DBP", "noninvasivemean": "MAP"}
def melt(df, mp, off):
    m = df.melt(id_vars=["patientunitstayid", off], value_vars=list(mp), var_name="col", value_name="value")
    return pd.DataFrame({"stay_id": m.patientunitstayid, "hours": num(m[off]) / 60,
                         "variable": m.col.map(mp), "value": num(m.value)}).dropna()
auto = pd.concat([melt(vp, VP, "observationoffset"), melt(va, VA, "observationoffset")])
auto = auto[(auto.hours >= 0) & (auto.hours <= 24)]
auto["hr"] = np.floor(auto.hours.clip(0, 23.999))
auto = auto.groupby(["stay_id", "variable", "hr"], as_index=False).agg(hours=("hours", "median"), value=("value", "median"))
have = set(zip(n.stay_id, n.variable))
fill = auto.loc[np.array([(s, v) not in have for s, v in zip(auto.stay_id, auto.variable)], dtype=bool)].drop(columns="hr")
src = pd.Series({"nurse charting": len(n), "monitor fallback (hourly)": len(fill)})
v = pd.concat([n, fill], ignore_index=True)
v = v[(v.hours >= 0) & (v.hours <= 24)]
b = len(v); v = v.drop_duplicates(["stay_id", "hours", "variable", "value"]); note("all vitals", "exact duplicates", b, len(v))

t = v[v.variable == "Temp"].copy()
f_in_c = t.value > 50
t["value"] = np.where(f_in_c, (t.value - 32) * 5 / 9, t.value)
b = len(t); t = t[t.value.between(25, 42)]; note("Temp", "range 25-42 C (after F->C)", b, len(t))

g = v[v.variable.isin(["GCS_Eye", "GCS_Verbal", "GCS_Motor", "GCS_Total"])].copy()
lim = {"GCS_Eye": (1, 4), "GCS_Verbal": (1, 5), "GCS_Motor": (1, 6), "GCS_Total": (3, 15)}
b = len(g); g = g.loc[np.array([lim[x][0] <= y <= lim[x][1] for x, y in zip(g.variable, g.value)], dtype=bool)]; note("GCS", "valid ranges", b, len(g))
comp = g[g.variable != "GCS_Total"].pivot_table(index=["stay_id", "hours"], columns="variable", values="value", aggfunc="min").reset_index()
for c in ["GCS_Eye", "GCS_Verbal", "GCS_Motor"]:
    if c not in comp: comp[c] = np.nan
comp = comp.dropna(subset=["GCS_Eye", "GCS_Verbal", "GCS_Motor"])
comp["value"] = comp.GCS_Eye + comp.GCS_Verbal + comp.GCS_Motor
tot = g[g.variable == "GCS_Total"][["stay_id", "hours", "value"]]
key_t = set(zip(tot.stay_id, tot.hours))
comp_only = comp.loc[np.array([(s, h) not in key_t for s, h in zip(comp.stay_id, comp.hours)], dtype=bool)]
gcs_total = pd.concat([tot, comp_only[["stay_id", "hours", "value"]]]).assign(variable="GCS_Total")
gcs = pd.concat([g[g.variable != "GCS_Total"], gcs_total])
verbal1 = set(g[(g.variable == "GCS_Verbal") & (g.value == 1)].stay_id)

RANGES = {"HR": (20, 250), "SBP": (40, 300), "DBP": (20, 200), "MAP": (20, 200), "RR": (4, 60), "SpO2": (1, 100)}
o = v[v.variable.isin(RANGES)].copy()
for var_, (lo, hi) in RANGES.items():
    m = o.variable == var_; b = m.sum()
    o = o[~m | o.value.between(lo, hi)]; note(var_, f"range {lo}-{hi}", b, (o.variable == var_).sum())
sp = o[o.variable == "SpO2"].sort_values(["stay_id", "hours"])
drop_idx = []
for idx, r in sp[sp.value < 50].iterrows():
    nb = sp[(sp.stay_id == r.stay_id) & (sp.index != idx) & ((sp.hours - r.hours).abs() <= 1.0)]
    if len(nb) == 0 or (nb.value >= 85).all():
        drop_idx.append(idx)
b = (o.variable == "SpO2").sum(); o = o.drop(index=drop_idx); note("SpO2", "isolated <50% (artefact rule)", b, (o.variable == "SpO2").sum())

# FiO2 (fraction 0.21-1.0), 0-24 h
f1 = rc[rc.respchartvaluelabel.isin(["FiO2", "FIO2 (%)"])]
f1 = pd.DataFrame({"stay_id": f1.patientunitstayid, "hours": num(f1.respchartoffset) / 60, "value": num(f1.respchartvalue)})
f2 = lb[lb.labname == "FiO2"]
f2 = pd.DataFrame({"stay_id": f2.patientunitstayid, "hours": num(f2.labresultoffset) / 60, "value": num(f2.labresult)})
fi = pd.concat([f1, f2]).dropna()
fi = fi[(fi.hours >= 0) & (fi.hours <= 24)]
fi["value"] = np.where(fi.value > 1, fi.value / 100, fi.value)
b = len(fi); fi = fi[fi.value.between(0.21, 1.0)]; note("FiO2", "fraction 0.21-1.0", b, len(fi))
fio2_long = fi.assign(variable="FiO2")
vitals = pd.concat([o, t, gcs, fio2_long])[["stay_id", "hours", "variable", "value"]]

# ============================================================
# B. LABS (-6 h to +24 h)
# ============================================================
LABMAP = {"WBC x 1000": "WBC", "lactate": "Lactate", "creatinine": "Creatinine", "platelets x 1000": "Platelets",
          "total bilirubin": "Bilirubin", "sodium": "Sodium", "potassium": "Potassium", "Hct": "Hematocrit",
          "pH": "pH_art", "paO2": "PaO2_art", "paCO2": "PaCO2_art"}
l = lb[lb.labname.isin(LABMAP)]
l = pd.DataFrame({"stay_id": l.patientunitstayid, "hours": num(l.labresultoffset) / 60,
                  "variable": l.labname.map(LABMAP), "value": num(l.labresult)}).dropna()
l = l[(l.hours >= -6) & (l.hours <= 24)]
b = len(l); l = l.drop_duplicates(["stay_id", "hours", "variable", "value"]); note("all labs", "exact duplicates", b, len(l))
LAB_RANGES = {"WBC": (0, 500), "Lactate": (0.1, 30), "Creatinine": (0.1, 25), "Platelets": (1, 2000),
              "Bilirubin": (0.1, 70), "Sodium": (100, 180), "Potassium": (1.5, 10), "Hematocrit": (5, 75),
              "pH_art": (6.8, 7.8), "PaO2_art": (10, 700), "PaCO2_art": (5, 200)}
for var_, (lo, hi) in LAB_RANGES.items():
    m = l.variable == var_; b = m.sum()
    l = l[~m | l.value.between(lo, hi)]; note(var_, f"range {lo}-{hi}", b, (l.variable == var_).sum())
L = pd.concat([vitals, l], ignore_index=True)
L = L[L.stay_id.isin(ids)].sort_values(["stay_id", "variable", "hours"]).reset_index(drop=True)
assert L.hours.max() <= 24 + 1e-9

# ============================================================
# C. CONTEXT: weight, ventilation, oxygen, urine
# ============================================================
ctx = pd.DataFrame(index=ids)
w = num(coh.admissionweight).where(num(coh.admissionweight).between(25, 300))
pw = num(inf.patientweight); pw = pw.where(pw.between(25, 300))
pw = pd.Series(pw.values, index=inf.patientunitstayid).groupby(level=0).median()
ctx["weight_kg"] = w.combine_first(pw).reindex(ids)

dev = nc[nc.nursingchartcelltypevallabel == "O2 Admin Device"]
dev = pd.DataFrame({"stay_id": dev.patientunitstayid, "d": dev.nursingchartvalue.fillna("").str.strip().str.lower()})
INV_DEV = {"ventilator", "vent"}
NIV_DEV = {"bipap/cpap", "bipap", "cpap", "niv"}
NO_O2 = {"ra", "room air", "r/air", "none", ""}
air = rcare.copy()
air_ok = air.airwaytype.isin(["Oral ETT", "Nasal ETT", "Tracheostomy", "Double-Lumen Tube", "Cricothyrotomy"]) & \
         ((num(air.ventstartoffset) <= 1440) | num(air.respcarestatusoffset).between(-360, 1440))
ts = trt.treatmentstring.fillna("").str.lower()
tro = num(trt.treatmentoffset) <= 1440
inv_ids = set(dev[dev.d.isin(INV_DEV)].stay_id) | set(air[air_ok].patientunitstayid) | \
          set(rc[rc.respchartvaluelabel == "Vent Rate"].patientunitstayid) | \
          set(trt[tro & ts.str.contains("mechanical ventilation") & ~ts.str.contains("non-invasive")].patientunitstayid)
niv_ids = set(dev[dev.d.isin(NIV_DEV)].stay_id) | \
          set(trt[tro & ts.str.contains("non-invasive ventilation|cpap/peep therapy")].patientunitstayid)
o2_ids = set(dev[~dev.d.isin(NO_O2)].stay_id) | inv_ids | niv_ids | set(fio2_long[fio2_long.value > 0.21].stay_id)
ctx["invasive_vent_24h"] = ids.isin(inv_ids)
ctx["niv_24h"] = ids.isin(niv_ids)
ctx["oxygen_support_24h"] = ids.isin(o2_ids)
ctx["gcs_verbal_ETT_24h"] = ids.isin(inv_ids & verbal1)      # proxy: ventilated AND verbal score 1

u = io[io.cellpath.isin(["flowsheet|Flowsheet Cell Labels|I&O|Output (ml)|Urine",
                         "flowsheet|Flowsheet Cell Labels|I&O|Output (ml)|foley"])]
u = pd.DataFrame({"stay_id": u.patientunitstayid, "hours": num(u.intakeoutputoffset) / 60, "ml": num(u.cellvaluenumeric)}).dropna()
u = u[(u.hours >= 0) & (u.hours <= 24)].drop_duplicates()
ctx["urine_24h_ml"] = u.groupby("stay_id").ml.sum().clip(lower=0).reindex(ids)
span = u.groupby("stay_id").hours.agg(lambda s: s.max() - s.min()).reindex(ids)
ctx["urine_monitored_hours"] = span
ctx["urine_complete_24h"] = (span >= 20).fillna(False)

# ============================================================
# D. VASOACTIVE DRUGS: charted rates -> intervals -> standardised rate + NEE
# ============================================================
d = inf.copy()
d["drug"] = d.drugname.fillna("").str.extract(r"^\s*([A-Za-z]+)")[0].str.lower().map(
    {"norepinephrine": "Norepinephrine", "epinephrine": "Epinephrine", "dopamine": "Dopamine",
     "dobutamine": "Dobutamine", "phenylephrine": "Phenylephrine", "vasopressin": "Vasopressin"})
d = d[d.drug.notna()].copy()
d["unit"] = d.drugname.str.extract(r"\(([^()]*)\)\s*$")[0].fillna("").str.strip().str.lower()
d["rate"] = num(d.drugrate)
d["off"] = num(d.infusionoffset) / 60
d = d.dropna(subset=["off"]).sort_values(["patientunitstayid", "drugname", "off"])
d["end"] = d.groupby(["patientunitstayid", "drugname"]).off.shift(-1).fillna(24.0)
d = d[d.rate.fillna(1) > 0]                                  # rate 0 = infusion off
d["wt"] = d.patientunitstayid.map(ctx.weight_kg).fillna(num(d.patientweight))
volume = d.unit.str.contains("ml") | d.drugname.str.contains("Volume", case=False)
d["rate_std"] = np.select([d.unit.eq("mcg/kg/min"), d.unit.eq("mcg/min"), d.unit.eq("units/min"), d.unit.eq("units/hr")],
                          [d.rate, d.rate / d.wt, d.rate, d.rate / 60], default=np.nan)
d.loc[volume, "rate_std"] = np.nan                            # ml/hr without concentration: dose not usable
CAPD = {"Norepinephrine": 5, "Epinephrine": 5, "Phenylephrine": 15, "Dopamine": 25, "Dobutamine": 25, "Vasopressin": 0.1}
d.loc[d.rate_std > d.drug.map(CAPD), "rate_std"] = np.nan
NEE_FACTOR = {"Norepinephrine": 1, "Epinephrine": 1, "Dopamine": 0.01, "Phenylephrine": 0.1, "Vasopressin": 2.5}
d["nee"] = d.rate_std * d.drug.map(NEE_FACTOR)
d["start_h"] = d.off.clip(lower=0); d["end_h"] = d.end.clip(upper=24)
d = d[d.end_h > d.start_h]
VAC = d.rename(columns={"patientunitstayid": "stay_id"})[["stay_id", "drug", "start_h", "end_h", "rate_std", "nee"]]

# ============================================================
# E. STATIC: age, sex, cancer type, Elixhauser (comorbidipy, Quan, van Walraven) on eICU ICD codes
# ============================================================
dx = pd.read_csv(EICU / "diagnosis.csv.gz", dtype=str, usecols=["patientunitstayid", "diagnosisstring", "icd9code"])
dx = dx[dx.patientunitstayid.isin(ids)]
rows = []
for sid, field in zip(dx.patientunitstayid, dx.icd9code):
    if not isinstance(field, str) or not field.strip():
        continue
    parts = [p.strip().replace(".", "").upper() for p in field.split(",") if p.strip()]
    for i, c in enumerate(parts):
        ver = (9 if i == 0 else 10) if len(parts) == 2 else (10 if c[:1].isalpha() and c[:1] not in "EV" else 9)
        rows.append((sid, c, ver))
codes = pd.DataFrame(rows, columns=["stay_id", "icd_code", "ver"]).drop_duplicates()
ELIX = list(colnames["elixhauser"].keys())
key = {s: i for i, s in enumerate(ids)}
parts_ = []
for ver, tag in [(9, "icd9"), (10, "icd10")]:
    sub = codes[codes.ver == ver]
    if len(sub) == 0:
        continue
    pl_in = pl.DataFrame({"sid": sub.stay_id.map(key).astype("int64").tolist(), "icd_code": sub.icd_code.tolist()})
    res = comorbidity(pl_in, id_col="sid", code_col="icd_code", score="elixhauser", icd=tag,
                      variant="quan", weighting="van_walraven", assign0=True)
    parts_.append(pd.DataFrame(res.to_dict(as_series=False)))
elix = pd.concat(parts_).groupby("sid")[ELIX].max()
elix.index = [ids[i] for i in elix.index]
elix = elix.reindex(ids).fillna(0).astype(int)
elix.columns = ["elix_" + c for c in elix.columns]
ST = pd.DataFrame(index=ids)
ST["age_at_admission"] = num(coh.age_num).reindex(ids)
ST["sex_male"] = (coh.gender == "Male").astype(int).reindex(ids)

# cancer type by primary site (haematological first; metastatic / unknown codes do not decide the site)
def site_icd(c, ver):
    if ver == 10 and c[:1] == "C" and c[1:3].isdigit():
        k = int(c[1:3])
        if 81 <= k <= 96: return "Hematologic"
        if 15 <= k <= 26: return "Gastrointestinal"
        if 30 <= k <= 39: return "Respiratory/Thoracic"
        if 51 <= k <= 68: return "Genitourinary"
        if 76 <= k <= 80: return None
        return "Other/Unspecified"
    if ver == 9 and c[:3].isdigit():
        k = int(c[:3])
        if 200 <= k <= 208: return "Hematologic"
        if 150 <= k <= 159: return "Gastrointestinal"
        if 160 <= k <= 165: return "Respiratory/Thoracic"
        if 179 <= k <= 189: return "Genitourinary"
        if 195 <= k <= 199: return None
        if 140 <= k <= 239: return "Other/Unspecified" if k <= 195 else None
    return None
def site_text(s):
    s = s.lower()
    if re.search(r"hematologic malignancy|leukemia|lymphoma|myeloma|lymphoproliferative", s): return "Hematologic"
    if re.search(r"gi tumors|colon|esophag|gastric|pancrea|liver ca|hepatocellular|bile duct|s/p surgery for cancer\|colon|rectal", s): return "Gastrointestinal"
    if re.search(r"chest tumors\|(primary lung|metastatic lung|mediastinal)|lung cancer|lung parenchyma\|malignancy", s): return "Respiratory/Thoracic"
    if re.search(r"gu tumors|prostate|bladder|renal cell|ovarian|uterine|cervical|testicular", s): return "Genitourinary"
    if re.search(r"oncology\||malignan|cancer", s): return "Other/Unspecified"
    return None
sites = {}
for sid, c, ver in codes.itertuples(index=False):
    s = site_icd(c, ver)
    if s: sites.setdefault(sid, set()).add(s)
for sid, s in zip(dx.patientunitstayid, dx.diagnosisstring.fillna("")):
    st = site_text(s) if s else None
    if st: sites.setdefault(sid, set()).add(st)
ORDER = ["Hematologic", "Gastrointestinal", "Respiratory/Thoracic", "Genitourinary", "Other/Unspecified"]
def pick(sset):
    if not sset: return "Other/Unspecified"
    for o_ in ORDER[:-1]:
        if o_ in sset: return o_
    return "Other/Unspecified"
ST["cancer_type"] = [pick(sites.get(s, set())) for s in ids]
ST = ST.join(elix)

# ============================================================
# F. SOFA + modified APACHE II (identical rules to Notebook 2, hours instead of clock time)
# ============================================================
CTX = ctx
def var(name):
    return L[L.variable == name]
def worst(name, fn):
    return var(name).groupby("stay_id").value.agg(fn).reindex(ids)
abg = var("PaO2_art")[["stay_id", "hours", "value"]].rename(columns={"value": "pao2"})
pc = var("PaCO2_art")[["stay_id", "hours", "value"]].rename(columns={"value": "paco2"})
abg = abg.merge(pc, on=["stay_id", "hours"], how="left")
fio2 = var("FiO2")[["stay_id", "hours", "value"]].rename(columns={"value": "fio2", "hours": "fio2_time"})
abg = pd.merge_asof(abg.sort_values("hours"), fio2.sort_values("fio2_time"), left_on="hours", right_on="fio2_time",
                    by="stay_id", direction="backward", tolerance=4.0)
no_support = ~abg.stay_id.map(CTX.oxygen_support_24h).fillna(False).astype(bool)
abg.loc[abg.fio2.isna() & no_support, "fio2"] = 0.21
abg["pf"] = abg.pao2 / abg.fio2
support = (CTX.invasive_vent_24h | CTX.niv_24h).reindex(ids)
spx = var("SpO2")[["stay_id", "hours", "value"]].rename(columns={"value": "spo2"})
spx = spx[spx.spo2 <= 97]
spx = pd.merge_asof(spx.sort_values("hours"), fio2.sort_values("fio2_time"), left_on="hours", right_on="fio2_time",
                    by="stay_id", direction="backward", tolerance=1.0)
spx.loc[spx.fio2.isna() & ~spx.stay_id.map(CTX.oxygen_support_24h).fillna(False).astype(bool), "fio2"] = 0.21
spx["sf"] = spx.spo2 / spx.fio2
def resp_from_ratio(r, cuts, supp):
    if pd.isna(r): return np.nan
    if r < cuts[3] and supp: return 4
    if r < cuts[2] and supp: return 3
    if r < cuts[2]: return 2
    if r < cuts[1]: return 2
    if r < cuts[0]: return 1
    return 0
min_pf = abg.groupby("stay_id").pf.min().reindex(ids)
min_sf = spx.groupby("stay_id").sf.min().reindex(ids)
resp_pf = [resp_from_ratio(r, (400, 300, 200, 100), s) for r, s in zip(min_pf, support)]
resp_sf = [resp_from_ratio(r, (512, 357, 214, 89), s) for r, s in zip(min_sf, support)]
resp = pd.Series(np.where(pd.notna(resp_pf), resp_pf, resp_sf), index=ids).astype(float)
plt_ = worst("Platelets", "min"); coag = plt_.apply(lambda x: np.nan if pd.isna(x) else 4 if x < 20 else 3 if x < 50 else 2 if x < 100 else 1 if x < 150 else 0)
bili = worst("Bilirubin", "max"); liver = bili.apply(lambda x: np.nan if pd.isna(x) else 4 if x >= 12 else 3 if x >= 6 else 2 if x >= 2 else 1 if x >= 1.2 else 0)
gcs_min = worst("GCS_Total", "min"); cns = gcs_min.apply(lambda x: np.nan if pd.isna(x) else 0 if x >= 15 else 1 if x >= 13 else 2 if x >= 10 else 3 if x >= 6 else 4)
cr = worst("Creatinine", "max")
cr_pts = cr.apply(lambda x: np.nan if pd.isna(x) else 4 if x >= 5 else 3 if x >= 3.5 else 2 if x >= 2 else 1 if x >= 1.2 else 0)
uo_ = CTX.urine_24h_ml.reindex(ids); uo_ok = CTX.urine_complete_24h.reindex(ids).fillna(False).astype(bool)
uo_pts = pd.Series(np.where(uo_ok & (uo_ < 200), 4, np.where(uo_ok & (uo_ < 500), 3, np.nan)), index=ids)
renal = pd.concat([cr_pts, uo_pts], axis=1).max(axis=1)
mx = VAC.groupby(["stay_id", "drug"]).rate_std.max().unstack().reindex(ids)
for dname in ["Norepinephrine", "Epinephrine", "Dopamine", "Dobutamine"]:
    if dname not in mx: mx[dname] = np.nan
map_min = worst("MAP", "min")
def cardio(i):
    ne, ep, dp, db, m = mx.loc[i, "Norepinephrine"], mx.loc[i, "Epinephrine"], mx.loc[i, "Dopamine"], mx.loc[i, "Dobutamine"], map_min.loc[i]
    if (dp > 15) or (ep > 0.1) or (ne > 0.1): return 4
    if (dp > 5) or (0 < ep <= 0.1) or (0 < ne <= 0.1): return 3
    if (0 < dp <= 5) or (db > 0): return 2
    if pd.notna(m): return 1 if m < 70 else 0
    return np.nan
cvs = pd.Series([cardio(i) for i in ids], index=ids)
sofa = pd.DataFrame({"sofa_resp": resp, "sofa_coag": coag, "sofa_liver": liver, "sofa_cardio": cvs,
                     "sofa_cns": cns, "sofa_renal": renal})
scomp = list(sofa.columns)
sofa["sofa_total"] = sofa[scomp].fillna(0).sum(axis=1)

def worst_points(name, fn):
    dd = var(name).copy(); dd["p"] = dd.value.apply(fn)
    return dd.groupby("stay_id").p.max().reindex(ids)
def p_temp(t): return 4 if t >= 41 or t < 30 else 3 if t >= 39 or t < 32 else 2 if t < 34 else 1 if t >= 38.5 or t < 36 else 0
def p_map(m): return 4 if m >= 160 or m < 50 else 3 if m >= 130 else 2 if m >= 110 or m < 70 else 0
def p_hr(h): return 4 if h >= 180 or h < 40 else 3 if h >= 140 or h < 55 else 2 if h >= 110 or h < 70 else 0
def p_rr(r): return 4 if r >= 50 or r < 6 else 3 if r >= 35 else 2 if r < 10 else 1 if r >= 25 or r < 12 else 0
def p_ph(x): return 4 if x >= 7.7 or x < 7.15 else 3 if x >= 7.6 or x < 7.25 else 2 if x < 7.33 else 1 if x >= 7.5 else 0
def p_na(x): return 4 if x >= 180 or x <= 110 else 3 if x >= 160 or x < 120 else 2 if x >= 155 or x < 130 else 1 if x >= 150 else 0
def p_k(x): return 4 if x >= 7 or x < 2.5 else 3 if x >= 6 else 2 if x < 3 else 1 if x >= 5.5 or x < 3.5 else 0
def p_cr(x): return 4 if x >= 3.5 else 3 if x >= 2 else 2 if x >= 1.5 or x < 0.6 else 0
def p_hct(x): return 4 if x >= 60 or x < 20 else 2 if x >= 50 or x < 30 else 1 if x >= 46 else 0
def p_wbc(x): return 4 if x >= 40 or x < 1 else 2 if x >= 20 or x < 3 else 1 if x >= 15 else 0
def p_oxy(r):
    if pd.isna(r.fio2) or pd.isna(r.pao2): return np.nan
    if r.fio2 >= 0.5:
        if pd.isna(r.paco2): return np.nan
        a = r.fio2 * 713 - r.paco2 / 0.8 - r.pao2
        return 4 if a >= 500 else 3 if a >= 350 else 2 if a >= 200 else 0
    return 0 if r.pao2 > 70 else 1 if r.pao2 >= 61 else 3 if r.pao2 >= 55 else 4
abg["oxy_p"] = abg.apply(p_oxy, axis=1) if len(abg) else np.nan
apx = pd.DataFrame({
    "ap_temp": worst_points("Temp", p_temp), "ap_map": worst_points("MAP", p_map), "ap_hr": worst_points("HR", p_hr),
    "ap_rr": worst_points("RR", p_rr), "ap_oxygenation": abg.groupby("stay_id").oxy_p.max().reindex(ids),
    "ap_ph": worst_points("pH_art", p_ph), "ap_sodium": worst_points("Sodium", p_na), "ap_potassium": worst_points("Potassium", p_k),
    "ap_creatinine": worst_points("Creatinine", p_cr), "ap_hct": worst_points("Hematocrit", p_hct), "ap_wbc": worst_points("WBC", p_wbc),
    "ap_gcs": (15 - gcs_min)})
apc = list(apx.columns)
apx["apache_aps"] = apx[apc].fillna(0).sum(axis=1)
apx["apache_age_pts"] = ST.age_at_admission.apply(lambda a: 0 if a <= 44 else 2 if a <= 54 else 3 if a <= 64 else 5 if a <= 74 else 6)
immuno = (ST.cancer_type.eq("Hematologic") | ST.elix_lymph.eq(1) | ST.elix_aids.eq(1))
organ = (ST.elix_ld.eq(1) | ST.elix_chf.eq(1) | ST.elix_cpd.eq(1) | ST.elix_rf.eq(1))
apx["apache_chronic_pts"] = np.where(immuno | organ, 5, 0)
apx["apache_ii_total"] = apx.apache_aps + apx.apache_age_pts + apx.apache_chronic_pts
SC = pd.concat([sofa, apx], axis=1)

# ============================================================
# G. BASIC + TPIFE features (identical to Notebook 2 Step 7)
# ============================================================
VITALS = ["HR", "MAP", "SBP", "RR", "SpO2", "Temp", "GCS_Total"]
LABS = ["Lactate", "Creatinine", "Platelets", "Bilirubin", "WBC", "Sodium", "Potassium",
        "Hematocrit", "pH_art", "PaO2_art", "PaCO2_art"]
TH = {"HR": (40, 60, 100, 130), "MAP": (55, 65, 110, 130), "SBP": (80, 90, 160, 180), "RR": (8, 12, 20, 30),
      "SpO2": (88, 92, None, None), "Temp": (35.0, 36.0, 38.0, 39.0), "GCS_Total": (9, 13, None, None),
      "Lactate": (None, None, 2.0, 4.0), "Creatinine": (None, None, 1.2, 2.0), "Platelets": (50, 150, None, None),
      "Bilirubin": (None, None, 1.2, 2.0), "WBC": (2, 4, 12, 20), "Sodium": (125, 135, 145, 155),
      "Potassium": (2.5, 3.5, 5.0, 6.0), "Hematocrit": (21, 30, None, None), "pH_art": (7.20, 7.35, 7.45, 7.55),
      "PaO2_art": (55, 80, None, None), "PaCO2_art": (25, 35, 45, 60)}
thf = pd.read_csv(ROOT / "data" / "processed_v2" / "tpife_thresholds.csv").set_index("variable")
for k_, (a_, b_, c_, d_) in TH.items():           # guard: identical to the frozen thresholds file
    row = thf.loc[k_]
    assert all((pd.isna(x) and y is None) or (pd.notna(x) and float(x) == float(y))
               for x, y in zip(row[["severe_low", "mild_low", "mild_high", "severe_high"]], (a_, b_, c_, d_))), k_
CAP = {v_: 3.0 for v_ in VITALS}; CAP.update({"Temp": 6.0, "GCS_Total": 6.0})
SEGS = [(0, 6, "early"), (6, 12, "mid"), (12, 24, "late")]
def state(x, th):
    sl, ml, mh, sh = th
    s = np.zeros(len(x), dtype=int)
    if ml is not None: s[x < ml] = 1
    if mh is not None: s[x > mh] = 1
    if sl is not None: s[x < sl] = 2
    if sh is not None: s[x > sh] = 2
    return s
def deviation(x, th):
    _, ml, mh, _ = th
    dv = np.zeros(len(x))
    if ml is not None: dv = np.maximum(dv, ml - x)
    if mh is not None: dv = np.maximum(dv, x - mh)
    return dv
def basic(t, x, p):
    f = {f"{p}_n": len(x), f"{p}_mean": x.mean(), f"{p}_min": x.min(), f"{p}_max": x.max(),
         f"{p}_std": x.std(ddof=1) if len(x) > 1 else np.nan, f"{p}_first": x[0], f"{p}_last": x[-1],
         f"{p}_delta": x[-1] - x[0] if len(x) > 1 else np.nan, f"{p}_slope": np.nan}
    if len(x) >= 3 and np.ptp(t) > 0:
        f[f"{p}_slope"] = np.polyfit(t, x, 1)[0]
    return f
def tpife(t, x, v_):
    th = TH[v_]; s = state(x, th); dev_ = deviation(x, th)
    end = np.minimum(np.append(t[1:], 24.0), t + CAP[v_]); end = np.minimum(end, 24.0)
    dur = np.clip(end - t, 0, None)
    obs = dur.sum()
    f = {f"{v_}_obs_hours": obs}
    if obs <= 0:
        return f
    t_norm, t_mild, t_sev = dur[s == 0].sum(), dur[s == 1].sum(), dur[s == 2].sum()
    tot_ = t_norm + t_mild + t_sev
    f[f"{v_}_frac_abn"] = (t_mild + t_sev) / tot_
    f[f"{v_}_frac_severe"] = t_sev / tot_
    f[f"{v_}_auc_dev"] = (dev_ * dur).sum() / obs
    gap = np.append(t[1:] > end[:-1] + 1e-9, False)
    trans = int(((s[1:] != s[:-1]) & ~gap[:-1]).sum())
    f[f"{v_}_transitions_per24h"] = trans * 24 / obs
    ep, cur, eps = [], 0.0, 0
    first_ep_dur, first_done = 0.0, False
    for i in range(len(x)):
        if s[i] >= 1:
            if cur == 0: eps += 1
            cur += dur[i]
        if cur > 0 and (s[i] == 0 or gap[i] or i == len(x) - 1):
            ep.append(cur)
            if not first_done: first_ep_dur, first_done = cur, True
            cur = 0.0
    f[f"{v_}_episodes_per24h"] = eps * 24 / obs
    f[f"{v_}_longest_episode_h"] = max(ep) if ep else 0.0
    f[f"{v_}_first_episode_h"] = first_ep_dur
    f[f"{v_}_unresolved"] = int(s[-1] >= 1)
    seg_frac = {}
    for a, b_, name in SEGS:
        ov = np.clip(np.minimum(end, b_) - np.maximum(t, a), 0, None)
        if ov.sum() > 0:
            f[f"{v_}_{name}_mean"] = (x * ov).sum() / ov.sum()
            o_abn, o_norm = ov[s >= 1].sum(), ov[s == 0].sum()
            seg_frac[name] = o_abn / (o_abn + o_norm)
    if "early" in seg_frac and "late" in seg_frac:
        f[f"{v_}_late_minus_early_mean"] = f[f"{v_}_late_mean"] - f[f"{v_}_early_mean"]
        f[f"{v_}_deterioration"] = seg_frac["late"] - seg_frac["early"]
    return f
Lw = L[((L.variable.isin(VITALS)) & (L.hours >= 0)) | ((L.variable.isin(LABS)) & (L.hours >= -6))]
rows_ = {sid: {} for sid in ids}
for (sid, v_), gg in Lw.sort_values(["stay_id", "variable", "hours"]).groupby(["stay_id", "variable"], sort=False):
    if v_ not in VITALS and v_ not in LABS:
        continue
    gg = gg.groupby("hours", as_index=False).value.mean()
    t_, x_ = gg.hours.to_numpy(float), gg.value.to_numpy(float)
    f = basic(t_, x_, v_)
    if v_ in VITALS:
        f.update(tpife(t_, x_, v_))
    else:
        s = state(x_, TH[v_])
        f.update({f"{v_}_ever_abn": int((s >= 1).any()), f"{v_}_ever_severe": int((s == 2).any()), f"{v_}_last_state": int(s[-1])})
    rows_[sid].update(f)
TF = pd.DataFrame.from_dict(rows_, orient="index").reindex(ids)
fa = TF[[f"{v_}_frac_abn" for v_ in VITALS if f"{v_}_frac_abn" in TF]]
TF["tpife_mean_frac_abn"] = fa.mean(axis=1)
TF["tpife_n_vitals_ever_severe"] = (TF[[f"{v_}_frac_severe" for v_ in VITALS if f"{v_}_frac_severe" in TF]] > 0).sum(axis=1)
TF["tpife_n_vitals_deteriorating"] = (TF[[f"{v_}_deterioration" for v_ in VITALS if f"{v_}_deterioration" in TF]] > 0).sum(axis=1)
TF["tpife_n_vitals_unresolved"] = TF[[f"{v_}_unresolved" for v_ in VITALS if f"{v_}_unresolved" in TF]].sum(axis=1)
TF["tpife_n_vitals_observed"] = fa.notna().sum(axis=1)
for v_ in VITALS:
    for c, lo, hi in [(f"{v_}_frac_abn", 0, 1), (f"{v_}_frac_severe", 0, 1), (f"{v_}_deterioration", -1, 1)]:
        if c in TF: TF[c] = TF[c].clip(lo, hi)

# ============================================================
# H. ORGAN-SYSTEM BURDEN + stability (identical to Notebook 2 Step 8)
# ============================================================
PRESSORS = ["Norepinephrine", "Epinephrine", "Dopamine", "Phenylephrine", "Vasopressin"]
V = VAC.copy(); V["s"] = V.start_h.clip(0, 24); V["e"] = V.end_h.clip(0, 24); V = V[V.e > V.s]
def pressor_summary(gp):
    p = gp[gp.drug.isin(PRESSORS)]
    out = {"vaso_any": int(len(p) > 0), "dobutamine_any": int((gp.drug == "Dobutamine").any()), "n_vasoactive_agents": gp.drug.nunique()}
    if len(p) == 0:
        out.update({"nee_max": 0.0, "nee_mean_24h": 0.0, "vaso_hours": 0.0}); return pd.Series(out)
    iv = p[["s", "e"]].sort_values("s").to_numpy()
    hours, cs, ce_ = 0.0, iv[0, 0], iv[0, 1]
    for s, e in iv[1:]:
        if s > ce_:
            hours += ce_ - cs; cs, ce_ = s, e
        else:
            ce_ = max(ce_, e)
    hours += ce_ - cs
    q = p[p.nee.notna()]
    if len(q) == 0:
        nee_max, nee_mean = np.nan, np.nan
    else:
        cuts = np.unique(np.r_[q.s, q.e]); mids = (cuts[:-1] + cuts[1:]) / 2
        totv = [q.nee[(q.s <= m) & (q.e > m)].sum() for m in mids]
        nee_max = float(max(totv)) if totv else 0.0
        nee_mean = float((q.nee * (q.e - q.s)).sum() / 24)
    out.update({"nee_max": nee_max, "nee_mean_24h": nee_mean, "vaso_hours": hours})
    return pd.Series(out)
cols_v = ["vaso_any", "dobutamine_any", "n_vasoactive_agents", "nee_max", "nee_mean_24h", "vaso_hours"]
vaso = pd.DataFrame([pressor_summary(gp).rename(sid) for sid, gp in V.groupby("stay_id")], columns=cols_v).reindex(ids)
fill0 = {"vaso_any": 0, "dobutamine_any": 0, "n_vasoactive_agents": 0, "nee_max": 0.0, "nee_mean_24h": 0.0, "vaso_hours": 0.0}
none_rows = ~ids.isin(V.stay_id)
vaso.loc[none_rows] = vaso.loc[none_rows].fillna(fill0)
o2 = CTX.oxygen_support_24h.astype(bool)
fio2_max = fio2.groupby("stay_id").fio2.max().reindex(ids)
fio2_max[fio2_max.isna() & ~o2] = 0.21
pf_min = (abg.pao2 / abg.fio2).groupby(abg.stay_id).min().reindex(ids)
sf_min = (spx.spo2 / spx.fio2).groupby(spx.stay_id).min().reindex(ids)
inv_ = CTX.invasive_vent_24h.astype(bool); niv_ = CTX.niv_24h.astype(bool)
support_level = np.select([inv_, niv_, o2], [3, 2, 1], 0)
uo_rate = (CTX.urine_24h_ml / CTX.weight_kg / CTX.urine_monitored_hours).where(CTX.urine_complete_24h.astype(bool))
def low_time_burden(name, mild, severe, cap=3.0):
    res = {}
    for sid, gp in var(name).sort_values("hours").groupby("stay_id"):
        gp = gp.groupby("hours", as_index=False).value.mean()
        t_, x_ = gp.hours.to_numpy(float), gp.value.to_numpy(float)
        end = np.minimum(np.minimum(np.append(t_[1:], 24.0), t_ + cap), 24.0)
        dur = np.clip(end - t_, 0, None)
        m = dur[(x_ < mild) & (x_ >= severe)].sum(); s = dur[x_ < severe].sum(); nn = dur[x_ >= mild].sum()
        tt = m + s + nn
        if tt > 0:
            res[sid] = ((0.5 * m + s) / tt, int((x_ < severe).any()), int((x_ < mild).any()))
    return pd.DataFrame.from_dict(res, orient="index", columns=["b", "sev", "any"]).reindex(ids)
def level(x, mild, severe, high=True):
    if high:
        return np.where(x.isna(), np.nan, np.where(x > severe, 1, np.where(x > mild, 0.5, 0)))
    return np.where(x.isna(), np.nan, np.where(x < severe, 1, np.where(x < mild, 0.5, 0)))
mapb = low_time_burden("MAP", 65, 55); spob = low_time_burden("SpO2", 92, 88); gcsb = low_time_burden("GCS_Total", 13, 9, cap=6.0)
nee = vaso.nee_max.astype(float)
vaso_lvl = np.where(vaso.vaso_any == 0, 0.0, np.where(nee > 0.1, 1.0, 0.5))
sup_lvl = pd.Series(support_level, index=ids).map({0: 0.0, 1: 0.0, 2: 0.5, 3: 1.0})
uo_lvl = np.where(uo_rate.isna(), np.nan, np.where(uo_rate < 0.3, 1.0, np.where(uo_rate < 0.5, 0.5, 0.0)))
def combine(*parts):
    return pd.concat([pd.Series(np.asarray(p, float), index=ids) for p in parts], axis=1).max(axis=1)
SYS = {"cardiovascular": combine(mapb.b, vaso_lvl),
       "respiratory": combine(spob.b, sup_lvl, level(pf_min, 300, 200, high=False)),
       "renal": combine(level(worst("Creatinine", "max"), 1.2, 2.0), uo_lvl),
       "hepatic": combine(level(worst("Bilirubin", "max"), 1.2, 2.0)),
       "coagulation": combine(level(worst("Platelets", "min"), 150, 50, high=False)),
       "neurological": combine(gcsb.b),
       "metabolic": combine(level(worst("Lactate", "max"), 2.0, 4.0), level(worst("pH_art", "min"), 7.35, 7.20, high=False))}
O = pd.DataFrame({f"osbi_{k}": v_ for k, v_ in SYS.items()})
sys_cols = list(O.columns)
O["osbi_total"] = O[sys_cols].fillna(0).sum(axis=1)
O["osbi_n_systems_involved"] = (O[sys_cols] > 0).sum(axis=1)
sev_df = pd.DataFrame({"cardiovascular": (nee > 0.1) | (mapb.sev == 1),
                       "respiratory": inv_ | (pf_min < 200) | (spob.sev == 1),
                       "renal": (worst("Creatinine", "max") > 2.0) | (uo_rate < 0.3),
                       "hepatic": worst("Bilirubin", "max") > 2.0, "coagulation": worst("Platelets", "min") < 50,
                       "neurological": gcsb.sev == 1,
                       "metabolic": (worst("Lactate", "max") > 4.0) | (worst("pH_art", "min") < 7.20)})
O["osbi_n_systems_severe"] = sev_df.sum(axis=1).astype(int)
stab = {}
for v_ in ["HR", "MAP", "RR", "SpO2"]:
    dd = var(v_).copy()
    dd["hr"] = np.floor(dd.hours.clip(0, 23.999)).astype(int)
    h = dd.groupby(["stay_id", "hr"]).value.median().reset_index()
    h["dv"] = h.groupby("stay_id").value.diff().abs(); h["dh"] = h.groupby("stay_id").hr.diff()
    stab[f"stab_{v_}_arv"] = h[h.dh == 1].groupby("stay_id").dv.mean().reindex(ids)
    gv = var(v_).groupby("stay_id").value
    stab[f"stab_{v_}_cv"] = (gv.std() / gv.mean()).reindex(ids)
OS = pd.concat([vaso.astype(float), pd.DataFrame({"fio2_max": fio2_max, "pf_min": pf_min, "sf_min": sf_min,
                "resp_support_level": support_level, "uo_ml_kg_h": uo_rate}, index=ids), O, pd.DataFrame(stab)], axis=1)

# ============================================================
# I. SIRS + interactions (identical to Notebook 2 Step 9)
# ============================================================
crit = pd.DataFrame({"sirs_temp": (worst("Temp", "max") > 38) | (worst("Temp", "min") < 36),
                     "sirs_hr": worst("HR", "max") > 90,
                     "sirs_resp": (worst("RR", "max") > 20) | (worst("PaCO2_art", "min") < 32),
                     "sirs_wbc": (worst("WBC", "max") > 12) | (worst("WBC", "min") < 4)}).astype(int)
sirs = crit.sum(axis=1).astype(int)
def hourly(name):
    dd = var(name); dd = dd[dd.hours >= 0].assign(hr=lambda z: np.floor(z.hours.clip(0, 23.999)).astype(int))
    gh = dd.groupby(["stay_id", "hr"]).value.median().unstack()
    return gh.reindex(index=ids, columns=range(24)).ffill(axis=1, limit=2)
H = {v_: hourly(v_) for v_ in ["MAP", "HR", "SpO2", "SBP"]}
def hours_where(mask, *need):
    known = np.logical_and.reduce([h.notna() for h in need])
    out = (mask & known).sum(axis=1).astype(float)
    out[~known.any(axis=1)] = np.nan
    return out
hypo_h = hours_where(H["MAP"] < 65, H["MAP"])
hypox_tachy_h = hours_where((H["SpO2"] < 92) & (H["HR"] > 100), H["SpO2"], H["HR"])
si = H["HR"] / H["SBP"]
crs = var("Creatinine").sort_values("hours").groupby("stay_id").value
cr_rise = (crs.last() - crs.first()).clip(lower=0).reindex(ids)
cr_rise[crs.count().reindex(ids) < 2] = np.nan
oliguria = np.where(OS.uo_ml_kg_h.isna(), np.nan, (OS.uo_ml_kg_h < 0.5).astype(float))
IX = pd.DataFrame({"sirs_count": sirs, **crit, "sirs_x_sofa": sirs * SC.sofa_total,
                   "hypotension_hours": hypo_h, "hypotension_x_lactate": hypo_h * worst("Lactate", "max"),
                   "hypoxaemia_tachycardia_hours": hypox_tachy_h, "creat_rise": cr_rise,
                   "creat_rise_x_oliguria": cr_rise * oliguria, "shock_index_max": si.max(axis=1)}, index=ids)

# ============================================================
# J. ASSEMBLE exactly like Notebook 2 Step 10
# ============================================================
ELIX_ACUTE = ["elix_fed", "elix_carit", "elix_coag", "elix_ond", "elix_wloss", "elix_pcd"]
VW = {"chf": 7, "carit": 5, "valv": -1, "pcd": 4, "pvd": 2, "para": 7, "ond": 6, "cpd": 3, "rf": 5, "ld": 11,
      "lymph": 9, "metacanc": 12, "solidtum": 4, "coag": 3, "obes": -4, "wloss": 6, "fed": 5, "blane": -2,
      "dane": -2, "drug": -7, "depre": -3}
ST["elix_vw_chronic"] = sum(ST["elix_" + k] * w_ for k, w_ in VW.items() if "elix_" + k not in ELIX_ACUTE)
ST["elix_diabetes"] = ST[["elix_diabc", "elix_diabunc"]].max(axis=1)
ST["elix_hypertension"] = ST[["elix_hypc", "elix_hypunc"]].max(axis=1)
MERGED = ["elix_diabc", "elix_diabunc", "elix_hypc", "elix_hypunc"]
elix_cols = [c for c in ST.columns if c.startswith("elix_") and c not in ELIX_ACUTE + MERGED + ["elix_vw_score"]]
sev_cols = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cardio", "sofa_cns", "sofa_renal", "sofa_total",
            "ap_temp", "ap_map", "ap_hr", "ap_rr", "ap_oxygenation", "ap_ph", "ap_sodium", "ap_potassium",
            "ap_creatinine", "ap_hct", "ap_wbc", "ap_gcs", "apache_aps", "apache_age_pts", "apache_chronic_pts", "apache_ii_total"]
X = pd.concat([ST[["age_at_admission", "sex_male", "cancer_type"] + elix_cols], SC[sev_cols],
               CTX[["weight_kg", "gcs_verbal_ETT_24h"]].astype({"gcs_verbal_ETT_24h": float}),
               TF, OS, IX], axis=1).reindex(ids)
X.loc[X.pf_min > 700, "pf_min"] = np.nan
X.index.name = "patientunitstayid"

# ---------- compare with what the frozen MIMIC preparation expects ----------
prep_r = read_retry(MODELS / "prep_random_split.joblib", mode="joblib")
need = [c for c in prep_r.cols if not c.startswith("cancer_")]
fixed = {"Lactate_measured"} | {f"{v_}_max_dev" for v_ in ["HR", "MAP", "RR", "SpO2"]}
missing_cols = [c for c in need if c not in X.columns and c not in fixed]
extra_cols = [c for c in X.columns if c not in need and c != "cancer_type"]
for c in missing_cols:
    X[c] = np.nan
X.to_csv(OUT / "eicu_model_inputs.csv")                       # patient-level: stays on this Mac
pd.DataFrame(log).to_csv(OUT / "eicu_cleaning_log.csv", index=False)

print(f"eICU predictor table: {X.shape[0]} patients x {X.shape[1]} columns")
print("Vital-sign rows by source:", src.to_dict())
print(f"Columns the MIMIC preparation expects but eICU could not build: {len(missing_cols)} {missing_cols[:15]}")
print(f"Extra columns (ignored by the preparation): {len(extra_cols)} {extra_cols[:10]}")
print("Cancer type:", X.cancer_type.value_counts().to_dict())
print(f"Invasive ventilation in 24 h: {int(CTX.invasive_vent_24h.sum())} | NIV: {int(CTX.niv_24h.sum())} | "
      f"vasopressor: {int(OS.vaso_any.sum())} | urine complete: {int(CTX.urine_complete_24h.sum())}")

# ---------- the 25 + 32 model predictors: eICU vs MIMIC training ----------
feats = list(dict.fromkeys(bundles["final_random_split"]["features"] + bundles["final_temporal_split"]["features"]))
mimic_mean = pd.Series(prep_r.imp.initial_imputer_.statistics_, index=prep_r.cols)
cmp = pd.DataFrame({"in_primary_25": [f in bundles["final_random_split"]["features"] for f in feats],
                    "eICU_missing_%": [100 * X[f].isna().mean() if f in X else np.nan for f in feats],
                    "MIMIC_missing_%": [100 * prep_r.train_missing.get(f, np.nan) for f in feats],
                    "eICU_median": [X[f].median() if f in X else np.nan for f in feats],
                    "MIMIC_mean": [mimic_mean.get(f, np.nan) for f in feats],
                    "MIMIC_p0.5": [prep_r.lo.get(f, np.nan) for f in feats],
                    "MIMIC_p99.5": [prep_r.hi.get(f, np.nan) for f in feats]}, index=feats)
cmp.to_csv(OUT / "eicu_vs_mimic_predictors.csv")
with pd.option_context("display.width", 200, "display.max_rows", 100):
    print("\n=== model predictors: eICU (raw) vs MIMIC training ===")
    print(cmp.round(3).to_string())
print("\nCell 6 done - predictors built; no outcome read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 6b (NEW): comorbidity harmonisation check - NO outcome read
# eICU ICD codes come from the ICU problem list, not full hospital discharge coding (MIMIC).
# Here: Elixhauser flags used by the model (van Walraven chronic categories) are ALSO taken from
# eICU pastHistory (present on admission) and the diagnosis text, and prevalence is compared.
# Prints prevalence + every matched label, for checking. Nothing is decided in this cell.
# ============================================================
import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")
X = pd.read_csv(OUT / "eicu_model_inputs.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
ids = X.index
ph = pd.read_csv(EICU / "pastHistory.csv.gz", dtype=str, usecols=["patientunitstayid", "pasthistorypath"])
ph = ph[ph.patientunitstayid.isin(ids)].copy()
ph["p"] = ph.pasthistorypath.fillna("")
dx = pd.read_csv(EICU / "diagnosis.csv.gz", dtype=str, usecols=["patientunitstayid", "diagnosisstring"])
dx = dx[dx.patientunitstayid.isin(ids)].copy()
dx["p"] = dx.diagnosisstring.fillna("")

# category -> (pastHistory regex, diagnosis-text regex). Only chronic categories with a van Walraven weight.
RULES = {
    "metacanc": (r"/Cancer/Metastases/", r"metasta|carcinomatosis"),
    "solidtum": (r"/Cancer/Cancer-Primary Site/(?!none)", r"^oncology\|(?!hematologic malignancy).*(CA\b|cancer|tumor|carcinoma|melanoma|sarcoma)"),
    "lymph":    (r"Hematologic Malignancy/(non-Hodgkins lymphoma|Hodgkins disease|multiple myeloma)", r"lymphoma|lymphoproliferative|multiple myeloma"),
    "chf":      (r"Congestive Heart Failure", r"congestive heart failure|\|CHF"),
    "valv":     (r"Valve Disease|valve replacement|Valve", r"valvular|valve (disease|replacement)|aortic stenosis|mitral (regurgitation|stenosis)"),
    "pvd":      (r"Peripheral Vascular Disease|Peripheral Vascular|/AAA|Abdominal aortic aneurysm", r"peripheral vascular disease"),
    "para":     (r"(para|quadri|hemi)plegia|Hemiparesis", r"(para|quadri|hemi)plegia"),
    "cpd":      (r"COPD|Asthma|Restrictive Disease|Pulmonary/Respiratory \(R\)/(Home Oxygen|COPD)", r"\|COPD|asthma|chronic obstructive"),
    "rf":       (r"Renal Insufficiency|Renal Failure|dialysis", r"chronic kidney disease|renal failure\|chronic|ESRD|end stage renal"),
    "ld":       (r"Cirrhosis|Liver/(cirrhosis|varices)|Liver \(R\)/(Cirrhosis|Varices|Liver Failure)", r"cirrhosis|chronic liver disease|portal hypertension"),
    "obes":     (r"Obesity|morbid obesity", r"obesity"),
    "blane":    (r"$^", r"blood loss anemia|anemia\|acute blood loss"),
    "dane":     (r"Anemia - |Iron Deficiency|anemia of chronic", r"anemia\|(chronic disease|iron deficiency|B12|folate)"),
    "drug":     (r"Drug abuse|Substance Abuse|drug abuse", r"drug (abuse|dependence)|opioid (abuse|dependence)|cocaine"),
    "depre":    (r"Depression", r"\|depression"),
}
VW = {"chf": 7, "valv": -1, "pvd": 2, "para": 7, "cpd": 3, "rf": 5, "ld": 11, "lymph": 9, "metacanc": 12,
      "solidtum": 4, "obes": -4, "blane": -2, "dane": -2, "drug": -7, "depre": -3}

prep_r = read_retry(MODELS / "prep_random_split.joblib", mode="joblib")
mimic_mean = pd.Series(prep_r.imp.initial_imputer_.statistics_, index=prep_r.cols)
H = pd.DataFrame(index=ids)
rows, matched = [], []
for cat, (rp, rd) in RULES.items():
    m_ph = ph.p.str.contains(rp, case=False, regex=True)
    m_dx = dx.p.str.contains(rd, case=False, regex=True)
    s_ph, s_dx = set(ph.loc[m_ph, "patientunitstayid"]), set(dx.loc[m_dx, "patientunitstayid"])
    icd = X[f"elix_{cat}"].astype(int) if f"elix_{cat}" in X else pd.Series(0, index=ids)
    H[f"elix_{cat}"] = ((icd == 1) | ids.isin(s_ph) | ids.isin(s_dx)).astype(int)
    rows.append({"category": cat, "vw_weight": VW[cat], "eICU_ICD_only_%": 100 * icd.mean(),
                 "eICU_harmonised_%": 100 * H[f"elix_{cat}"].mean(),
                 "added_by_pastHistory": len(s_ph - set(ids[icd == 1])), "added_by_dx_text": len(s_dx - set(ids[icd == 1])),
                 "MIMIC_training_%": 100 * mimic_mean.get(f"elix_{cat}", np.nan)})
    for src_, df_, m_ in [("pastHistory", ph, m_ph), ("diagnosis", dx, m_dx)]:
        lab = df_.loc[m_, "p"].str.replace("notes/Progress Notes/Past History/Organ Systems/", "", regex=False)
        for k, cnt in lab.value_counts().head(8).items():
            matched.append({"category": cat, "source": src_, "label": k[:95], "rows": cnt})
P = pd.DataFrame(rows).set_index("category")
# Elixhauser hierarchy, as comorbidipy assign0=True did in MIMIC: metastatic cancer overrides solid tumour
H.loc[H.elix_metacanc == 1, "elix_solidtum"] = 0
P.loc["solidtum", "eICU_harmonised_%"] = 100 * H.elix_solidtum.mean()
H["elix_vw_chronic"] = sum(H[f"elix_{k}"] * w for k, w in VW.items())
with pd.option_context("display.width", 200, "display.max_rows", 200, "display.max_colwidth", 95):
    print("=== Elixhauser prevalence: eICU ICD only vs harmonised vs MIMIC training ===")
    print(P.round(1).to_string())
    print(f"\nvan Walraven chronic score  eICU ICD only: median {X.elix_vw_chronic.median():.0f}, mean {X.elix_vw_chronic.mean():.1f}"
          f" | harmonised: median {H.elix_vw_chronic.median():.0f}, mean {H.elix_vw_chronic.mean():.1f}"
          f" | MIMIC training mean {mimic_mean['elix_vw_chronic']:.1f}")
    print("\n=== labels that matched (top 8 per category and source) ===")
    print(pd.DataFrame(matched).to_string(index=False))
H.to_csv(OUT / "eicu_comorbidity_harmonised.csv")            # patient-level: stays on this Mac
P.to_csv(OUT / "eicu_comorbidity_prevalence.csv")
print("\nCell 6b done - nothing applied yet; no outcome read.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 7 (NEW): amendment log + frozen-model predictions for eICU - NO outcome read
#  (a) AMENDMENT (logged with time, before any outcome): comorbidity flags harmonised with eICU pastHistory and
#      diagnosis text (eICU ICD codes come from the ICU problem list, not full discharge coding). ICD-only = sensitivity.
#  (b) frozen MIMIC preparation (prep_<arm>.joblib) -> frozen final models -> calibrated risk, uncalibrated score,
#      Venn-Abers interval, locked threshold / tiers; SOFA + APACHE II comparator models; eICU APACHE IVa prediction
#  (c) predictions saved + fingerprinted BEFORE the outcome cell
# ============================================================
import warnings, datetime, importlib, sys
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
assert hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"], "cohort changed - STOP"
assert not (OUT / "outcome_opened_log.json").exists(), "outcome already opened - predictions must not be rebuilt"

# ---------- (a) harmonised comorbidity flags (Cell 6b rules, 2 corrections) ----------
X_icd = pd.read_csv(OUT / "eicu_model_inputs.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
ids = X_icd.index
ph = pd.read_csv(EICU / "pastHistory.csv.gz", dtype=str, usecols=["patientunitstayid", "pasthistorypath"])
ph = ph[ph.patientunitstayid.isin(ids)]; php = ph.pasthistorypath.fillna("")
dxt = pd.read_csv(EICU / "diagnosis.csv.gz", dtype=str, usecols=["patientunitstayid", "diagnosisstring"])
dxt = dxt[dxt.patientunitstayid.isin(ids)]; dxp = dxt.diagnosisstring.fillna("")
RULES = {
    "metacanc": (r"/Cancer/Metastases/", r"metasta|carcinomatosis"),
    "solidtum": (r"/Cancer/Cancer-Primary Site/(?!none)", r"^oncology\|(?!hematologic malignancy).*(CA\b|cancer|tumor|carcinoma|melanoma|sarcoma)"),
    "lymph":    (r"Hematologic Malignancy/(non-Hodgkins lymphoma|Hodgkins disease|multiple myeloma)", r"lymphoma|lymphoproliferative|multiple myeloma"),
    "chf":      (r"Congestive Heart Failure", r"congestive heart failure(?!\|cor pulmonale)|\|CHF"),          # correction 1
    "valv":     (r"Valve Disease|valve replacement|Valve", r"valvular|valve (disease|replacement)|aortic stenosis|mitral (regurgitation|stenosis)"),
    "pvd":      (r"Peripheral Vascular Disease|Peripheral Vascular|/AAA|Abdominal aortic aneurysm", r"peripheral vascular disease"),
    "para":     (r"(para|quadri|hemi)plegia|Hemiparesis", r"(para|quadri|hemi)plegia"),
    "cpd":      (r"COPD|Asthma|Restrictive Disease|Pulmonary/Respiratory \(R\)/(Home Oxygen|COPD)", r"\|COPD|asthma|chronic obstructive"),
    "rf":       (r"Renal Insufficiency|Renal Failure|dialysis", r"chronic kidney disease|renal failure\|chronic|ESRD|end stage renal"),
    "ld":       (r"Cirrhosis|Liver/(cirrhosis|varices)|Liver \(R\)/(Cirrhosis|Varices|Liver Failure)", r"cirrhosis|chronic liver disease|portal hypertension"),
    "obes":     (r"Obesity|morbid obesity", r"obesity"),
    "blane":    (r"$^", r"chronic blood loss anemia"),                                                         # correction 2
    "dane":     (r"Anemia - |Iron Deficiency|anemia of chronic", r"anemia\|(chronic disease|iron deficiency|B12|folate)"),
    "drug":     (r"Drug abuse|Substance Abuse|drug abuse", r"drug (abuse|dependence)|opioid (abuse|dependence)|cocaine"),
    "depre":    (r"Depression", r"\|depression"),
}
VW = {"chf": 7, "valv": -1, "pvd": 2, "para": 7, "cpd": 3, "rf": 5, "ld": 11, "lymph": 9, "metacanc": 12,
      "solidtum": 4, "obes": -4, "blane": -2, "dane": -2, "drug": -7, "depre": -3}
X_h = X_icd.copy()
for cat, (rp, rd) in RULES.items():
    s_ph = set(ph.loc[php.str.contains(rp, case=False, regex=True), "patientunitstayid"])
    s_dx = set(dxt.loc[dxp.str.contains(rd, case=False, regex=True), "patientunitstayid"])
    X_h[f"elix_{cat}"] = ((X_icd[f"elix_{cat}"] == 1) | ids.isin(s_ph) | ids.isin(s_dx)).astype(int)
X_h.loc[X_h.elix_metacanc == 1, "elix_solidtum"] = 0                       # Elixhauser hierarchy (as assign0 in MIMIC)
X_h["elix_vw_chronic"] = sum(X_h[f"elix_{k}"] * w for k, w in VW.items())
immuno = X_h.cancer_type.eq("Hematologic") | X_h.elix_lymph.eq(1) | X_h.elix_aids.eq(1)
organ = X_h.elix_ld.eq(1) | X_h.elix_chf.eq(1) | X_h.elix_cpd.eq(1) | X_h.elix_rf.eq(1)
X_h["apache_chronic_pts"] = np.where(immuno | organ, 5, 0)
X_h["apache_ii_total"] = X_h.apache_aps + X_h.apache_age_pts + X_h.apache_chronic_pts
X_h.to_csv(OUT / "eicu_model_inputs_harmonised.csv")

amend_file = OUT / "analysis_plan_amendments.json"
if not amend_file.exists():
    amend_file.write_text(json.dumps([{
        "amended_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "before_outcome": True,
        "change": ("Elixhauser chronic comorbidity flags (and van Walraven score, APACHE II chronic points) harmonised: "
                   "eICU ICD codes OR eICU pastHistory OR ICU diagnosis text; Elixhauser hierarchy applied. "
                   "PRIMARY = harmonised; SENSITIVITY = ICD codes only (as in the locked plan)."),
        "reason": ("eICU diagnosis ICD codes come from the ICU problem list, not full hospital discharge coding: "
                   "ICD-only van Walraven mean 6.4 vs 13.1 in MIMIC; metastatic cancer 12.4% vs 41.8%. "
                   "Harmonised: 9.6 and 27.5%. Two label corrections: cor pulmonale not CHF; only chronic blood-loss anaemia."),
        "harmonised_file_md5": hashlib.md5((OUT / "eicu_model_inputs_harmonised.csv").read_bytes()).hexdigest()}], indent=2))
    print("Amendment logged (before outcome).")
else:
    print("Amendment already logged:", json.loads(amend_file.read_text())[0]["amended_at"])

# ---------- (b) frozen preparation + models ----------
import v2_prep
v2_prep.P2 = str(ROOT / "data" / "processed_v2") + "/"                      # tpife_thresholds.csv location
locked = {a: json.loads((MODELS / f"locked_rules_{a}.json").read_text()) for a in ["random_split", "temporal_split"]}

def cvap_interval(cv, Z):
    """Return (p0, p1) from the hand-written cross Venn-Abers object, whatever its method is called."""
    for name in ["predict_interval", "predict_p0p1", "predict_proba_interval", "predict_proba", "predict"]:
        if hasattr(cv, name):
            try:
                out = getattr(cv, name)(Z)
            except Exception:
                continue
            if isinstance(out, dict):
                keys = {k.lower(): k for k in out}
                lo_k = next((keys[k] for k in ["p0", "lower", "low", "lo", "p_lower", "pl"] if k in keys), None)
                hi_k = next((keys[k] for k in ["p1", "upper", "high", "hi", "p_upper", "pu"] if k in keys), None)
                print(f"  Venn-Abers '{name}' returned a dict with keys {list(out)} -> using {lo_k} / {hi_k}")
                if lo_k is None or hi_k is None:
                    return None, None, f"{name}: dict keys {list(out)}"
                return np.asarray(out[lo_k], float), np.asarray(out[hi_k], float), name
            if isinstance(out, tuple) and len(out) >= 2:
                return np.asarray(out[0], float), np.asarray(out[1], float), name
            arr = np.asarray(out, float)
            if arr.ndim == 2 and arr.shape[1] >= 2:
                return arr[:, 0], arr[:, 1], name
    return None, None, None

apr = pd.read_csv(EICU / "apachePatientResult.csv.gz", dtype=str,
                  usecols=["patientunitstayid", "apacheversion", "predictedhospitalmortality"])   # NO actual-outcome columns read
apr = apr[apr.patientunitstayid.isin(ids) & (apr.apacheversion == "IVa")]
apr["p"] = pd.to_numeric(apr.predictedhospitalmortality, errors="coerce")
apache4 = apr[apr.p.between(0, 1)].groupby("patientunitstayid").p.first().reindex(ids)

PRED, diag = {}, []
for version, Xv in [("harmonised", X_h), ("icd_only", X_icd)]:
    for arm in ["random_split", "temporal_split"]:
        B = bundles[f"final_{arm}"]
        prep = read_retry(MODELS / f"prep_{arm}.joblib", mode="joblib")
        Xin = v2_prep.add_fixed_features(Xv)
        Z = prep.transform(Xin)
        feats = list(B["features"])
        Zf = Z[feats]
        p_cal = B["final_calibrated"].predict_proba(Zf)[:, 1]
        p_unc = B["final_uncalibrated"].predict_proba(Zf)[:, 1]
        p0, p1, how = cvap_interval(B["cvap"], Zf)
        lr = locked[arm]
        P = pd.DataFrame({"risk": p_cal, "score_uncal": p_unc}, index=ids)
        P["above_threshold"] = (P.risk >= lr["threshold"]).astype(int)
        P["tier"] = np.where(P.risk < lr["tier_cut_low"], "Low", np.where(P.risk >= lr["tier_cut_high"], "High", "Intermediate"))
        if p0 is not None:
            P["va_p0"], P["va_p1"] = p0, p1
            lo_, hi_ = np.minimum(p0, p1), np.maximum(p0, p1)
            P["uncertain_tier"] = (((lo_ < lr["tier_cut_low"]) & (hi_ >= lr["tier_cut_low"])) |
                                   ((lo_ < lr["tier_cut_high"]) & (hi_ >= lr["tier_cut_high"]))).astype(int)
        for comp_, col in [("SOFA", "sofa_total"), ("APACHE-II", "apache_ii_total")]:
            m = B[comp_]; fn = list(getattr(m, "feature_names_in_", [col]))
            src_ = Z if all(f in Z.columns for f in fn) else Xv
            P[f"p_{comp_}"] = m.predict_proba(src_[fn].astype(float).fillna(src_[fn].median()))[:, 1]
        P["sofa_total"], P["apache_ii_total"] = Xv.sofa_total.values, Xv.apache_ii_total.values
        P["p_APACHE_IVa_eICU"] = apache4.values
        PRED[(version, arm)] = P
        # diagnostics: imputation and clipping of the model's own predictors
        raw = Xv.reindex(columns=feats).astype(float)
        n_imp = raw.isna().sum()
        lo, hi = prep.lo.reindex(feats), prep.hi.reindex(feats)
        n_clip = ((raw.lt(lo, axis=1)) | (raw.gt(hi, axis=1))).sum()
        diag.append({"version": version, "arm": arm, "n": len(P), "VA_method": how,
                     "mean_risk_%": 100 * P.risk.mean(), "expected_deaths": P.risk.sum(),
                     "above_threshold_%": 100 * P.above_threshold.mean(),
                     "tier_low/int/high": "/".join(str(int((P.tier == t).sum())) for t in ["Low", "Intermediate", "High"]),
                     "uncertain_%": 100 * P.get("uncertain_tier", pd.Series(np.nan)).mean(),
                     "patients_with_any_imputed_predictor_%": 100 * raw.isna().any(axis=1).mean(),
                     "predictor_values_clipped": int(n_clip.sum())})

coh_ = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
for k, P in PRED.items():
    P["cam_pos_by_24h"] = coh_.cam_pos_by_24h.reindex(ids).values
    P["hospitalid"] = coh_.hospitalid.reindex(ids).values
    P.to_csv(OUT / f"eicu_predictions_{k[0]}_{k[1]}.csv")                 # patient-level: stays on this Mac

# ---------- (c) fingerprint predictions BEFORE the outcome ----------
fps = {f.name: hashlib.md5(f.read_bytes()).hexdigest() for f in sorted(OUT.glob("eicu_predictions_*.csv"))}
(OUT / "predictions_locked.json").write_text(json.dumps({"saved_at": datetime.datetime.now().isoformat(timespec="seconds"),
                                                         "files": fps}, indent=2))
D = pd.DataFrame(diag)
with pd.option_context("display.width", 220):
    print("\n=== frozen-model predictions on eICU (no outcome used) ===")
    print(D.round(2).to_string(index=False))
    print(f"\nAPACHE IVa prediction available: {apache4.notna().sum()}/{len(ids)}")
    Pm = PRED[("harmonised", "random_split")]
    print("Primary model, mean predicted risk by CAM-ICU+ by 24 h:",
          Pm.groupby("cam_pos_by_24h").risk.mean().round(3).to_dict())
    print("Correlation of primary risk with SOFA / APACHE II / APACHE IVa:",
          Pm[["risk", "sofa_total", "apache_ii_total", "p_APACHE_IVa_eICU"]].corr(method="spearman").risk.round(2).to_dict())
print("\nPredictions saved and fingerprinted:", list(fps))
print("Cell 7 done - NO outcome read. Next cell opens the outcome once.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 8 (NEW): OPEN THE eICU OUTCOME ONCE and run the locked external-validation plan
#  - checks plan, amendment and prediction fingerprints first; logs the opening time
#  - outcome: in-hospital death within 28 days of ICU admission (hospitaldischargeoffset <= 40320 min)
#  - primary: frozen random-split model, harmonised comorbidity, all patients; secondary: CAM-ICU+ by 24 h,
#    temporal model; sensitivity: ICD-only comorbidity, ICD-confirmed cancer, complete case, hospital-cluster bootstrap
#  - updated model (secondary): 10-fold cross-validated logistic recalibration
# ============================================================
import warnings, datetime
warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess

# ---------- 0. integrity checks, then open the outcome ----------
plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
assert hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"], "cohort changed"
amend = json.loads((OUT / "analysis_plan_amendments.json").read_text())
assert hashlib.md5((OUT / "eicu_model_inputs_harmonised.csv").read_bytes()).hexdigest() == amend[0]["harmonised_file_md5"]
lockp = json.loads((OUT / "predictions_locked.json").read_text())
for f, h in lockp["files"].items():
    assert hashlib.md5((OUT / f).read_bytes()).hexdigest() == h, f"{f} changed after locking - STOP"
log_f = OUT / "outcome_opened_log.json"
if not log_f.exists():
    log_f.write_text(json.dumps({"outcome_first_opened": datetime.datetime.now().isoformat(timespec="seconds"),
                                 "plan_locked": plan["locked_at"], "amendment": amend[0]["amended_at"],
                                 "predictions_locked": lockp["saved_at"]}, indent=2))
print("Audit trail:", json.loads(log_f.read_text()))

coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str,
                 usecols=["patientunitstayid", "hospitaldischargestatus", "hospitaldischargeoffset"]).set_index("patientunitstayid")
pt = pt.reindex(coh.index)
off = pd.to_numeric(pt.hospitaldischargeoffset, errors="coerce")
status = pt.hospitaldischargestatus.str.strip()
known = status.isin(["Alive", "Expired"]) & off.notna()
y_all = ((status == "Expired") & (off <= 40320)).astype(int)
print(f"\nOutcome known for {known.sum()}/{len(coh)} patients (unknown discharge status excluded: {(~known).sum()})")
print(f"28-day in-hospital deaths: {y_all[known].sum()} ({100*y_all[known].mean():.1f}%) | "
      f"deaths after day 28 in hospital (counted as survivors at 28 d): {int(((status=='Expired') & (off>40320)).sum())}")

# ---------- metric helpers ----------
def delong(y, s):
    """AUROC + DeLong variance components (Sun & Xu fast algorithm)."""
    y, s = np.asarray(y), np.asarray(s, float)
    pos, neg = s[y == 1], s[y == 0]
    m, n = len(pos), len(neg)
    allv = np.r_[pos, neg]
    r_all = stats.rankdata(allv); r_pos = stats.rankdata(pos); r_neg = stats.rankdata(neg)
    v10 = (r_all[:m] - r_pos) / n
    v01 = 1 - (r_all[m:] - r_neg) / m
    auc = v10.mean()
    return auc, v10, v01
def auc_ci(y, s):
    auc, v10, v01 = delong(y, s)
    se = np.sqrt(np.var(v10, ddof=1) / len(v10) + np.var(v01, ddof=1) / len(v01))
    lo, hi = stats.norm.ppf([0.025, 0.975]) * se + auc
    return auc, max(lo, 0), min(hi, 1)
def delong_test(y, s1, s2):
    a1, x10, x01 = delong(y, s1); a2, y10, y01 = delong(y, s2)
    m, n = len(x10), len(x01)
    var = (np.var(x10 - y10, ddof=1) / m) + (np.var(x01 - y01, ddof=1) / n)
    z = (a1 - a2) / np.sqrt(var) if var > 0 else np.nan
    return a1 - a2, 2 * stats.norm.sf(abs(z))
def wilson(k, n):
    if n == 0: return (np.nan, np.nan)
    p, z = k / n, 1.959964
    c = (p + z * z / (2 * n)) / (1 + z * z / n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * (c - h), 100 * (c + h)
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def calib(y, p):
    lp = logit(p)
    slope = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
    intercept = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
    sm_ = lowess(y, p, frac=0.75, it=0, return_sorted=False)
    ici = np.mean(np.abs(sm_ - p)); e90 = np.quantile(np.abs(sm_ - p), 0.9)
    return slope, intercept, p.mean() / y.mean(), ici, e90
def net_benefit(y, p, t=0.30):
    n = len(y); tp = ((p >= t) & (y == 1)).sum(); fp = ((p >= t) & (y == 0)).sum()
    return tp / n - fp / n * t / (1 - t), y.mean() - (1 - y.mean()) * t / (1 - t)
rng = np.random.default_rng(2026)
def boot_thr(y, p, thr, B=1000):
    def f(yy, pp):
        pr = pp >= thr; tp = (pr & (yy == 1)).sum(); fn = (~pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum()
        return [tp / (tp + fn) if tp + fn else np.nan, tn / (tn + fp) if tn + fp else np.nan,
                tp / (tp + fp) if tp + fp else np.nan, tn / (tn + fn) if tn + fn else np.nan]
    est = f(y, p); bs = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y)); bs.append(f(y[i], p[i]))
    lo, hi = np.nanpercentile(bs, [2.5, 97.5], axis=0)
    return est, lo, hi
def cluster_auc(y, s, groups, B=1000):
    g = np.asarray(groups); u = np.unique(g); idx = {k: np.where(g == k)[0] for k in u}; out = []
    for _ in range(B):
        take = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)])
        if len(np.unique(y[take])) == 2: out.append(roc_auc_score(y[take], s[take]))
    return np.percentile(out, [2.5, 97.5])

locked = {a: json.loads((MODELS / f"locked_rules_{a}.json").read_text()) for a in ["random_split", "temporal_split"]}
def evaluate(P, y, arm, label):
    y = np.asarray(y); p = P.risk.to_numpy(); s = P.score_uncal.to_numpy(); lr = locked[arm]
    n, d = len(y), int(y.sum())
    r = {"analysis": label, "n": n, "deaths": d, "mortality_%": 100 * y.mean(), "reliable": d >= 30 and n - d >= 30}
    a, lo, hi = auc_ci(y, p); r.update({"AUROC": a, "AUROC_lo": lo, "AUROC_hi": hi})
    r["AUPRC"] = average_precision_score(y, p); r["Brier"] = brier_score_loss(y, p)
    r["slope"], r["intercept"], r["E/O"], r["ICI"], r["E90"] = calib(y, p)
    for comp, col in [("SOFA", "sofa_total"), ("APACHE_II", "apache_ii_total")]:
        ca, clo, chi = auc_ci(y, P[col].to_numpy()); dlt, pv = delong_test(y, s, P[col].to_numpy())
        r.update({f"AUROC_{comp}": ca, f"AUROC_{comp}_lo": clo, f"AUROC_{comp}_hi": chi, f"p_vs_{comp}": pv})
    m4 = P.p_APACHE_IVa_eICU.notna().to_numpy()
    if m4.sum() > 20 and len(np.unique(y[m4])) == 2:
        a4 = auc_ci(y[m4], P.p_APACHE_IVa_eICU.to_numpy()[m4]); am = auc_ci(y[m4], p[m4]); dl4, p4 = delong_test(y[m4], s[m4], P.p_APACHE_IVa_eICU.to_numpy()[m4])
        r.update({"n_APACHE_IVa": int(m4.sum()), "AUROC_APACHE_IVa": a4[0], "AUROC_APACHE_IVa_lo": a4[1], "AUROC_APACHE_IVa_hi": a4[2],
                  "AUROC_model_same_pts": am[0], "p_vs_APACHE_IVa": p4})
    est, blo, bhi = boot_thr(y, p, lr["threshold"])
    for k, e, l_, h_ in zip(["sens", "spec", "PPV", "NPV"], est, blo, bhi):
        r[k], r[f"{k}_lo"], r[f"{k}_hi"] = e, l_, h_
    r["NB_model_30"], r["NB_treat_all_30"] = net_benefit(y, p)
    for t in ["Low", "Intermediate", "High"]:
        m = (P.tier == t).to_numpy(); k = int(y[m].sum()); nn = int(m.sum())
        lo_, hi_ = wilson(k, nn)
        r[f"tier_{t}_n"], r[f"tier_{t}_obs_%"], r[f"tier_{t}_lo"], r[f"tier_{t}_hi"] = nn, 100 * k / nn if nn else np.nan, lo_, hi_
        r[f"tier_{t}_pred_%"] = 100 * p[m].mean() if nn else np.nan
    if "uncertain_tier" in P:
        u = P.uncertain_tier.to_numpy() == 1
        r["uncertain_%"] = 100 * u.mean(); r["uncertain_mortality_%"] = 100 * y[u].mean() if u.any() else np.nan
    return r

# ---------- 1. load locked predictions ----------
def load(version, arm):
    P = pd.read_csv(OUT / f"eicu_predictions_{version}_{arm}.csv", dtype={"patientunitstayid": str, "hospitalid": str}).set_index("patientunitstayid")
    return P.loc[known[known].index]
y = y_all[known]
Xh = pd.read_csv(OUT / "eicu_model_inputs_harmonised.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid").loc[y.index]
f25 = list(bundles["final_random_split"]["features"])
complete = Xh[f25].notna().all(axis=1)

R = []
Pm = load("harmonised", "random_split")
cam = Pm.cam_pos_by_24h == 1
R.append(evaluate(Pm, y, "random_split", "PRIMARY: all patients"))
R.append(evaluate(Pm[cam], y[cam], "random_split", "Intended use: CAM-ICU+ by 24 h"))
R.append(evaluate(Pm[~cam], y[~cam], "random_split", "Not CAM-ICU+ by 24 h"))
Pt = load("harmonised", "temporal_split")
R.append(evaluate(Pt, y, "temporal_split", "Temporal model: all patients"))
R.append(evaluate(Pt[cam], y[cam], "temporal_split", "Temporal model: CAM-ICU+ by 24 h"))
Pi = load("icd_only", "random_split")
R.append(evaluate(Pi, y, "random_split", "Sensitivity: ICD-only comorbidity"))
icd_c = coh.loc[y.index, "cancer_icd"] == 1
R.append(evaluate(Pm[icd_c], y[icd_c], "random_split", "Sensitivity: ICD-confirmed cancer"))
R.append(evaluate(Pm[complete], y[complete], "random_split", "Sensitivity: complete case (all 25 measured)"))
res = pd.DataFrame(R).set_index("analysis")

# AUROC difference CAM+ vs not (independent groups)
a1, v1a, v1b = delong(y[cam].to_numpy(), Pm.risk[cam].to_numpy()); a2, v2a, v2b = delong(y[~cam].to_numpy(), Pm.risk[~cam].to_numpy())
se = np.sqrt(np.var(v1a, ddof=1)/len(v1a) + np.var(v1b, ddof=1)/len(v1b) + np.var(v2a, ddof=1)/len(v2a) + np.var(v2b, ddof=1)/len(v2b))
p_het = 2 * stats.norm.sf(abs((a1 - a2) / se))
clu = cluster_auc(y.to_numpy(), Pm.risk.to_numpy(), Pm.hospitalid.to_numpy())

# ---------- 2. updated model: 10-fold cross-validated logistic recalibration ----------
lp = logit(Pm.risk.to_numpy()); yy = y.to_numpy(); p_rc = np.zeros(len(yy))
for tr, te in StratifiedKFold(10, shuffle=True, random_state=2026).split(lp, yy):
    fit = sm.GLM(yy[tr], sm.add_constant(lp[tr]), family=sm.families.Binomial()).fit()
    p_rc[te] = fit.predict(sm.add_constant(lp[te], has_constant="add"))
rc = calib(yy, p_rc)
full_fit = sm.GLM(yy, sm.add_constant(lp), family=sm.families.Binomial()).fit()

res.to_csv(OUT / "eicu_external_validation_results.csv")
pd.DataFrame({"patientunitstayid": y.index, "y": y.values, "risk": Pm.risk.values, "risk_recalibrated_cv": p_rc}).to_csv(
    OUT / "eicu_outcome_and_risk.csv", index=False)                                  # patient-level: stays on this Mac

# ---------- 3. print ----------
show = ["n", "deaths", "mortality_%", "AUROC", "AUROC_lo", "AUROC_hi", "AUROC_SOFA", "p_vs_SOFA", "AUROC_APACHE_II", "p_vs_APACHE_II",
        "AUPRC", "Brier", "slope", "intercept", "E/O", "ICI", "sens", "spec", "PPV", "NPV", "NB_model_30", "NB_treat_all_30", "reliable"]
with pd.option_context("display.width", 250, "display.max_columns", 40):
    print("\n=== eICU EXTERNAL VALIDATION (frozen MIMIC model) ===")
    print(res[show].T.round(3).to_string())
    t_cols = [c for c in res.columns if c.startswith("tier_") or c.startswith("uncertain")]
    print("\n=== locked risk tiers ===")
    print(res.loc[["PRIMARY: all patients", "Intended use: CAM-ICU+ by 24 h"], t_cols].T.round(1).to_string())
    a4 = [c for c in res.columns if "APACHE_IVa" in c or c == "AUROC_model_same_pts"]
    print("\n=== vs eICU's own APACHE IVa (patients with an APACHE IVa prediction) ===")
    print(res.loc[["PRIMARY: all patients", "Intended use: CAM-ICU+ by 24 h"], a4].T.round(3).to_string())
print(f"\nAUROC CAM-ICU+ by 24 h vs not: difference {a1-a2:.3f}, p = {p_het:.3f}")
print(f"Hospital-cluster bootstrap 95% CI for primary AUROC: {clu[0]:.3f}-{clu[1]:.3f} ({Pm.hospitalid.nunique()} hospitals)")
print(f"Updated model (10-fold CV recalibration): slope {rc[0]:.2f}, intercept {rc[1]:.2f}, E/O {rc[2]:.2f}, ICI {rc[3]:.3f}, "
      f"Brier {brier_score_loss(yy, p_rc):.3f} | full-data recalibration: intercept {full_fit.params[0]:.2f}, slope {full_fit.params[1]:.2f}")

# ---------- 4. figure (Springer: 174 mm wide, Arial, vector PDF + 600-dpi PNG) ----------
import matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({"font.family": "Arial", "font.size": 8, "pdf.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(1, 2, figsize=(174 / 25.4, 80 / 25.4))
OK = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00", "grey": "#7F7F7F", "purple": "#CC79A7"}
for lab, yy_, ss, col, ls in [(f"Model, all patients (AUROC {res.loc['PRIMARY: all patients','AUROC']:.3f})", y, Pm.risk, OK["blue"], "-"),
                              (f"Model, CAM-ICU+ by 24 h ({res.loc['Intended use: CAM-ICU+ by 24 h','AUROC']:.3f})", y[cam], Pm.risk[cam], OK["green"], "-"),
                              (f"SOFA ({res.loc['PRIMARY: all patients','AUROC_SOFA']:.3f})", y, Pm.sofa_total, OK["orange"], "--"),
                              (f"APACHE II ({res.loc['PRIMARY: all patients','AUROC_APACHE_II']:.3f})", y, Pm.apache_ii_total, OK["red"], "--")]:
    fpr, tpr, _ = roc_curve(yy_, ss); ax[0].plot(fpr, tpr, ls, color=col, lw=1.2, label=lab)
m4 = Pm.p_APACHE_IVa_eICU.notna()
if "AUROC_APACHE_IVa" in res:
    fpr, tpr, _ = roc_curve(y[m4], Pm.p_APACHE_IVa_eICU[m4])
    ax[0].plot(fpr, tpr, ":", color=OK["purple"], lw=1.2, label=f"eICU APACHE IVa ({res.loc['PRIMARY: all patients','AUROC_APACHE_IVa']:.3f})")
ax[0].plot([0, 1], [0, 1], color=OK["grey"], lw=0.6)
ax[0].set(xlabel="1 - specificity", ylabel="Sensitivity", title="A  Discrimination (eICU)")
ax[0].legend(loc="lower right", fontsize=6.5, frameon=False)
pp = Pm.risk.to_numpy(); sm_ = lowess(yy, pp, frac=0.75, it=0)
ax[1].plot(sm_[:, 0], sm_[:, 1], color=OK["blue"], lw=1.4, label="LOESS calibration curve")
q = pd.qcut(pd.Series(pp).rank(method="first"), 10, labels=False)
for k in range(10):
    m = (q == k).to_numpy(); kk = int(yy[m].sum()); nn = int(m.sum()); lo_, hi_ = wilson(kk, nn)
    ax[1].errorbar(pp[m].mean(), kk / nn, yerr=[[kk / nn - lo_ / 100], [hi_ / 100 - kk / nn]], fmt="o", ms=3, color="black", lw=0.8,
                   label="Risk deciles (95% CI)" if k == 0 else None)
ax[1].plot([0, 1], [0, 1], ":", color=OK["grey"], lw=0.8, label="Perfect calibration")
r0 = res.loc["PRIMARY: all patients"]
ax[1].text(0.03, 0.97, f"Slope {r0.slope:.2f}; intercept {r0.intercept:.2f}\nE/O {r0['E/O']:.2f}; ICI {r0.ICI:.3f}; Brier {r0.Brier:.3f}",
           transform=ax[1].transAxes, va="top", fontsize=6.5)
ax[1].set(xlim=(0, 1), ylim=(0, 1), xlabel="Predicted 28-day mortality risk", ylabel="Observed 28-day mortality", title="B  Calibration (eICU)")
ax[1].legend(loc="lower right", fontsize=6.5, frameon=False)
fig.tight_layout()
fig.savefig(FIGS / "fig_eicu_external_validation.pdf"); fig.savefig(FIGS / "fig_eicu_external_validation.png", dpi=600)
plt.close(fig)
print("\nSaved: nb5/eicu_external_validation_results.csv, figures_v2/nb5/fig_eicu_external_validation.pdf/.png")
print("Cell 8 done.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 9 (REDO: legend moved clear of histogram, decision-curve y-axis widened): reporting tables + final figures for the paper
#  - reads ONLY files already saved by Cells 3-8 (outcome is NOT re-opened, nothing is refitted)
#  - Table: eICU vs MIMIC patient characteristics (with standardised mean differences)
#  - Table: missing data of the 25 model predictors, eICU vs MIMIC training
#  - Decision curve 5-50%: frozen model vs locally recalibrated model (supplement)
#  - Main figure redrawn with 4 cosmetic fixes (numbers unchanged)
#  - privacy check: lists every patient-level file that must stay on this Mac
# ============================================================
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import roc_curve
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib as mpl, matplotlib.pyplot as plt

TAB = OUT / "tables"; TAB.mkdir(exist_ok=True)
MINUS = "−"
def fmt_num(x, d=2): return f"{x:.{d}f}".replace("-", MINUS)

# ---------- 0. the outcome log must already exist (Cell 8 ran once) ----------
log = json.loads((OUT / "outcome_opened_log.json").read_text())
print("Outcome opened once at", log["outcome_first_opened"], "- this cell only reads saved files.\n")

R = pd.read_csv(OUT / "eicu_outcome_and_risk.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
y = R.y.astype(int)
P = pd.read_csv(OUT / "eicu_predictions_harmonised_random_split.csv", dtype={"patientunitstayid": str, "hospitalid": str}).set_index("patientunitstayid").loc[y.index]
Xh = pd.read_csv(OUT / "eicu_model_inputs_harmonised.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid").loc[y.index]
coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid").loc[y.index]
res = pd.read_csv(OUT / "eicu_external_validation_results.csv").set_index("analysis")
assert np.allclose(R.risk.to_numpy(), P.risk.to_numpy()), "saved risk and locked predictions differ - STOP"

# ---------- 1. MIMIC reference cohort (all 2,777 patients, raw unimputed values) ----------
P2_ = ROOT / "data" / "processed_v2"
try:
    M = pd.read_csv(P2_ / "model_table_v2.csv")
    D = pd.read_csv(P2_ / "cohort_delirium_timing.csv")
    M = M.merge(D[["stay_id", "delirium_route", "cam_positive_by_24h"]], on="stay_id", how="left", validate="1:1")
    have_mimic = True
    print(f"MIMIC reference loaded: {len(M)} patients")
except Exception as e:
    have_mimic = False
    print("MIMIC files could not be read (iCloud placeholder?) - restore model_table_v2.csv and cohort_delirium_timing.csv,",
          "then rerun this cell. Error:", type(e).__name__)

# eICU delirium route (Cell 3 column names detected automatically)
def find_col(df, pats):
    import re
    for c in df.columns:
        if any(re.search(p, c.lower()) for p in pats): return c
    return None
c_icd = find_col(coh, [r"delir.*icd", r"icd.*delir"])
c_cam = find_col(coh, [r"cam.*(any|ever)", r"^cam_pos$", r"cam_positive$", r"^delirium_cam"])
E = Xh.copy()
E["mortality_28d"] = y
E["cam_positive_by_24h"] = coh.cam_pos_by_24h.astype(int)
if c_icd and c_cam:
    di, dc = coh[c_icd].fillna(0).astype(int) == 1, coh[c_cam].fillna(0).astype(int) == 1
    E["delirium_route"] = np.select([di & dc, di, dc], ["ICD + CAM-ICU", "ICD only", "CAM-ICU only"], "NONE")
else:
    print("NOTE: delirium-route columns not found in eicu_cohort.csv; columns are:", list(coh.columns))

def smd_cont(a, b):
    a, b = pd.Series(a, dtype=float).dropna(), pd.Series(b, dtype=float).dropna()
    sd = np.sqrt((a.var() + b.var()) / 2); return (a.mean() - b.mean()) / sd if sd > 0 else np.nan
def smd_bin(a, b):
    p1, p2 = pd.Series(a, dtype=float).dropna().mean(), pd.Series(b, dtype=float).dropna().mean()
    sd = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2); return (p1 - p2) / sd if sd > 0 else np.nan
def med(s, d=0):
    s = pd.Series(s, dtype=float).dropna()
    return f"{s.median():.{d}f} [{s.quantile(.25):.{d}f}{chr(8211)}{s.quantile(.75):.{d}f}]" if len(s) else "n/a"
def pct(s):
    s = pd.Series(s, dtype=float).dropna(); return f"{int(s.sum())} ({100*s.mean():.1f})" if len(s) else "n/a"
def miss(s): return f"{100*pd.Series(s).isna().mean():.1f}"

rows = []
def add(label, col, kind, d=0, fn=None):
    e = fn(E) if fn else (E[col] if col in E else pd.Series(np.nan, index=E.index))
    m = (fn(M) if fn else (M[col] if col in M else pd.Series(np.nan, index=M.index))) if have_mimic else None
    r = {"characteristic": label}
    r["eICU"] = med(e, d) if kind == "cont" else pct(e)
    r["eICU_missing_%"] = miss(e)
    if have_mimic:
        r["MIMIC-IV"] = med(m, d) if kind == "cont" else pct(m)
        r["MIMIC_missing_%"] = miss(m)
        r["SMD"] = round((smd_cont if kind == "cont" else smd_bin)(e, m), 2)
    rows.append(r)
add("Age, years, median [IQR]", "age_at_admission", "cont")
add("Male sex, n (%)", "sex_male", "bin")
add("Weight, kg, median [IQR]", "weight_kg", "cont", 1)
for ct in ["Gastrointestinal", "Genitourinary", "Hematologic", "Respiratory/Thoracic", "Other/Unspecified"]:
    add(f"Cancer type: {ct}, n (%)", None, "bin", fn=lambda df, ct=ct: (df.cancer_type == ct).astype(float).where(df.cancer_type.notna()))
add("Metastatic cancer (Elixhauser), n (%)", "elix_metacanc", "bin")
add("van Walraven chronic score, median [IQR]", "elix_vw_chronic", "cont")
add("SOFA at 24 h, median [IQR]", "sofa_total", "cont")
add("APACHE II at 24 h, median [IQR]", "apache_ii_total", "cont")
add("Vasopressor within 24 h, n (%)", "vaso_any", "bin")
add("Intubated within 24 h (verbal GCS = ETT), n (%)", "gcs_verbal_ETT_24h", "bin")
add("Lactate measured within 24 h, n (%)", None, "bin", fn=lambda df: df.Lactate_min.notna().astype(float))
add("GCS measured within 24 h, n (%)", None, "bin", fn=lambda df: df.GCS_Total_mean.notna().astype(float))
for rt in ["CAM-ICU only", "ICD + CAM-ICU", "ICD only"]:
    add(f"Delirium identified by {rt}, n (%)", None, "bin",
        fn=lambda df, rt=rt: (df.delirium_route == rt).astype(float) if "delirium_route" in df else pd.Series(np.nan, index=df.index))
add("CAM-ICU positive by 24 h, n (%)", "cam_positive_by_24h", "bin")
add("28-day mortality, n (%)", "mortality_28d", "bin")
T1 = pd.DataFrame(rows).set_index("characteristic")
if have_mimic:
    # cross-check against NB4 Table 1 (verified): age 70, male 60.7%, metastatic 41.7%, SOFA 7, APACHE II 25, vasopressor 37.1%, mortality 38.2%
    chk = {"age": M.age_at_admission.median(), "male_%": 100 * M.sex_male.mean(), "metastatic_%": 100 * M.elix_metacanc.mean(),
           "SOFA": M.sofa_total.median(), "APACHE_II": M.apache_ii_total.median(),
           "vasopressor_%": 100 * M.vaso_any.mean() if "vaso_any" in M else np.nan, "mortality_%": 100 * M.mortality_28d.mean()}
    print("MIMIC cross-check vs NB4 Table 1 (expect 70 / 60.7 / 41.7 / 7 / 25 / 37.1 / 38.2):",
          {k: round(v, 1) for k, v in chk.items()})
apiv = P.p_APACHE_IVa_eICU
T1.loc["eICU APACHE IVa predicted hospital mortality, %, median [IQR]", "eICU"] = med(100 * apiv, 1)
T1.loc["eICU APACHE IVa predicted hospital mortality, %, median [IQR]", "eICU_missing_%"] = miss(apiv)
T1.loc["Hospitals, n", "eICU"] = str(P.hospitalid.nunique())
T1.to_csv(TAB / "table_eicu_vs_mimic_characteristics.csv")

# ---------- 2. missing data of the 25 predictors ----------
f25 = list(bundles["final_random_split"]["features"])
cmp = pd.read_csv(OUT / "eicu_vs_mimic_predictors.csv", index_col=0)
MS = pd.DataFrame({"eICU_missing_%": [100 * Xh[f].isna().mean() if f in Xh else 100.0 for f in f25],
                   "MIMIC_train_missing_%": [cmp.loc[f, "MIMIC_missing_%"] if f in cmp.index else np.nan for f in f25]}, index=f25)
MS.index.name = "predictor"
n_imp = Xh[[f for f in f25 if f in Xh]].isna().sum(axis=1)
MS.to_csv(TAB / "table_eicu_predictor_missingness.csv")

# ---------- 3. decision curve: frozen vs locally recalibrated (10-fold CV) ----------
yy = y.to_numpy(); pf = R.risk.to_numpy(); pr = R.risk_recalibrated_cv.to_numpy()
def nb(p, t): n = len(yy); return ((p >= t) & (yy == 1)).sum() / n - ((p >= t) & (yy == 0)).sum() / n * t / (1 - t)
ts = np.round(np.arange(0.05, 0.501, 0.01), 2)
DC = pd.DataFrame({"threshold": ts, "frozen_model": [nb(pf, t) for t in ts], "recalibrated_model": [nb(pr, t) for t in ts],
                   "treat_all": [yy.mean() - (1 - yy.mean()) * t / (1 - t) for t in ts], "treat_none": 0.0})
DC.to_csv(TAB / "table_eicu_decision_curve.csv", index=False)
best = lambda col: 100 * np.mean(DC[col] > np.maximum(DC.treat_all, 0) + 1e-12)

# ---------- 4. print ----------
with pd.option_context("display.width", 220, "display.max_rows", 100, "display.max_colwidth", 70):
    print("\n=== TABLE: eICU (external) vs MIMIC-IV (development) characteristics ===")
    print(T1.fillna("").to_string())
    print("\n=== missing data of the 25 model predictors (%), eICU validation vs MIMIC training ===")
    print(MS.round(1).sort_values("eICU_missing_%", ascending=False).to_string())
    print(f"\nPatients with >=1 imputed predictor: {100*(n_imp>0).mean():.1f}% | median imputed per patient: {n_imp.median():.0f} (max {n_imp.max()})")
    print("\n=== decision curve (net benefit) at key thresholds ===")
    print(DC[DC.threshold.isin([0.10, 0.15, 0.20, 0.25, 0.30, 0.40])].round(3).to_string(index=False))
print(f"\nThresholds 5-50% where model beats both treat-all and treat-none: frozen {best('frozen_model'):.0f}%, "
      f"recalibrated {best('recalibrated_model'):.0f}%")

# ---------- 5. figures (Springer: 174 mm, Arial, vector PDF + 600-dpi PNG) ----------
mpl.rcParams.update({"font.family": "Arial", "font.size": 8, "pdf.fonttype": 42, "axes.unicode_minus": True,
                     "axes.spines.top": False, "axes.spines.right": False})
OK = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00", "grey": "#7F7F7F", "purple": "#CC79A7"}
def wilson(k, n, z=1.959964):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n); return c - h, c + h
def calib_panel(ax, p, title, stats_txt):
    lo_, hi_ = np.quantile(p, [0.01, 0.99])
    sm_ = lowess(yy, p, frac=0.75, it=0); keep = (sm_[:, 0] >= lo_) & (sm_[:, 0] <= hi_)
    ax.plot(sm_[keep, 0], sm_[keep, 1], color=OK["blue"], lw=1.4, label="LOESS calibration curve")
    q = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False).to_numpy()
    for k in range(10):
        m = q == k; kk, nn = int(yy[m].sum()), int(m.sum()); l, h = wilson(kk, nn); o = kk / nn
        ax.errorbar(p[m].mean(), o, yerr=[[o - l], [h - o]], fmt="o", ms=3, color="black", lw=0.8,
                    label="Risk deciles (95% CI)" if k == 0 else None)
    ax.plot([0, 1], [0, 1], ":", color=OK["grey"], lw=0.8, label="Perfect calibration")
    cnt, edges = np.histogram(p, bins=50, range=(0, 1))
    ax.bar(edges[:-1], 0.07 * cnt / cnt.max(), width=0.02, align="edge", color=OK["grey"], alpha=0.6, lw=0,
           label="Distribution of predicted risk")
    ax.text(0.03, 0.97, stats_txt, transform=ax.transAxes, va="top", fontsize=6.5)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Predicted 28-day mortality risk", ylabel="Observed 28-day mortality", title=title)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=6.5, frameon=False)   # clear of the histogram

# Figure (main): A ROC, B calibration of the frozen model
cam = (P.cam_pos_by_24h == 1).to_numpy(); r0 = res.loc["PRIMARY: all patients"]; rc_ = res.loc["Intended use: CAM-ICU+ by 24 h"]
m4 = P.p_APACHE_IVa_eICU.notna().to_numpy()
fig, ax = plt.subplots(1, 2, figsize=(174 / 25.4, 80 / 25.4))
for lab, yv, sv, col, ls in [(f"Model, all patients (n = {len(yy)}; AUROC {r0.AUROC:.3f})", yy, pf, OK["blue"], "-"),
                             (f"Model, CAM-ICU+ by 24 h (n = {cam.sum()}; {rc_.AUROC:.3f})", yy[cam], pf[cam], OK["green"], "-"),
                             (f"SOFA ({r0.AUROC_SOFA:.3f})", yy, P.sofa_total.to_numpy(), OK["orange"], "--"),
                             (f"APACHE II ({r0.AUROC_APACHE_II:.3f})", yy, P.apache_ii_total.to_numpy(), OK["red"], "--"),
                             (f"eICU APACHE IVa (n = {m4.sum()}; {r0.AUROC_APACHE_IVa:.3f})", yy[m4], P.p_APACHE_IVa_eICU.to_numpy()[m4], OK["purple"], ":")]:
    fpr, tpr, _ = roc_curve(yv, sv); ax[0].plot(fpr, tpr, ls, color=col, lw=1.2, label=lab)
ax[0].plot([0, 1], [0, 1], color=OK["grey"], lw=0.6)
ax[0].set(xlabel=f"1 {MINUS} specificity", ylabel="Sensitivity", title="A  Discrimination (eICU)")
ax[0].legend(loc="lower right", fontsize=6, frameon=False)
calib_panel(ax[1], pf, "B  Calibration (eICU)",
            f"Slope {fmt_num(r0.slope)}; intercept {fmt_num(r0.intercept)}\nE/O {r0['E/O']:.2f}; ICI {r0.ICI:.3f}; Brier {r0.Brier:.3f}")
fig.tight_layout()
fig.savefig(FIGS / "fig_eicu_external_validation.pdf"); fig.savefig(FIGS / "fig_eicu_external_validation.png", dpi=600)
plt.close(fig)

# Supplementary figure: A decision curve, B calibration after local recalibration
from sklearn.metrics import brier_score_loss
import statsmodels.api as sm
lp = np.log(np.clip(pr, 1e-6, 1 - 1e-6) / (1 - np.clip(pr, 1e-6, 1 - 1e-6)))
slope_rc = sm.GLM(yy, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
icept_rc = sm.GLM(yy, np.ones((len(yy), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
sm_rc = lowess(yy, pr, frac=0.75, it=0, return_sorted=False)
fig, ax = plt.subplots(1, 2, figsize=(174 / 25.4, 80 / 25.4))
ax[0].plot(DC.threshold, DC.frozen_model, color=OK["blue"], lw=1.3, label="Frozen MIMIC model")
ax[0].plot(DC.threshold, DC.recalibrated_model, color=OK["green"], lw=1.3, label="Locally recalibrated model (10-fold CV)")
ax[0].plot(DC.threshold, DC.treat_all, "--", color=OK["grey"], lw=1, label="Treat all")
ax[0].axhline(0, color="black", lw=0.8, label="Treat none")
ax[0].set(xlim=(0.05, 0.50), ylim=(-0.12, max(0.12, DC[["frozen_model", "recalibrated_model", "treat_all"]].max().max() + 0.02)),
          xlabel="Threshold probability", ylabel="Net benefit", title="A  Decision curve (eICU)")
ax[0].legend(loc="upper right", fontsize=6.5, frameon=False)
calib_panel(ax[1], pr, "B  Calibration after local recalibration",
            f"Slope {fmt_num(slope_rc)}; intercept {fmt_num(icept_rc)}\nE/O {pr.mean()/yy.mean():.2f}; ICI {np.mean(np.abs(sm_rc-pr)):.3f}; "
            f"Brier {brier_score_loss(yy, pr):.3f}")
fig.tight_layout()
fig.savefig(FIGS / "figS_eicu_decision_curve_recalibration.pdf"); fig.savefig(FIGS / "figS_eicu_decision_curve_recalibration.png", dpi=600)
plt.close(fig)

# ---------- 6. privacy check: patient-level files that must never leave this Mac ----------
pl = []
for f in sorted(OUT.glob("*.csv*")):
    try:
        head = pd.read_csv(f, nrows=0).columns
        if "patientunitstayid" in head or "stay_id" in head: pl.append(f.name)
    except Exception:
        pass
agg = sorted(p.name for p in TAB.glob("*.csv"))
print(f"\nPatient-level files in nb5 (keep on this Mac only, never share or push): {len(pl)}")
print("  " + ", ".join(pl))
print("Aggregate tables safe for the paper (nb5/tables):", ", ".join(agg))
print("Add this line to .gitignore of any repository:  data/")
print("\nSaved: figures_v2/nb5/fig_eicu_external_validation.pdf/.png (redrawn), figS_eicu_decision_curve_recalibration.pdf/.png")
print("Cell 9 done - external validation complete.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 10 (REDO: figures read from the v6 paper zip first, so iCloud placeholders cannot block it): Springer Nature figure package for the paper (for Arya)
#  - collects every paper figure from YOUR files (figures_v2, figures_v2/nb5, and the v6 paper zip
#    for the two schematic figures), in paper order: Fig 1-7 + Supplementary Fig S1-S6
#  - trims white margins and fits each figure to <= 174 mm wide (Springer double column), text size unchanged
#    unless a figure is wider than 174 mm after trimming (then scaled down slightly)
#  - writes FigN.pdf (vector, fonts embedded) + FigN.tif (600 dpi, LZW) + Figure_legends.txt + README.txt
#  - zips everything to ROOT/Springer_figures.zip  (figures only - no patient data)
#  - reads figures only; does not touch data, models or the outcome
# ============================================================
import sys, os, subprocess, shutil, zipfile, time
import numpy as np
from PIL import Image
try:
    import pymupdf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
    import pymupdf

PKG = ROOT / "Springer_figures"
shutil.rmtree(PKG, ignore_errors=True)
(PKG / "Main_figures").mkdir(parents=True); (PKG / "Supplementary_figures").mkdir()
MM = 72 / 25.4; MAX_W, MAX_H = 174.0, 234.0

# ---------- where to look (your Mac) ----------
SEARCH_DIRS = [ROOT / "figures_v2" / "nb5", ROOT / "figures_v2", ROOT,
               Path.home() / "Downloads", Path.home() / "Desktop", Path.home() / "Documents"]
SKIP = {"data", ".git", "github_release", "Springer_figures", "node_modules", "Library", ".Trash"}
def find_file(names):
    """first match in priority order: each name is searched in every folder (shallow), then recursively"""
    for n in names:
        for d in SEARCH_DIRS:
            if (d / n).is_file(): return d / n
    for n in names:
        for d in SEARCH_DIRS:
            if not d.exists(): continue
            for dp, dn, fn in os.walk(d):
                dn[:] = [x for x in dn if x not in SKIP and not x.startswith(".")]
                if n in fn: return Path(dp) / n
    return None
def from_v6_zip(member):
    """Fig 1 and Fig 2 as used in paper v6 live inside MIMIC_cancer_delirium_paper_v6.zip"""
    z = find_file(["MIMIC_cancer_delirium_paper_v6.zip"])
    if z is None: return None
    with zipfile.ZipFile(z) as zf:
        hit = [m for m in zf.namelist() if m.endswith("/" + member) or m == member]
        if not hit: return None
        tmp = PKG / "_from_v6"; tmp.mkdir(exist_ok=True)
        p = tmp / member; p.write_bytes(zf.read(hit[0])); return p

# paper figure  ->  file names to look for, in order of preference
# paper figure -> sources in order of preference. "v6:" = inside MIMIC_cancer_delirium_paper_v6.zip
# (byte-identical copies of the NB4 figures, plus the paper versions of Fig 1-2); then your figures_v2 files
MAIN = [("Fig1", ["v6:fig_study_flow.pdf", "fig_study_flow.pdf", "fig1_flow_diagram.pdf", "Fig1_study_flow.pdf"]),
        ("Fig2", ["v6:fig_workflow.pdf", "fig_workflow.pdf"]),
        ("Fig3", ["v6:fig_roc_pr.pdf", "fig2_roc_pr.pdf", "Fig2_discrimination_ROC_PR.pdf", "fig_roc_pr.pdf"]),
        ("Fig4", ["v6:fig_calibration.pdf", "fig3_calibration.pdf", "Fig3_calibration.pdf", "fig_calibration.pdf"]),
        ("Fig5", ["v6:fig_risk_tiers.pdf", "fig7_risk_tiers_uncertainty.pdf", "Fig6_risk_tiers_uncertainty.pdf", "fig_risk_tiers.pdf"]),
        ("Fig6", ["v6:fig_shap_random.pdf", "fig5_shap.pdf", "Fig5_SHAP_random_split.pdf", "fig_shap_random.pdf"]),
        ("Fig7", ["fig_eicu_external_validation.pdf"])]
SUPP = [("FigS1", ["v6:figS_decision_curves.pdf", "fig4_decision_curve.pdf", "Fig4_decision_curves.pdf", "figS_decision_curves.pdf"]),
        ("FigS2", ["v6:figS_dependence_pdp.pdf", "fig6_dependence_pdp.pdf", "FigS1_dependence_PDP.pdf", "figS_dependence_pdp.pdf"]),
        ("FigS3", ["v6:figS_shap_temporal.pdf", "figS2_shap_temporal.pdf", "FigS2_SHAP_temporal_split.pdf", "figS_shap_temporal.pdf"]),
        ("FigS4", ["v6:figS_subgroups.pdf", "fig8_subgroups_forest.pdf", "Fig7_subgroups_fairness.pdf", "figS_subgroups.pdf"]),
        ("FigS5", ["v6:figS_gcs_paradox.pdf", "figS3_gcs_paradox.pdf", "FigS3_GCS_paradox.pdf", "figS_gcs_paradox.pdf"]),
        ("FigS6", ["figS_eicu_decision_curve_recalibration.pdf"])]

# ---------- trim margins, fit to Springer width, export PDF + 600-dpi TIFF ----------
def open_retry(p, tries=2):
    for k in range(tries):
        try: return pymupdf.open(p)
        except Exception as e:
            if k == tries - 1: raise
            time.sleep(5)
def package(src, dst_pdf, dst_tif):
    s = open_retry(src); pg = s[0]
    pix = pg.get_pixmap(dpi=100, alpha=False)
    a = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n).min(axis=2) < 250
    ys, xs = np.where(a); r = pg.rect; fx, fy = r.width / pix.width, r.height / pix.height
    clip = pymupdf.Rect(max(r.x0, xs.min() * fx - MM), max(r.y0, ys.min() * fy - MM),
                        min(r.x1, (xs.max() + 1) * fx + MM), min(r.y1, (ys.max() + 1) * fy + MM))
    scale = min(1.0, MAX_W * MM / clip.width, MAX_H * MM / clip.height)
    out = pymupdf.open(); page = out.new_page(width=clip.width * scale, height=clip.height * scale)
    page.show_pdf_page(page.rect, s, 0, clip=clip)              # vector content kept, fonts stay embedded
    out.save(dst_pdf, garbage=3, deflate=True)
    hi = page.get_pixmap(dpi=600, alpha=False)
    Image.frombytes("RGB", (hi.width, hi.height), hi.samples).save(dst_tif, compression="tiff_lzw", dpi=(600, 600))
    fonts = sorted({f[3] for f in s[0].get_fonts()})
    return page.rect.width / MM, page.rect.height / MM, scale, fonts

rows, missing = [], []
for sub, lst in [("Main_figures", MAIN), ("Supplementary_figures", SUPP)]:
    for name, cands in lst:
        done, tried = False, []
        for c in cands:
            src = from_v6_zip(c[3:]) if c.startswith("v6:") else find_file([c])
            if src is None: continue
            try:
                w, h, sc, fonts = package(src, PKG / sub / f"{name}.pdf", PKG / sub / f"{name}.tif")
            except Exception as e:                       # unreadable (e.g. iCloud placeholder) -> next source
                tried.append(f"{c}: {type(e).__name__}"); continue
            label = f"paper v6 zip: {c[3:]}" if c.startswith("v6:") else str(src).replace(str(Path.home()), "~")
            rows.append({"figure": name, "source": label, "width_mm": round(w, 1), "height_mm": round(h, 1),
                         "scaled_%": round(100 * sc, 1), "ok_springer": w <= MAX_W + 0.05 and h <= MAX_H + 0.05,
                         "fonts": ", ".join(f.split("+")[-1] for f in fonts)})
            done = True; break
        if not done: missing.append((name, cands + (["unreadable: " + "; ".join(tried)] if tried else [])))
shutil.rmtree(PKG / "_from_v6", ignore_errors=True)

LEGENDS = {
    'Fig1': 'Study flow. Selection of ICU patients with cancer and delirium from MIMIC-IV v3.1 and the two validation designs. Each analysis uses the whole final cohort, divided in a different way. CAM-ICU positive: acute change or fluctuating course and inattention, together with disorganised thinking or an altered level of consciousness, at the same assessment. CAM-ICU+ by 24 h: intended-use subgroup in which delirium was already detected at the 24-h prediction time',
    'Fig2': 'Study workflow. Overview of the data source, cohort, predictor groups, the two validation designs, model development on development data only, the single evaluation on the held-out test sets and the verification steps. Schematic drawn with Python (matplotlib). TPIFE, temporal physiological information feature extraction; MICE, multivariate imputation by chained equations; SHAP-RFE, recursive feature elimination based on SHAP values',
    'Fig3': 'Discrimination on the two held-out test sets. (A, B) Receiver operating characteristic curves with AUROC and 95% DeLong confidence intervals; (C, D) precision–recall curves with the area under the curve (AUPRC); grey line, outcome prevalence. Final model: isotonic-calibrated Soft-Voting ensemble (XGBoost, random forest, CatBoost). Isotonic calibration produces stepwise risk values, which appear as straight segments of the final-model curves. SOFA, Sequential Organ Failure Assessment; APACHE II, Acute Physiology and Chronic Health Evaluation II',
    'Fig4': 'Calibration of the final model. LOESS-smoothed calibration curve, observed mortality in ten equal-size groups of predicted risk (95% Wilson intervals) and the line of perfect calibration, for (A) the random-split and (B) the temporal test set. Bottom histograms: predicted risk of survivors (grey) and non-survivors (orange). ICI, integrated calibration index; E90, 90th percentile of the absolute calibration error',
    'Fig5': 'Risk tiers and prediction uncertainty. Tier cut-offs (tertiles of out-of-fold risk in the development data) were fixed before the test sets were opened. (A) Observed 28-day mortality (95% Wilson interval) and mean predicted risk by tier in both test sets. (B) Width of the cross Venn–Abers interval by tier (random-split test set). (C) Observed mortality in confident patients and in uncertain patients, whose Venn–Abers interval crosses a tier cut-off; numbers above the bars are numbers of patients',
    'Fig6': "Explanation of the final model (SHAP), random-split test set. (A) SHAP values (contribution to each patient's predicted 28-day mortality risk, probability scale) of the 15 most important predictors; colour shows the predictor value from low to high (5th to 95th percentile). (B) Mean absolute SHAP value with 95% bootstrap confidence interval (1,000 resamples); pink bars mark predictors from the TPIFE framework, stability components or the organ-system burden index. The corresponding plot for the temporal test set is Supplementary Fig. S2",
    'Fig7': 'External validation in eICU-CRD v2.0 (frozen model; 399 patients, 64 hospitals; outcome analysed once after the analysis plan and predictions were locked). (A) Receiver operating characteristic curves of the frozen final model in all patients and in the intended-use subgroup (CAM-ICU positive by 24 h), compared with SOFA, APACHE II and the APACHE IVa prediction recorded in eICU (330 patients with an APACHE IVa value); AUROC in brackets. (B) Calibration of the frozen model: LOESS-smoothed calibration curve (shown between the 1st and 99th percentiles of predicted risk), observed 28-day mortality in ten equal-size groups of predicted risk (95% Wilson intervals), line of perfect calibration, and distribution of predicted risk (grey bars). E/O, ratio of mean predicted to observed mortality; ICI, integrated calibration index',
    'FigS1': 'Decision curve analysis. Net benefit of the final model, SOFA and APACHE II compared with treating all or no patients, across threshold probabilities from 5% to 80%, for (A) the random-split and (B) the temporal test set',
    'FigS2': 'SHAP dependence and partial dependence of the six most important predictors (random-split test set). Blue points, SHAP values for measured predictor values; open grey points, values that were imputed; black line, LOWESS trend of the SHAP values; dashed orange line, partial dependence centred on the test patients. (A) Metastatic cancer; (B) mean temperature 12–24 h; (C) chronic Elixhauser score; (D) maximum haematocrit; (E) minimum lactate; (F) SOFA score',
    'FigS3': 'Explanation of the final model (SHAP) on the temporal test set (2018–2022). Layout as in Fig. 6 of the main text: (A) SHAP values of the 15 most important predictors; (B) mean absolute SHAP value with 95% bootstrap confidence interval; pink bars, TPIFE, stability or organ-burden predictors',
    'FigS4': 'Performance across pre-specified subgroups. AUROC with 95% bootstrap confidence interval by cancer type, sex, age group and delirium timing in (A) the random-split and (B) the temporal test set. Marker size is proportional to subgroup size; grey markers, fewer than 30 deaths or 30 survivors (estimate not reliable); dashed line, overall AUROC. E/O, mean predicted divided by observed mortality',
    'FigS5': 'Post-hoc analysis of the GCS paradox in the whole cohort. (A) 28-day mortality by first GCS in the ICU, overall and by intubation within 24 h. (B) Mortality by change in GCS over the first 24 h within each first-GCS band (bars with fewer than 20 patients omitted). (C) Odds ratio of death per one-point higher first GCS in logistic models adjusted stepwise for intubation, GCS trajectory (slope and late-minus-early mean, per SD), and age, metastatic cancer and SOFA without its central nervous system component, and within strata. Error bars, 95% confidence intervals',
    'FigS6': 'Clinical utility and local updating in eICU-CRD. (A) Net benefit across threshold probabilities from 5% to 50% for the frozen MIMIC model and for the model after logistic recalibration of intercept and slope (10-fold cross-validated predictions), compared with treating all or no patients. (B) Calibration after local recalibration: LOESS curve, observed mortality in ten equal-size risk groups (95% Wilson intervals) and distribution of predicted risk',
}

order = [r_["figure"] for r_ in rows]
with open(PKG / "Figure_legends.txt", "w") as f:
    f.write("FIGURE LEGENDS - paste each one under its figure in the manuscript (Springer: legends go in the text, not in the image)\n\n")
    for k in [m[0] for m in MAIN] + [s_[0] for s_ in SUPP]:
        f.write(("Supplementary Fig. " + k[3:] if k.startswith("FigS") else "Fig. " + k[3:]) + "  " + LEGENDS[k] + "\n\n")
README = """FIGURES FOR THE MANUSCRIPT - Springer Nature artwork format
Order of citation in the text:
  Main: Fig1 study flow | Fig2 workflow | Fig3 ROC + precision-recall | Fig4 calibration |
        Fig5 risk tiers + uncertainty | Fig6 SHAP (random split) | Fig7 eICU external validation (NEW)
  Supplementary (ESM): FigS1 decision curves | FigS2 SHAP dependence/PDP | FigS3 SHAP temporal |
        FigS4 subgroups | FigS5 GCS paradox | FigS6 eICU decision curve + local recalibration (NEW)
Files: FigN.pdf = vector artwork (upload this; fonts embedded) | FigN.tif = 600 dpi TIFF (if TIFF is requested)
Legends: Figure_legends.txt (paste under each figure). figure_check.csv = size/font check of every figure.
Springer checks: width <= 174 mm, height <= 234 mm, 600 dpi raster, Arial, one figure per file, no legend inside
the image, colour-blind-safe colours, aggregate results only (no patient-level data).
Note: panels use capital letters (A, B). Springer's general guide shows lowercase (a, b); most Springer journals
accept either - check the chosen journal's submission guidelines.
"""
(PKG / "README.txt").write_text(README)
chk = pd.DataFrame(rows); chk.to_csv(PKG / "figure_check.csv", index=False)

zp = ROOT / "Springer_figures.zip"
if zp.exists(): zp.unlink()
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(PKG.rglob("*")):
        if p.is_file():
            assert p.suffix.lower() in {".pdf", ".tif", ".txt", ".csv"}, p
            z.write(p, p.relative_to(ROOT))
with pd.option_context("display.width", 250, "display.max_colwidth", 70):
    print(chk.drop(columns="fonts").to_string(index=False))
    print("\nFonts:", sorted({f_ for r_ in rows for f_ in r_["fonts"].split(", ")}))
if missing:
    print("\nNOT FOUND (zip is incomplete):")
    for n, names in missing: print(f"  {n}: looked for {names}")
    print("  -> Fig1/Fig2 need MIMIC_cancer_delirium_paper_v6.zip in Downloads/Desktop/Documents; "
          "Fig7/FigS6 need Notebook 5 Cell 9; restore iCloud placeholders if a file timed out.")
print(f"\nAll within Springer size: {bool(chk.ok_springer.all()) if len(chk) else False} | figures packed: {len(rows)}/13")
print(f"Zip: {zp}  ({zp.stat().st_size/1e6:.1f} MB) - send this to Arya")
print("Cell 10 done.")

In [ ]:
# ============================================================
# Notebook 5 - Cell 11 (NEW): VERIFICATION of the eICU external validation (read-only)
#  V1 independent recomputation of every headline number (own AUROC / DeLong / logistic code, not Cell 8's)
#  V2 audit timeline + fingerprints (plan -> amendment -> predictions locked -> outcome opened; nothing changed after)
#  V3 reproducibility: frozen MIMIC models re-run on the saved eICU inputs give the SAME locked predictions
#  V4 cohort / outcome consistency, leakage guard, privacy scan of everything that will be shared
#  Saves nb5/verification/nb5_verification.csv + eicu_paper_numbers.csv (number registry for paper v7)
#  Changes nothing: no file used by Cells 3-10 is written.
# ============================================================
import warnings, re, datetime, zipfile, os
warnings.filterwarnings("ignore")
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.model_selection import StratifiedKFold
VER = OUT / "verification"; VER.mkdir(exist_ok=True)
checks = []
def check(group, name, ok, detail=""):
    checks.append({"group": group, "check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})
def info(group, name, detail):
    checks.append({"group": group, "check": name, "result": "INFO", "detail": str(detail)})
def close(a, b, tol): return (a is not None and b is not None and np.isfinite(a) and np.isfinite(b) and abs(a - b) <= tol)

# ---------- independent metric code (written separately from Cell 8) ----------
def auc_pairs(y, s):
    """AUROC by explicit comparison of every death with every survivor (ties count 1/2)."""
    y, s = np.asarray(y), np.asarray(s, float); P, N = s[y == 1], s[y == 0]
    psi = (P[:, None] > N[None, :]) + 0.5 * (P[:, None] == N[None, :])
    return psi.mean(), psi
def delong_ci(y, s):
    a, psi = auc_pairs(y, s); v10, v01 = psi.mean(axis=1), psi.mean(axis=0)
    se = np.sqrt(v10.var(ddof=1) / len(v10) + v01.var(ddof=1) / len(v01))
    return a, max(0, a - 1.959964 * se), min(1, a + 1.959964 * se)
def delong_paired_p(y, s1, s2):
    a1, p1 = auc_pairs(y, s1); a2, p2 = auc_pairs(y, s2)
    d10 = p1.mean(1) - p2.mean(1); d01 = p1.mean(0) - p2.mean(0)
    se = np.sqrt(d10.var(ddof=1) / len(d10) + d01.var(ddof=1) / len(d01))
    return 2 * stats.norm.sf(abs((a1 - a2) / se))
def irls(X, y, offset=None, it=50):
    """plain Newton-Raphson logistic regression (no statsmodels)"""
    X = np.asarray(X, float); y = np.asarray(y, float); off = np.zeros(len(y)) if offset is None else offset
    b = np.zeros(X.shape[1])
    for _ in range(it):
        eta = X @ b + off; p = 1 / (1 + np.exp(-eta)); W = p * (1 - p)
        step = np.linalg.solve(X.T @ (X * W[:, None]), X.T @ (y - p)); b += step
        if np.max(np.abs(step)) < 1e-10: break
    return b
def lg(p): p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def calib_own(y, p):
    slope = irls(np.c_[np.ones(len(y)), lg(p)], y)[1]
    icpt = irls(np.ones((len(y), 1)), y, offset=lg(p))[0]
    ici = np.mean(np.abs(lowess(y, p, frac=0.75, it=0, return_sorted=False) - p))
    return slope, icpt, p.mean() / y.mean(), ici

# ---------- load ----------
plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
amend = json.loads((OUT / "analysis_plan_amendments.json").read_text())
lockp = json.loads((OUT / "predictions_locked.json").read_text())
olog = json.loads((OUT / "outcome_opened_log.json").read_text())
res = pd.read_csv(OUT / "eicu_external_validation_results.csv").set_index("analysis")
R = pd.read_csv(OUT / "eicu_outcome_and_risk.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
def loadP(v, a): return pd.read_csv(OUT / f"eicu_predictions_{v}_{a}.csv", dtype={"patientunitstayid": str, "hospitalid": str}).set_index("patientunitstayid")
P = loadP("harmonised", "random_split")
locked = {a: json.loads((MODELS / f"locked_rules_{a}.json").read_text()) for a in ["random_split", "temporal_split"]}

# ========== V2 audit timeline + fingerprints ==========
G = "V2 audit"
t = lambda s: datetime.datetime.fromisoformat(s)
check(G, "order: plan locked < amendment < predictions locked < outcome opened",
      t(plan["locked_at"]) < t(amend[0]["amended_at"]) <= t(lockp["saved_at"]) < t(olog["outcome_first_opened"]),
      f'{plan["locked_at"]} < {amend[0]["amended_at"]} <= {lockp["saved_at"]} < {olog["outcome_first_opened"]}')
check(G, "amendment marked before outcome", amend[0].get("before_outcome") is True)
check(G, "cohort file unchanged since plan was locked",
      hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"])
check(G, "harmonised model inputs unchanged since amendment",
      hashlib.md5((OUT / "eicu_model_inputs_harmonised.csv").read_bytes()).hexdigest() == amend[0]["harmonised_file_md5"])
for f, h in lockp["files"].items():
    check(G, f"{f} unchanged since locking", hashlib.md5((OUT / f).read_bytes()).hexdigest() == h)
opened = t(olog["outcome_first_opened"]).timestamp()
pre = ["eicu_cohort.csv", "eicu_model_inputs_harmonised.csv"] + list(lockp["files"])
check(G, "all prediction-side files last written BEFORE the outcome was opened",
      all(os.path.getmtime(OUT / f) <= opened + 1 for f in pre),
      {f: datetime.datetime.fromtimestamp(os.path.getmtime(OUT / f)).isoformat(timespec="seconds") for f in pre})
check(G, "outcome-based result files written AFTER the outcome was opened",
      all(os.path.getmtime(OUT / f) >= opened - 1 for f in ["eicu_external_validation_results.csv", "eicu_outcome_and_risk.csv"]))
ck = OUT / "nb5_file_checksums.csv"
if ck.exists():
    C = pd.read_csv(ck, dtype=str); n_ok = n_bad = n_skip = 0; bad = []
    for _, r_ in C.iterrows():
        vals = [str(v) for v in r_.values if isinstance(v, str)]
        h = next((v for v in vals if re.fullmatch(r"[0-9a-f]{32}", v)), None)
        cand = [Path(v) for v in vals if v != h] + [ROOT / v for v in vals if v != h] + [MODELS / v for v in vals if v != h]
        f = next((c for c in cand if c.is_file()), None)
        if h is None or f is None: n_skip += 1; continue
        try:
            if hashlib.md5(f.read_bytes()).hexdigest() == h: n_ok += 1
            else: n_bad += 1; bad.append(f.name)
        except Exception: n_skip += 1
    check(G, "frozen MIMIC models / NB1-4 files unchanged since Cell 1 (checksums)", n_bad == 0 and n_ok > 0,
          f"{n_ok} identical, {n_bad} changed {bad}, {n_skip} not checkable")
else:
    info(G, "Cell 1 checksum file", "nb5_file_checksums.csv not found")

# ========== V3 reproducibility: re-run the frozen models ==========
G = "V3 reproduce"
import v2_prep
v2_prep.P2 = str(ROOT / "data" / "processed_v2") + "/"
Xh = pd.read_csv(OUT / "eicu_model_inputs_harmonised.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
Xi = pd.read_csv(OUT / "eicu_model_inputs.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
for v, Xv in [("harmonised", Xh), ("icd_only", Xi)]:
    for a in ["random_split", "temporal_split"]:
        B = bundles[f"final_{a}"]; prep = read_retry(MODELS / f"prep_{a}.joblib", mode="joblib")
        Z = prep.transform(v2_prep.add_fixed_features(Xv))[list(B["features"])]
        p_new = B["final_calibrated"].predict_proba(Z)[:, 1]
        Ps = loadP(v, a).loc[Xv.index]
        d = np.max(np.abs(p_new - Ps.risk.to_numpy()))
        lr = locked[a]
        tier = np.where(p_new < lr["tier_cut_low"], "Low", np.where(p_new >= lr["tier_cut_high"], "High", "Intermediate"))
        check(G, f"{v} / {a}: frozen model reproduces locked risk", d < 1e-9, f"max difference {d:.2e}")
        check(G, f"{v} / {a}: locked tiers + threshold reproduced",
              (tier == Ps.tier.to_numpy()).all() and ((p_new >= lr["threshold"]).astype(int) == Ps.above_threshold.to_numpy()).all())

# ========== V4 cohort, outcome, leakage ==========
G = "V4 cohort/outcome"
check(G, "cohort: 403 patients, one stay per patient", len(coh) == 403 and coh.uniquepid.is_unique and coh.index.is_unique,
      f"{len(coh)} stays, {coh.uniquepid.nunique()} patients")
check(G, "all adults (age >= 18)", (Xh.loc[coh.index, "age_at_admission"] >= 18).all())
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str, usecols=["patientunitstayid", "hospitaldischargestatus", "hospitaldischargeoffset"]
                 ).set_index("patientunitstayid").reindex(coh.index)
st_ = pt.hospitaldischargestatus.str.strip(); off = pd.to_numeric(pt.hospitaldischargeoffset, errors="coerce")
known = st_.isin(["Alive", "Expired"]) & off.notna()
y_re = ((st_ == "Expired") & (off <= 40320)).astype(int)[known]
check(G, "outcome re-derived from raw patient table matches saved outcome",
      list(y_re.index) == list(R.index) and (y_re.to_numpy() == R.y.to_numpy()).all(), f"{int(known.sum())} known, {int(y_re.sum())} deaths")
check(G, "analysed 399 patients, 87 deaths (21.8%)", len(R) == 399 and int(R.y.sum()) == 87, f"{len(R)}, {int(R.y.sum())}")
Pk = P.loc[R.index]; y = R.y.to_numpy(); p = Pk.risk.to_numpy(); s = Pk.score_uncal.to_numpy()
check(G, "64 hospitals; 138 CAM-ICU+ by 24 h among analysed", Pk.hospitalid.nunique() == 64 and int(Pk.cam_pos_by_24h.sum()) == 138,
      f"{Pk.hospitalid.nunique()} hospitals, {int(Pk.cam_pos_by_24h.sum())} CAM+")
check(G, "saved risk = locked prediction file", np.max(np.abs(R.risk.to_numpy() - p)) < 1e-12)
BAN = ["expire", "discharge", "mortality", "death", "dead", "died", "outcome", "label", "status"]
bad_cols = [c for c in list(Xh.columns) + list(P.columns) if any(w in c.lower() for w in BAN)]
check(G, "leakage guard: no outcome-like column in model inputs or predictions", not bad_cols, bad_cols)
check(G, "APACHE IVa read without its actual-outcome columns (Cell 7 usecols)", "actual" not in " ".join(P.columns).lower())

# ========== V1 independent recomputation ==========
G = "V1 recompute"
num = []   # paper-number registry
def reg(key, val, src): num.append({"number": key, "value": val, "source": src})
def compare(label, P_, y_, arm, row):
    p_, s_ = P_.risk.to_numpy(), P_.score_uncal.to_numpy(); r0 = res.loc[row]
    a, lo, hi = delong_ci(y_, p_)
    check(G, f"{label}: AUROC + DeLong CI", close(a, r0.AUROC, 1e-9) and close(lo, r0.AUROC_lo, 1e-6) and close(hi, r0.AUROC_hi, 1e-6),
          f"{a:.4f} ({lo:.4f}-{hi:.4f}) vs {r0.AUROC:.4f} ({r0.AUROC_lo:.4f}-{r0.AUROC_hi:.4f})")
    sl, ic, eo, ici = calib_own(y_, p_)
    check(G, f"{label}: calibration slope / intercept / E/O / ICI",
          close(sl, r0.slope, 1e-5) and close(ic, r0.intercept, 1e-5) and close(eo, r0["E/O"], 1e-9) and close(ici, r0.ICI, 1e-6),
          f"slope {sl:.3f} int {ic:.3f} E/O {eo:.3f} ICI {ici:.3f}")
    check(G, f"{label}: Brier", close(np.mean((p_ - y_) ** 2), r0.Brier, 1e-9), f"{np.mean((p_-y_)**2):.4f}")
    thr = locked[arm]["threshold"]; pr = p_ >= thr
    sens, spec = (pr & (y_ == 1)).sum() / (y_ == 1).sum(), (~pr & (y_ == 0)).sum() / (y_ == 0).sum()
    ppv, npv = (pr & (y_ == 1)).sum() / pr.sum(), (~pr & (y_ == 0)).sum() / (~pr).sum()
    check(G, f"{label}: sens / spec / PPV / NPV at locked threshold {thr}",
          all(close(u, v, 1e-9) for u, v in [(sens, r0.sens), (spec, r0.spec), (ppv, r0.PPV), (npv, r0.NPV)]),
          f"{sens:.3f}/{spec:.3f}/{ppv:.3f}/{npv:.3f}")
    nb = p_ >= 0.30; nbv = (nb & (y_ == 1)).mean() - (nb & (y_ == 0)).mean() * 0.30 / 0.70
    check(G, f"{label}: net benefit at 30%", close(nbv, r0.NB_model_30, 1e-9), f"{nbv:.4f}")
    for comp, col in [("SOFA", "sofa_total"), ("APACHE_II", "apache_ii_total")]:
        ac = auc_pairs(y_, P_[col].to_numpy())[0]; pv = delong_paired_p(y_, s_, P_[col].to_numpy())
        check(G, f"{label}: AUROC {comp} + DeLong p vs model", close(ac, r0[f"AUROC_{comp}"], 1e-9) and close(pv, r0[f"p_vs_{comp}"], 1e-6),
              f"{ac:.3f}, p={pv:.3f}")
    return a, lo, hi, sl, ic, eo, ici
cam = (Pk.cam_pos_by_24h == 1).to_numpy()
a, lo, hi, sl, ic, eo, ici = compare("primary (all)", Pk, y, "random_split", "PRIMARY: all patients")
for k, v in [("eICU AUROC all", f"{a:.3f} ({lo:.3f}-{hi:.3f})"), ("eICU calibration slope", f"{sl:.2f}"), ("eICU intercept", f"{ic:.2f}"),
             ("eICU E/O", f"{eo:.2f}"), ("eICU ICI", f"{ici:.3f}"), ("eICU Brier", f"{np.mean((p-y)**2):.3f}")]:
    reg(k, v, "V1 recomputed")
a2, lo2, hi2, *_ = compare("CAM-ICU+ by 24 h", Pk[cam], y[cam], "random_split", "Intended use: CAM-ICU+ by 24 h")
reg(f"eICU AUROC CAM-ICU+ by 24 h (n={int(cam.sum())})", f"{a2:.3f} ({lo2:.3f}-{hi2:.3f})", "V1 recomputed")
compare("not CAM-ICU+ by 24 h", Pk[~cam], y[~cam], "random_split", "Not CAM-ICU+ by 24 h")
Pt = loadP("harmonised", "temporal_split").loc[R.index]
at, lot, hit, slt, *_ = compare("temporal model", Pt, y, "temporal_split", "Temporal model: all patients")
reg("eICU temporal model AUROC / slope", f"{at:.3f} ({lot:.3f}-{hit:.3f}) / {slt:.2f}", "V1 recomputed")
Pi = loadP("icd_only", "random_split").loc[R.index]
ai, loi, hii, *_ = compare("ICD-only comorbidity", Pi, y, "random_split", "Sensitivity: ICD-only comorbidity")
reg("eICU AUROC ICD-only comorbidity", f"{ai:.3f} ({loi:.3f}-{hii:.3f})", "V1 recomputed")
# APACHE IVa
m4 = Pk.p_APACHE_IVa_eICU.notna().to_numpy(); r0 = res.loc["PRIMARY: all patients"]
a4, lo4, hi4 = delong_ci(y[m4], Pk.p_APACHE_IVa_eICU.to_numpy()[m4]); am = auc_pairs(y[m4], p[m4])[0]
p4 = delong_paired_p(y[m4], s[m4], Pk.p_APACHE_IVa_eICU.to_numpy()[m4])
check(G, "APACHE IVa: n, AUROC CI, model AUROC same patients, DeLong p",
      int(m4.sum()) == int(r0.n_APACHE_IVa) and close(a4, r0.AUROC_APACHE_IVa, 1e-9) and close(lo4, r0.AUROC_APACHE_IVa_lo, 1e-6)
      and close(am, r0.AUROC_model_same_pts, 1e-9) and close(p4, r0.p_vs_APACHE_IVa, 1e-6),
      f"n {m4.sum()}, {a4:.3f} ({lo4:.3f}-{hi4:.3f}), model {am:.3f}, p={p4:.3f}")
reg("APACHE IVa AUROC (n)", f"{a4:.3f} ({lo4:.3f}-{hi4:.3f}) (n={m4.sum()}); model same patients {am:.3f}; p={p4:.2f}", "V1 recomputed")
reg("SOFA / APACHE II AUROC (all)", f"{auc_pairs(y, Pk.sofa_total.to_numpy())[0]:.3f} / {auc_pairs(y, Pk.apache_ii_total.to_numpy())[0]:.3f}", "V1 recomputed")
# tiers
for t_ in ["Low", "Intermediate", "High"]:
    m = (Pk.tier == t_).to_numpy()
    check(G, f"tier {t_}: n and observed mortality", int(m.sum()) == int(r0[f"tier_{t_}_n"]) and close(100 * y[m].mean(), r0[f"tier_{t_}_obs_%"], 1e-9),
          f"n {m.sum()}, observed {100*y[m].mean():.1f}%, predicted {100*p[m].mean():.1f}%")
    reg(f"tier {t_} n / observed % / predicted %", f"{m.sum()} / {100*y[m].mean():.1f} / {100*p[m].mean():.1f}", "V1 recomputed")
u = Pk.uncertain_tier.to_numpy() == 1
check(G, "uncertain share", close(100 * u.mean(), r0["uncertain_%"], 1e-9), f"{100*u.mean():.1f}%")
# heterogeneity CAM+ vs not (independent groups)
_, pa = auc_pairs(y[cam], p[cam]); _, pb = auc_pairs(y[~cam], p[~cam])
se_h = np.sqrt(sum(v.var(ddof=1) / len(v) for v in [pa.mean(1), pa.mean(0), pb.mean(1), pb.mean(0)]))
ph = 2 * stats.norm.sf(abs((pa.mean() - pb.mean()) / se_h))
info(G, "AUROC CAM-ICU+ vs not (Cell 8 printed p = 0.847)", f"difference {pa.mean()-pb.mean():.3f}, p = {ph:.3f}")
reg("AUROC difference CAM+ vs not", f"{pa.mean()-pb.mean():.3f}, p = {ph:.3f}", "V1 recomputed")
# recalibration: same 10 folds, own logistic code
lp = lg(p); p_rc = np.zeros(len(y))
for tr_, te_ in StratifiedKFold(10, shuffle=True, random_state=2026).split(lp, y):
    b = irls(np.c_[np.ones(len(tr_)), lp[tr_]], y[tr_]); p_rc[te_] = 1 / (1 + np.exp(-(b[0] + b[1] * lp[te_])))
d = np.max(np.abs(p_rc - R.risk_recalibrated_cv.to_numpy()))
slr, icr, eor, icir = calib_own(y, p_rc)
check(G, "10-fold CV recalibration reproduced", d < 1e-6, f"max difference {d:.1e}; slope {slr:.2f}, E/O {eor:.2f}, ICI {icir:.3f}, Brier {np.mean((p_rc-y)**2):.3f}")
reg("recalibrated (10-fold CV) slope / E/O / ICI / Brier", f"{slr:.2f} / {eor:.2f} / {icir:.3f} / {np.mean((p_rc-y)**2):.3f}", "V1 recomputed")
bfull = irls(np.c_[np.ones(len(y)), lp], y); reg("full-data recalibration intercept / slope", f"{bfull[0]:.2f} / {bfull[1]:.2f}", "V1 recomputed")
# decision curve table
dc_f = OUT / "tables" / "table_eicu_decision_curve.csv"
if dc_f.exists():
    DC = pd.read_csv(dc_f); errs = []
    for _, r_ in DC.iterrows():
        th = r_.threshold
        nbf = ((p >= th) & (y == 1)).mean() - ((p >= th) & (y == 0)).mean() * th / (1 - th)
        nbr = ((p_rc >= th) & (y == 1)).mean() - ((p_rc >= th) & (y == 0)).mean() * th / (1 - th)
        errs += [abs(nbf - r_.frozen_model), abs(nbr - r_.recalibrated_model)]
    check(G, "decision-curve table (all thresholds) recomputed", max(errs) < 1e-9, f"{len(DC)} thresholds, max diff {max(errs):.1e}")
    for th in [0.20, 0.25, 0.30]:
        r_ = DC.loc[np.isclose(DC.threshold, th)].iloc[0]
        reg(f"net benefit at {int(th*100)}% frozen / recalibrated / treat-all",
            f"{r_.frozen_model:.3f} / {r_.recalibrated_model:.3f} / {r_.treat_all:.3f}", "table_eicu_decision_curve.csv")
reg("cohort: stays / analysed / deaths / hospitals / CAM+ by 24 h", f"{len(coh)} / {len(R)} / {int(y.sum())} ({100*y.mean():.1f}%) / {Pk.hospitalid.nunique()} / {int(cam.sum())}", "V4")

# ========== privacy scan of everything that will be shared ==========
G = "V4 privacy"
ids = set(coh.index.astype(str)) | set(coh.uniquepid.astype(str))
tok = re.compile(r"\d{3}-\d{3,}|\d{5,9}")
def leaked(text): return sorted(set(tok.findall(text)) & ids)
shared_txt = list((OUT / "tables").glob("*.csv")) if (OUT / "tables").exists() else []
hits = {f.name: leaked(f.read_text(errors="ignore")) for f in shared_txt}
check(G, "nb5/tables: aggregate only (no patient or stay IDs)", not any(hits.values()), {k: v[:3] for k, v in hits.items() if v})
zp = ROOT / "Springer_figures.zip"
if zp.exists():
    with zipfile.ZipFile(zp) as z:
        names = z.namelist(); exts = {Path(n).suffix.lower() for n in names if not n.endswith("/")}
        check(G, "figure zip contains only figures / legends / check file", exts <= {".pdf", ".tif", ".txt", ".csv"}, sorted(exts))
        zl = {}
        for n in names:
            if n.endswith((".txt", ".csv")):
                zl[n] = leaked(z.read(n).decode("utf-8", "ignore"))
            elif n.endswith(".pdf"):
                try:
                    import pymupdf
                    doc = pymupdf.open(stream=z.read(n), filetype="pdf"); zl[n] = leaked(" ".join(pg.get_text() for pg in doc))
                except Exception:
                    pass
        check(G, "figure zip: no patient or stay IDs in any text, legend or figure", not any(zl.values()), {k: v[:3] for k, v in zl.items() if v})
else:
    info(G, "figure zip", "Springer_figures.zip not found (run Cell 10 first)")

# ---------- save + print ----------
V = pd.DataFrame(checks); V.to_csv(VER / "nb5_verification.csv", index=False)
N = pd.DataFrame(num); N.to_csv(VER / "eicu_paper_numbers.csv", index=False)
with pd.option_context("display.width", 230, "display.max_colwidth", 120, "display.max_rows", 200):
    print(V.assign(detail=V.detail.str.slice(0, 110)).to_string(index=False))
    print("\n=== eICU paper-number registry (use ONLY these in paper v7) ===")
    print(N.to_string(index=False))
n_pass, n_fail = (V.result == "PASS").sum(), (V.result == "FAIL").sum()
print(f"\nVERIFICATION: {n_pass} PASS, {n_fail} FAIL, {(V.result=='INFO').sum()} INFO")
print("ALL CHECKS PASSED - Notebook 5 verified." if n_fail == 0 else "SOME CHECKS FAILED - send me the FAIL rows.")
print("Saved: nb5/verification/nb5_verification.csv, nb5/verification/eicu_paper_numbers.csv")

In [ ]:
# ============================================================
# Notebook 5 - Cell 11 (REDO: checksum file read with any path style / md5 or sha256): VERIFICATION of the eICU external validation (read-only)
#  V1 independent recomputation of every headline number (own AUROC / DeLong / logistic code, not Cell 8's)
#  V2 audit timeline + fingerprints (plan -> amendment -> predictions locked -> outcome opened; nothing changed after)
#  V3 reproducibility: frozen MIMIC models re-run on the saved eICU inputs give the SAME locked predictions
#  V4 cohort / outcome consistency, leakage guard, privacy scan of everything that will be shared
#  Saves nb5/verification/nb5_verification.csv + eicu_paper_numbers.csv (number registry for paper v7)
#  Changes nothing: no file used by Cells 3-10 is written.
# ============================================================
import warnings, re, datetime, zipfile, os
warnings.filterwarnings("ignore")
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.model_selection import StratifiedKFold
VER = OUT / "verification"; VER.mkdir(exist_ok=True)
checks = []
def check(group, name, ok, detail=""):
    checks.append({"group": group, "check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})
def info(group, name, detail):
    checks.append({"group": group, "check": name, "result": "INFO", "detail": str(detail)})
def close(a, b, tol): return (a is not None and b is not None and np.isfinite(a) and np.isfinite(b) and abs(a - b) <= tol)

# ---------- independent metric code (written separately from Cell 8) ----------
def auc_pairs(y, s):
    """AUROC by explicit comparison of every death with every survivor (ties count 1/2)."""
    y, s = np.asarray(y), np.asarray(s, float); P, N = s[y == 1], s[y == 0]
    psi = (P[:, None] > N[None, :]) + 0.5 * (P[:, None] == N[None, :])
    return psi.mean(), psi
def delong_ci(y, s):
    a, psi = auc_pairs(y, s); v10, v01 = psi.mean(axis=1), psi.mean(axis=0)
    se = np.sqrt(v10.var(ddof=1) / len(v10) + v01.var(ddof=1) / len(v01))
    return a, max(0, a - 1.959964 * se), min(1, a + 1.959964 * se)
def delong_paired_p(y, s1, s2):
    a1, p1 = auc_pairs(y, s1); a2, p2 = auc_pairs(y, s2)
    d10 = p1.mean(1) - p2.mean(1); d01 = p1.mean(0) - p2.mean(0)
    se = np.sqrt(d10.var(ddof=1) / len(d10) + d01.var(ddof=1) / len(d01))
    return 2 * stats.norm.sf(abs((a1 - a2) / se))
def irls(X, y, offset=None, it=50):
    """plain Newton-Raphson logistic regression (no statsmodels)"""
    X = np.asarray(X, float); y = np.asarray(y, float); off = np.zeros(len(y)) if offset is None else offset
    b = np.zeros(X.shape[1])
    for _ in range(it):
        eta = X @ b + off; p = 1 / (1 + np.exp(-eta)); W = p * (1 - p)
        step = np.linalg.solve(X.T @ (X * W[:, None]), X.T @ (y - p)); b += step
        if np.max(np.abs(step)) < 1e-10: break
    return b
def lg(p): p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def calib_own(y, p):
    slope = irls(np.c_[np.ones(len(y)), lg(p)], y)[1]
    icpt = irls(np.ones((len(y), 1)), y, offset=lg(p))[0]
    ici = np.mean(np.abs(lowess(y, p, frac=0.75, it=0, return_sorted=False) - p))
    return slope, icpt, p.mean() / y.mean(), ici

# ---------- load ----------
plan = json.loads((OUT / "analysis_plan_locked.json").read_text())
amend = json.loads((OUT / "analysis_plan_amendments.json").read_text())
lockp = json.loads((OUT / "predictions_locked.json").read_text())
olog = json.loads((OUT / "outcome_opened_log.json").read_text())
res = pd.read_csv(OUT / "eicu_external_validation_results.csv").set_index("analysis")
R = pd.read_csv(OUT / "eicu_outcome_and_risk.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
coh = pd.read_csv(OUT / "eicu_cohort.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
def loadP(v, a): return pd.read_csv(OUT / f"eicu_predictions_{v}_{a}.csv", dtype={"patientunitstayid": str, "hospitalid": str}).set_index("patientunitstayid")
P = loadP("harmonised", "random_split")
locked = {a: json.loads((MODELS / f"locked_rules_{a}.json").read_text()) for a in ["random_split", "temporal_split"]}

# ========== V2 audit timeline + fingerprints ==========
G = "V2 audit"
t = lambda s: datetime.datetime.fromisoformat(s)
check(G, "order: plan locked < amendment < predictions locked < outcome opened",
      t(plan["locked_at"]) < t(amend[0]["amended_at"]) <= t(lockp["saved_at"]) < t(olog["outcome_first_opened"]),
      f'{plan["locked_at"]} < {amend[0]["amended_at"]} <= {lockp["saved_at"]} < {olog["outcome_first_opened"]}')
check(G, "amendment marked before outcome", amend[0].get("before_outcome") is True)
check(G, "cohort file unchanged since plan was locked",
      hashlib.md5((OUT / "eicu_cohort.csv").read_bytes()).hexdigest() == plan["cohort_file_md5"])
check(G, "harmonised model inputs unchanged since amendment",
      hashlib.md5((OUT / "eicu_model_inputs_harmonised.csv").read_bytes()).hexdigest() == amend[0]["harmonised_file_md5"])
for f, h in lockp["files"].items():
    check(G, f"{f} unchanged since locking", hashlib.md5((OUT / f).read_bytes()).hexdigest() == h)
opened = t(olog["outcome_first_opened"]).timestamp()
pre = ["eicu_cohort.csv", "eicu_model_inputs_harmonised.csv"] + list(lockp["files"])
check(G, "all prediction-side files last written BEFORE the outcome was opened",
      all(os.path.getmtime(OUT / f) <= opened + 1 for f in pre),
      {f: datetime.datetime.fromtimestamp(os.path.getmtime(OUT / f)).isoformat(timespec="seconds") for f in pre})
check(G, "outcome-based result files written AFTER the outcome was opened",
      all(os.path.getmtime(OUT / f) >= opened - 1 for f in ["eicu_external_validation_results.csv", "eicu_outcome_and_risk.csv"]))
ck = OUT / "nb5_file_checksums.csv"
if ck.exists():
    C = pd.read_csv(ck, dtype=str); n_ok = n_bad = n_skip = 0; bad = []
    HEX = re.compile(r"[0-9a-fA-F]{32}|[0-9a-fA-F]{64}")
    bases = [ROOT, ROOT / "notebooks", MODELS, ROOT / "data" / "processed_v2", ROOT / "data"]
    for _, r_ in C.iterrows():
        vals = [str(v).strip() for v in r_.values if isinstance(v, str)]
        h = next((v for v in vals if HEX.fullmatch(v)), None)
        f = None
        for v in vals:
            if v == h: continue
            for c in [Path(v).expanduser()] + [b_ / v for b_ in bases] + [b_ / Path(v).name for b_ in bases]:
                try:
                    if c.is_file(): f = c.resolve(); break
                except OSError: pass
            if f: break
        if h is None or f is None: n_skip += 1; continue
        try:
            algo = hashlib.md5 if len(h) == 32 else hashlib.sha256
            if algo(f.read_bytes()).hexdigest() == h.lower(): n_ok += 1
            else: n_bad += 1; bad.append(f.name)
        except Exception: n_skip += 1
    if n_ok + n_bad == 0:
        info(G, "Cell 1 checksum file could not be parsed", f"columns {list(C.columns)}; first row {C.iloc[0].to_dict() if len(C) else {}}")
    else:
        check(G, "frozen MIMIC models / NB1-4 files unchanged since Cell 1 (checksums)", n_bad == 0,
              f"{n_ok} identical, {n_bad} changed {bad}, {n_skip} not checkable")
else:
    info(G, "Cell 1 checksum file", "nb5_file_checksums.csv not found")

# ========== V3 reproducibility: re-run the frozen models ==========
G = "V3 reproduce"
import v2_prep
v2_prep.P2 = str(ROOT / "data" / "processed_v2") + "/"
Xh = pd.read_csv(OUT / "eicu_model_inputs_harmonised.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
Xi = pd.read_csv(OUT / "eicu_model_inputs.csv", dtype={"patientunitstayid": str}).set_index("patientunitstayid")
for v, Xv in [("harmonised", Xh), ("icd_only", Xi)]:
    for a in ["random_split", "temporal_split"]:
        B = bundles[f"final_{a}"]; prep = read_retry(MODELS / f"prep_{a}.joblib", mode="joblib")
        Z = prep.transform(v2_prep.add_fixed_features(Xv))[list(B["features"])]
        p_new = B["final_calibrated"].predict_proba(Z)[:, 1]
        Ps = loadP(v, a).loc[Xv.index]
        d = np.max(np.abs(p_new - Ps.risk.to_numpy()))
        lr = locked[a]
        tier = np.where(p_new < lr["tier_cut_low"], "Low", np.where(p_new >= lr["tier_cut_high"], "High", "Intermediate"))
        check(G, f"{v} / {a}: frozen model reproduces locked risk", d < 1e-9, f"max difference {d:.2e}")
        check(G, f"{v} / {a}: locked tiers + threshold reproduced",
              (tier == Ps.tier.to_numpy()).all() and ((p_new >= lr["threshold"]).astype(int) == Ps.above_threshold.to_numpy()).all())

# ========== V4 cohort, outcome, leakage ==========
G = "V4 cohort/outcome"
check(G, "cohort: 403 patients, one stay per patient", len(coh) == 403 and coh.uniquepid.is_unique and coh.index.is_unique,
      f"{len(coh)} stays, {coh.uniquepid.nunique()} patients")
check(G, "all adults (age >= 18)", (Xh.loc[coh.index, "age_at_admission"] >= 18).all())
pt = pd.read_csv(EICU / "patient.csv.gz", dtype=str, usecols=["patientunitstayid", "hospitaldischargestatus", "hospitaldischargeoffset"]
                 ).set_index("patientunitstayid").reindex(coh.index)
st_ = pt.hospitaldischargestatus.str.strip(); off = pd.to_numeric(pt.hospitaldischargeoffset, errors="coerce")
known = st_.isin(["Alive", "Expired"]) & off.notna()
y_re = ((st_ == "Expired") & (off <= 40320)).astype(int)[known]
check(G, "outcome re-derived from raw patient table matches saved outcome",
      list(y_re.index) == list(R.index) and (y_re.to_numpy() == R.y.to_numpy()).all(), f"{int(known.sum())} known, {int(y_re.sum())} deaths")
check(G, "analysed 399 patients, 87 deaths (21.8%)", len(R) == 399 and int(R.y.sum()) == 87, f"{len(R)}, {int(R.y.sum())}")
Pk = P.loc[R.index]; y = R.y.to_numpy(); p = Pk.risk.to_numpy(); s = Pk.score_uncal.to_numpy()
check(G, "64 hospitals; 138 CAM-ICU+ by 24 h among analysed", Pk.hospitalid.nunique() == 64 and int(Pk.cam_pos_by_24h.sum()) == 138,
      f"{Pk.hospitalid.nunique()} hospitals, {int(Pk.cam_pos_by_24h.sum())} CAM+")
check(G, "saved risk = locked prediction file", np.max(np.abs(R.risk.to_numpy() - p)) < 1e-12)
BAN = ["expire", "discharge", "mortality", "death", "dead", "died", "outcome", "label", "status"]
bad_cols = [c for c in list(Xh.columns) + list(P.columns) if any(w in c.lower() for w in BAN)]
check(G, "leakage guard: no outcome-like column in model inputs or predictions", not bad_cols, bad_cols)
check(G, "APACHE IVa read without its actual-outcome columns (Cell 7 usecols)", "actual" not in " ".join(P.columns).lower())

# ========== V1 independent recomputation ==========
G = "V1 recompute"
num = []   # paper-number registry
def reg(key, val, src): num.append({"number": key, "value": val, "source": src})
def compare(label, P_, y_, arm, row):
    p_, s_ = P_.risk.to_numpy(), P_.score_uncal.to_numpy(); r0 = res.loc[row]
    a, lo, hi = delong_ci(y_, p_)
    check(G, f"{label}: AUROC + DeLong CI", close(a, r0.AUROC, 1e-9) and close(lo, r0.AUROC_lo, 1e-6) and close(hi, r0.AUROC_hi, 1e-6),
          f"{a:.4f} ({lo:.4f}-{hi:.4f}) vs {r0.AUROC:.4f} ({r0.AUROC_lo:.4f}-{r0.AUROC_hi:.4f})")
    sl, ic, eo, ici = calib_own(y_, p_)
    check(G, f"{label}: calibration slope / intercept / E/O / ICI",
          close(sl, r0.slope, 1e-5) and close(ic, r0.intercept, 1e-5) and close(eo, r0["E/O"], 1e-9) and close(ici, r0.ICI, 1e-6),
          f"slope {sl:.3f} int {ic:.3f} E/O {eo:.3f} ICI {ici:.3f}")
    check(G, f"{label}: Brier", close(np.mean((p_ - y_) ** 2), r0.Brier, 1e-9), f"{np.mean((p_-y_)**2):.4f}")
    thr = locked[arm]["threshold"]; pr = p_ >= thr
    sens, spec = (pr & (y_ == 1)).sum() / (y_ == 1).sum(), (~pr & (y_ == 0)).sum() / (y_ == 0).sum()
    ppv, npv = (pr & (y_ == 1)).sum() / pr.sum(), (~pr & (y_ == 0)).sum() / (~pr).sum()
    check(G, f"{label}: sens / spec / PPV / NPV at locked threshold {thr}",
          all(close(u, v, 1e-9) for u, v in [(sens, r0.sens), (spec, r0.spec), (ppv, r0.PPV), (npv, r0.NPV)]),
          f"{sens:.3f}/{spec:.3f}/{ppv:.3f}/{npv:.3f}")
    nb = p_ >= 0.30; nbv = (nb & (y_ == 1)).mean() - (nb & (y_ == 0)).mean() * 0.30 / 0.70
    check(G, f"{label}: net benefit at 30%", close(nbv, r0.NB_model_30, 1e-9), f"{nbv:.4f}")
    for comp, col in [("SOFA", "sofa_total"), ("APACHE_II", "apache_ii_total")]:
        ac = auc_pairs(y_, P_[col].to_numpy())[0]; pv = delong_paired_p(y_, s_, P_[col].to_numpy())
        check(G, f"{label}: AUROC {comp} + DeLong p vs model", close(ac, r0[f"AUROC_{comp}"], 1e-9) and close(pv, r0[f"p_vs_{comp}"], 1e-6),
              f"{ac:.3f}, p={pv:.3f}")
    return a, lo, hi, sl, ic, eo, ici
cam = (Pk.cam_pos_by_24h == 1).to_numpy()
a, lo, hi, sl, ic, eo, ici = compare("primary (all)", Pk, y, "random_split", "PRIMARY: all patients")
for k, v in [("eICU AUROC all", f"{a:.3f} ({lo:.3f}-{hi:.3f})"), ("eICU calibration slope", f"{sl:.2f}"), ("eICU intercept", f"{ic:.2f}"),
             ("eICU E/O", f"{eo:.2f}"), ("eICU ICI", f"{ici:.3f}"), ("eICU Brier", f"{np.mean((p-y)**2):.3f}")]:
    reg(k, v, "V1 recomputed")
a2, lo2, hi2, *_ = compare("CAM-ICU+ by 24 h", Pk[cam], y[cam], "random_split", "Intended use: CAM-ICU+ by 24 h")
reg(f"eICU AUROC CAM-ICU+ by 24 h (n={int(cam.sum())})", f"{a2:.3f} ({lo2:.3f}-{hi2:.3f})", "V1 recomputed")
compare("not CAM-ICU+ by 24 h", Pk[~cam], y[~cam], "random_split", "Not CAM-ICU+ by 24 h")
Pt = loadP("harmonised", "temporal_split").loc[R.index]
at, lot, hit, slt, *_ = compare("temporal model", Pt, y, "temporal_split", "Temporal model: all patients")
reg("eICU temporal model AUROC / slope", f"{at:.3f} ({lot:.3f}-{hit:.3f}) / {slt:.2f}", "V1 recomputed")
Pi = loadP("icd_only", "random_split").loc[R.index]
ai, loi, hii, *_ = compare("ICD-only comorbidity", Pi, y, "random_split", "Sensitivity: ICD-only comorbidity")
reg("eICU AUROC ICD-only comorbidity", f"{ai:.3f} ({loi:.3f}-{hii:.3f})", "V1 recomputed")
# APACHE IVa
m4 = Pk.p_APACHE_IVa_eICU.notna().to_numpy(); r0 = res.loc["PRIMARY: all patients"]
a4, lo4, hi4 = delong_ci(y[m4], Pk.p_APACHE_IVa_eICU.to_numpy()[m4]); am = auc_pairs(y[m4], p[m4])[0]
p4 = delong_paired_p(y[m4], s[m4], Pk.p_APACHE_IVa_eICU.to_numpy()[m4])
check(G, "APACHE IVa: n, AUROC CI, model AUROC same patients, DeLong p",
      int(m4.sum()) == int(r0.n_APACHE_IVa) and close(a4, r0.AUROC_APACHE_IVa, 1e-9) and close(lo4, r0.AUROC_APACHE_IVa_lo, 1e-6)
      and close(am, r0.AUROC_model_same_pts, 1e-9) and close(p4, r0.p_vs_APACHE_IVa, 1e-6),
      f"n {m4.sum()}, {a4:.3f} ({lo4:.3f}-{hi4:.3f}), model {am:.3f}, p={p4:.3f}")
reg("APACHE IVa AUROC (n)", f"{a4:.3f} ({lo4:.3f}-{hi4:.3f}) (n={m4.sum()}); model same patients {am:.3f}; p={p4:.2f}", "V1 recomputed")
reg("SOFA / APACHE II AUROC (all)", f"{auc_pairs(y, Pk.sofa_total.to_numpy())[0]:.3f} / {auc_pairs(y, Pk.apache_ii_total.to_numpy())[0]:.3f}", "V1 recomputed")
# tiers
for t_ in ["Low", "Intermediate", "High"]:
    m = (Pk.tier == t_).to_numpy()
    check(G, f"tier {t_}: n and observed mortality", int(m.sum()) == int(r0[f"tier_{t_}_n"]) and close(100 * y[m].mean(), r0[f"tier_{t_}_obs_%"], 1e-9),
          f"n {m.sum()}, observed {100*y[m].mean():.1f}%, predicted {100*p[m].mean():.1f}%")
    reg(f"tier {t_} n / observed % / predicted %", f"{m.sum()} / {100*y[m].mean():.1f} / {100*p[m].mean():.1f}", "V1 recomputed")
u = Pk.uncertain_tier.to_numpy() == 1
check(G, "uncertain share", close(100 * u.mean(), r0["uncertain_%"], 1e-9), f"{100*u.mean():.1f}%")
# heterogeneity CAM+ vs not (independent groups)
_, pa = auc_pairs(y[cam], p[cam]); _, pb = auc_pairs(y[~cam], p[~cam])
se_h = np.sqrt(sum(v.var(ddof=1) / len(v) for v in [pa.mean(1), pa.mean(0), pb.mean(1), pb.mean(0)]))
ph = 2 * stats.norm.sf(abs((pa.mean() - pb.mean()) / se_h))
info(G, "AUROC CAM-ICU+ vs not (Cell 8 printed p = 0.847)", f"difference {pa.mean()-pb.mean():.3f}, p = {ph:.3f}")
reg("AUROC difference CAM+ vs not", f"{pa.mean()-pb.mean():.3f}, p = {ph:.3f}", "V1 recomputed")
# recalibration: same 10 folds, own logistic code
lp = lg(p); p_rc = np.zeros(len(y))
for tr_, te_ in StratifiedKFold(10, shuffle=True, random_state=2026).split(lp, y):
    b = irls(np.c_[np.ones(len(tr_)), lp[tr_]], y[tr_]); p_rc[te_] = 1 / (1 + np.exp(-(b[0] + b[1] * lp[te_])))
d = np.max(np.abs(p_rc - R.risk_recalibrated_cv.to_numpy()))
slr, icr, eor, icir = calib_own(y, p_rc)
check(G, "10-fold CV recalibration reproduced", d < 1e-6, f"max difference {d:.1e}; slope {slr:.2f}, E/O {eor:.2f}, ICI {icir:.3f}, Brier {np.mean((p_rc-y)**2):.3f}")
reg("recalibrated (10-fold CV) slope / E/O / ICI / Brier", f"{slr:.2f} / {eor:.2f} / {icir:.3f} / {np.mean((p_rc-y)**2):.3f}", "V1 recomputed")
bfull = irls(np.c_[np.ones(len(y)), lp], y); reg("full-data recalibration intercept / slope", f"{bfull[0]:.2f} / {bfull[1]:.2f}", "V1 recomputed")
# decision curve table
dc_f = OUT / "tables" / "table_eicu_decision_curve.csv"
if dc_f.exists():
    DC = pd.read_csv(dc_f); errs = []
    for _, r_ in DC.iterrows():
        th = r_.threshold
        nbf = ((p >= th) & (y == 1)).mean() - ((p >= th) & (y == 0)).mean() * th / (1 - th)
        nbr = ((p_rc >= th) & (y == 1)).mean() - ((p_rc >= th) & (y == 0)).mean() * th / (1 - th)
        errs += [abs(nbf - r_.frozen_model), abs(nbr - r_.recalibrated_model)]
    check(G, "decision-curve table (all thresholds) recomputed", max(errs) < 1e-9, f"{len(DC)} thresholds, max diff {max(errs):.1e}")
    for th in [0.20, 0.25, 0.30]:
        r_ = DC.loc[np.isclose(DC.threshold, th)].iloc[0]
        reg(f"net benefit at {int(th*100)}% frozen / recalibrated / treat-all",
            f"{r_.frozen_model:.3f} / {r_.recalibrated_model:.3f} / {r_.treat_all:.3f}", "table_eicu_decision_curve.csv")
reg("cohort: stays / analysed / deaths / hospitals / CAM+ by 24 h", f"{len(coh)} / {len(R)} / {int(y.sum())} ({100*y.mean():.1f}%) / {Pk.hospitalid.nunique()} / {int(cam.sum())}", "V4")

# ========== privacy scan of everything that will be shared ==========
G = "V4 privacy"
ids = set(coh.index.astype(str)) | set(coh.uniquepid.astype(str))
tok = re.compile(r"\d{3}-\d{3,}|\d{5,9}")
def leaked(text): return sorted(set(tok.findall(text)) & ids)
shared_txt = list((OUT / "tables").glob("*.csv")) if (OUT / "tables").exists() else []
hits = {f.name: leaked(f.read_text(errors="ignore")) for f in shared_txt}
check(G, "nb5/tables: aggregate only (no patient or stay IDs)", not any(hits.values()), {k: v[:3] for k, v in hits.items() if v})
zp = ROOT / "Springer_figures.zip"
if zp.exists():
    with zipfile.ZipFile(zp) as z:
        names = z.namelist(); exts = {Path(n).suffix.lower() for n in names if not n.endswith("/")}
        check(G, "figure zip contains only figures / legends / check file", exts <= {".pdf", ".tif", ".txt", ".csv"}, sorted(exts))
        zl = {}
        for n in names:
            if n.endswith((".txt", ".csv")):
                zl[n] = leaked(z.read(n).decode("utf-8", "ignore"))
            elif n.endswith(".pdf"):
                try:
                    import pymupdf
                    doc = pymupdf.open(stream=z.read(n), filetype="pdf"); zl[n] = leaked(" ".join(pg.get_text() for pg in doc))
                except Exception:
                    pass
        check(G, "figure zip: no patient or stay IDs in any text, legend or figure", not any(zl.values()), {k: v[:3] for k, v in zl.items() if v})
else:
    info(G, "figure zip", "Springer_figures.zip not found (run Cell 10 first)")

# ---------- save + print ----------
V = pd.DataFrame(checks); V.to_csv(VER / "nb5_verification.csv", index=False)
N = pd.DataFrame(num); N.to_csv(VER / "eicu_paper_numbers.csv", index=False)
with pd.option_context("display.width", 230, "display.max_colwidth", 120, "display.max_rows", 200):
    print(V.assign(detail=V.detail.str.slice(0, 110)).to_string(index=False))
    print("\n=== eICU paper-number registry (use ONLY these in paper v7) ===")
    print(N.to_string(index=False))
n_pass, n_fail = (V.result == "PASS").sum(), (V.result == "FAIL").sum()
print(f"\nVERIFICATION: {n_pass} PASS, {n_fail} FAIL, {(V.result=='INFO').sum()} INFO")
print("ALL CHECKS PASSED - Notebook 5 verified." if n_fail == 0 else "SOME CHECKS FAILED - send me the FAIL rows.")
print("Saved: nb5/verification/nb5_verification.csv, nb5/verification/eicu_paper_numbers.csv")